In [ ]:

# ============================================================
# NETGUARD — RESEARCH PAPER
# CELL 1 — FINAL DIRECTORY SETUP
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

RESEARCH_OUTPUTS = PROJECT / "Research_Outputs"
PAPER_ROOT = PROJECT / "Research_Paper"

if not PROJECT.is_dir():
    raise FileNotFoundError("NetGuard project not found.")

if not RESEARCH_OUTPUTS.is_dir():
    raise FileNotFoundError("Research_Outputs not found.")

# Create the publication folder
PAPER_ROOT.mkdir(parents=True, exist_ok=True)

# ONLY THREE FOLDERS
FOLDERS = [
    "01_Final_Figures",
    "02_Final_Tables",
    "03_Architecture_Diagram"
]

print("=" * 75)
print("NETGUARD — FINAL RESEARCH PAPER DIRECTORY")
print("=" * 75)

for folder_name in FOLDERS:

    folder = PAPER_ROOT / folder_name
    existed = folder.is_dir()

    folder.mkdir(parents=True, exist_ok=True)

    print(
        ("REUSED " if existed else "CREATED"),
        folder_name
    )

# Remove unwanted folders ONLY if empty.
UNWANTED = [
    "04_References",
    "05_Submission",
    "04_Manuscript",
    "References",
    "Submission",
    "Manuscript"
]

for name in UNWANTED:

    folder = PAPER_ROOT / name

    if folder.is_dir():

        if not any(folder.iterdir()):
            folder.rmdir()
            print("REMOVED EMPTY FOLDER:", name)

        else:
            print(
                "PRESERVED NON-EMPTY FOLDER:",
                name
            )

print("\nFINAL STRUCTURE")
print("-" * 75)

print("Research_Paper/")

for i, name in enumerate(FOLDERS):

    connector = (
        "└──" if i == len(FOLDERS)-1 else "├──"
    )

    print(f"    {connector} {name}/")

print("\n" + "=" * 75)
print("CELL 1 COMPLETE")
print("=" * 75)

print("Required folders:", len(FOLDERS))
print("Original research outputs modified: NO")
print("Model files modified: NO")
print("Non-empty folders deleted: NO")



In [ ]:

# ================================================================
# NETGUARD — RESEARCH PAPER
# CELL 2 — PUBLICATION TABLE 1: FORECASTING MODEL BENCHMARK
#
# Nine architectures across H5, H10, H15.
# Data from verified Research_Outputs T10 and T14.
#
# Exports:
#   PDF, SVG, PNG  -> publication table
#   CSV            -> editable underlying values
#   TEX            -> LaTeX booktabs table
#   JSON           -> source provenance
#
# No retraining, no TEST access, no overwriting.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 92)
print("CELL 2 — FINAL PAPER TABLE 1: FORECASTING BENCHMARK")
print("=" * 92)

PROJECT2 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

SOURCE2 = PROJECT2 / "Research_Outputs" / "02_Tables"
DEST2 = PROJECT2 / "Research_Paper" / "02_Final_Tables"

if not DEST2.is_dir():
    raise FileNotFoundError(
        "Run the successful directory setup Cell 1 first."
    )

AP_SOURCE2 = SOURCE2 / (
    "T10_Seven_Model_AP_Benchmark_"
    "20261027_064511_790770.csv"
)

GRAPH_SOURCE2 = SOURCE2 / (
    "T14_Graph_Architecture_Benchmark_"
    "20261027_070711_965908.csv"
)

for path in [AP_SOURCE2, GRAPH_SOURCE2]:
    if not path.is_file():
        raise FileNotFoundError(path)

ap_data2 = pd.read_csv(AP_SOURCE2)
graph_data2 = pd.read_csv(GRAPH_SOURCE2)

# ---------------------------------------------------------------
# A. VERIFY SOURCE SCHEMAS
# ---------------------------------------------------------------

needed_ap2 = {
    "Model", "Horizon", "Average_Precision",
    "Available", "Positive_Windows", "Negative_Windows"
}

needed_graph2 = {
    "Model", "Horizon", "AP", "TP", "FP", "FN", "TN"
}

if not needed_ap2.issubset(ap_data2.columns):
    raise RuntimeError(
        f"T10 missing: {needed_ap2 - set(ap_data2.columns)}"
    )

if not needed_graph2.issubset(graph_data2.columns):
    raise RuntimeError(
        f"T14 missing: {needed_graph2 - set(graph_data2.columns)}"
    )

if ap_data2.duplicated(["Model", "Horizon"]).any():
    raise RuntimeError("Duplicate T10 model/horizon entries.")

if graph_data2.duplicated(["Model", "Horizon"]).any():
    raise RuntimeError("Duplicate T14 model/horizon entries.")

if len(ap_data2) != 21:
    raise RuntimeError("Expected 21 T10 rows.")

if len(graph_data2) != 12:
    raise RuntimeError("Expected 12 T14 rows.")

MODELS2 = [
    ("Single LSTM", "Recurrent"),
    ("Stacked LSTM", "Recurrent"),
    ("XGBoost V2", "Boosted trees"),
    ("CR-LSTM", "Competing risk"),
    ("CR-TCN", "Competing risk"),
    ("Original Graph LSTM", "Graph-based"),
    ("Optimized Graph LSTM", "Graph-based"),
    ("GCN-LSTM", "Graph-based"),
    ("GAT-LSTM", "Graph-based")
]

HORIZONS2 = ["H5", "H10", "H15"]

expected_support2 = {
    "H5": 1125,
    "H10": 2250,
    "H15": 3375
}

N2 = 18839

# ---------------------------------------------------------------
# B. VERIFY COMMON VALIDATION SUPPORT
# ---------------------------------------------------------------

for row in ap_data2.itertuples():

    if row.Horizon not in HORIZONS2:
        raise RuntimeError("Unexpected T10 horizon.")

    positive = int(row.Positive_Windows)
    negative = int(row.Negative_Windows)

    if positive != expected_support2[row.Horizon]:
        raise RuntimeError("T10 positive support mismatch.")

    if positive + negative != N2:
        raise RuntimeError("T10 validation count mismatch.")

for row in graph_data2.itertuples():

    if row.Horizon not in HORIZONS2:
        raise RuntimeError("Unexpected T14 horizon.")

    if int(row.TP + row.FN) != expected_support2[row.Horizon]:
        raise RuntimeError("T14 positive support mismatch.")

    if int(row.TP + row.FP + row.FN + row.TN) != N2:
        raise RuntimeError("T14 validation count mismatch.")

# ---------------------------------------------------------------
# C. RECONCILE THE TWO VERIFIED BENCHMARKS
# ---------------------------------------------------------------

AP_VALUES2 = {}

for row in ap_data2.itertuples():

    available = (
        str(row.Available).strip().lower()
        in {"true", "1"}
    )

    value = (
        float(row.Average_Precision)
        if available
        else np.nan
    )

    AP_VALUES2[(row.Model, row.Horizon)] = value

# GCN and GAT appear in both source reports.
# Verify agreement before using graph-derived records.

for row in graph_data2.itertuples():

    name = str(row.Model)
    horizon = str(row.Horizon)
    value = float(row.AP)

    key = (name, horizon)

    if key in AP_VALUES2:

        np.testing.assert_allclose(
            AP_VALUES2[key],
            value,
            rtol=1e-6,
            atol=1e-6,
            err_msg=f"Disagreement for {name}/{horizon}"
        )

    else:
        AP_VALUES2[key] = value

expected_keys2 = {
    (model, horizon)
    for model, _ in MODELS2
    for horizon in HORIZONS2
}

if set(AP_VALUES2) != expected_keys2:
    raise RuntimeError(
        "Expected all nine models across three horizons."
    )

if not np.isnan(AP_VALUES2[("XGBoost V2", "H15")]):
    raise RuntimeError(
        "XGBoost H15 should be marked unavailable."
    )

for (name, horizon), value in AP_VALUES2.items():

    if np.isnan(value):
        continue

    if not 0 <= value <= 1:
        raise RuntimeError(
            f"Invalid AP: {name}/{horizon}"
        )

BEST2 = {
    horizon: max(
        float(AP_VALUES2[(model, horizon)])
        for model, _ in MODELS2
        if np.isfinite(
            AP_VALUES2[(model, horizon)]
        )
    )
    for horizon in HORIZONS2
}

ROWS2 = []

for model, architecture in MODELS2:

    record = {
        "Model": model,
        "Architecture": architecture
    }

    for horizon in HORIZONS2:

        ap = AP_VALUES2[(model, horizon)]

        record[f"{horizon}_AP"] = (
            float(ap) if np.isfinite(ap) else np.nan
        )

        record[f"{horizon}_Difference_pp"] = (
            100 * (ap - BEST2[horizon])
            if np.isfinite(ap)
            else np.nan
        )

    ROWS2.append(record)

TABLE2 = pd.DataFrame(ROWS2)

print("\nA. VERIFIED SOURCE BENCHMARKS")
print("-" * 92)

print("Validation windows:", N2)
print("Models compared:", len(TABLE2))
print("Forecast horizons:", HORIZONS2)
print("T10 / T14 shared model agreement: PASS")

print("\nB. PUBLICATION TABLE VALUES")
print("-" * 92)

print(
    TABLE2.round(4).to_string(index=False)
)

print("\nC. BEST MODEL PER HORIZON")
print("-" * 92)

for horizon in HORIZONS2:

    winner = [
        model for model, _ in MODELS2
        if np.isclose(
            AP_VALUES2[(model, horizon)],
            BEST2[horizon],
            atol=1e-12
        )
    ]

    print(
        f"{horizon}: {', '.join(winner)} "
        f"| AP={BEST2[horizon]:.6f}"
    )

# ---------------------------------------------------------------
# D. ACADEMIC TABLE RENDERING
# ---------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 10,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig2, ax2 = plt.subplots(
    figsize=(15.5, 7.8),
    facecolor="white"
)

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis("off")

LEFT2 = 0.055
RIGHT2 = 0.96

# Column widths:
# Model | Architecture | H5(AP, delta) |
# H10(AP, delta) | H15(AP, delta)

WIDTHS2 = [
    0.235, 0.145,
    0.10, 0.09,
    0.10, 0.09,
    0.10, 0.09
]

total_width2 = sum(WIDTHS2)
scale2 = (RIGHT2 - LEFT2) / total_width2

boundaries2 = [LEFT2]

for width in WIDTHS2:
    boundaries2.append(
        boundaries2[-1] + width*scale2
    )

centers2 = [
    (boundaries2[i] + boundaries2[i+1])/2
    for i in range(len(WIDTHS2))
]

ax2.text(
    LEFT2, 0.956,
    "Table 1",
    fontsize=13.5,
    fontweight="bold",
    ha="left"
)

ax2.text(
    LEFT2, 0.914,
    "Forecasting performance across "
    "network attack prediction horizons",
    fontsize=13,
    ha="left"
)

# Booktabs-style horizontal rules
ax2.plot(
    [LEFT2, RIGHT2],
    [0.865, 0.865],
    color="#242424",
    linewidth=1.2
)

# Top-level grouped headings
ax2.text(
    centers2[0], 0.815,
    "Forecasting model",
    ha="center",
    va="center",
    fontsize=10.5,
    fontweight="bold"
)

ax2.text(
    centers2[1], 0.815,
    "Architecture",
    ha="center",
    va="center",
    fontsize=10.5,
    fontweight="bold"
)

for horizon, a, b in [
    ("H5", 2, 3),
    ("H10", 4, 5),
    ("H15", 6, 7)
]:

    midpoint = (
        centers2[a] + centers2[b]
    ) / 2

    ax2.text(
        midpoint,
        0.820,
        f"{horizon} ({horizon[1:]} min)",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax2.plot(
        [
            boundaries2[a] + 0.006,
            boundaries2[b+1] - 0.006
        ],
        [0.788, 0.788],
        color="#444444",
        linewidth=0.8
    )

for j in [2, 4, 6]:

    ax2.text(
        centers2[j], 0.751,
        "AP",
        ha="center",
        va="center",
        fontsize=10
    )

for j in [3, 5, 7]:

    ax2.text(
        centers2[j], 0.751,
        "Δ (pp)",
        ha="center",
        va="center",
        fontsize=10
    )

ax2.plot(
    [LEFT2, RIGHT2],
    [0.716, 0.716],
    color="#242424",
    linewidth=1.0
)

ROW_START2 = 0.683
ROW_STEP2 = 0.049

for i, row in TABLE2.iterrows():

    y = ROW_START2 - i*ROW_STEP2

    ax2.text(
        boundaries2[0] + 0.006,
        y,
        row["Model"],
        ha="left",
        va="center",
        fontsize=9.65
    )

    ax2.text(
        centers2[1],
        y,
        row["Architecture"],
        ha="center",
        va="center",
        fontsize=9.2,
        color="#4C5057"
    )

    for horizon, ap_idx, diff_idx in [
        ("H5", 2, 3),
        ("H10", 4, 5),
        ("H15", 6, 7)
    ]:

        ap = row[f"{horizon}_AP"]
        delta = row[f"{horizon}_Difference_pp"]

        if pd.isna(ap):

            ax2.text(
                (centers2[ap_idx] + centers2[diff_idx])/2,
                y,
                "N/A",
                ha="center",
                va="center",
                fontsize=9.5,
                color="#555555"
            )

            continue

        is_best = np.isclose(
            ap, BEST2[horizon], atol=1e-12
        )

        ax2.text(
            centers2[ap_idx],
            y,
            f"{ap:.4f}",
            ha="center",
            va="center",
            fontsize=9.8,
            fontweight="bold" if is_best else "normal"
        )

        ax2.text(
            centers2[diff_idx],
            y,
            f"{delta:+.2f}",
            ha="center",
            va="center",
            fontsize=9.4,
            color="#444444"
        )

    # Thin rules separate model architecture families.
    if i in [2, 4]:

        rule_y = y - ROW_STEP2/2

        ax2.plot(
            [LEFT2, RIGHT2],
            [rule_y, rule_y],
            color="#A6A6A6",
            linewidth=0.55
        )

last_y2 = (
    ROW_START2 - (len(TABLE2)-1)*ROW_STEP2
)

ax2.plot(
    [LEFT2, RIGHT2],
    [last_y2 - 0.035, last_y2 - 0.035],
    color="#242424",
    linewidth=1.2
)

ax2.text(
    LEFT2, 0.180,
    "Notes: AP = average precision; "
    "Δ = AP difference from the best model at the "
    "same prediction horizon, in percentage points.",
    fontsize=9.3,
    ha="left"
)

ax2.text(
    LEFT2, 0.148,
    "Bold values mark the highest measured AP "
    "per horizon. N/A indicates an unavailable "
    "saved model prediction, not a score of zero.",
    fontsize=9.3,
    ha="left"
)

ax2.text(
    LEFT2, 0.116,
    "Synthetic NAF v2 development validation "
    "(18,839 aligned windows). "
    "H5/H10/H15 positives: 1,125 / 2,250 / 3,375.",
    fontsize=9.2,
    ha="left"
)

ax2.text(
    LEFT2, 0.084,
    "These descriptive rankings do not establish "
    "statistical significance or live-network performance.",
    fontsize=9.2,
    ha="left",
    color="#555555"
)

# ---------------------------------------------------------------
# E. CREATE UNIQUE OUTPUT NAMES
# ---------------------------------------------------------------

STAMP2 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM2 = (
    f"Table_01_Forecasting_Model_Benchmark_{STAMP2}"
)

PATHS2 = {
    ext: DEST2 / f"{STEM2}.{ext}"
    for ext in ["pdf", "svg", "png", "csv", "tex", "json"]
}

for path in PATHS2.values():

    if path.exists():
        raise FileExistsError(path)

# Save paper-ready visual versions.
for ext in ["pdf", "svg", "png"]:

    fig2.savefig(
        PATHS2[ext],
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig2)

# Save the editable data table.
TABLE2.to_csv(
    PATHS2["csv"],
    index=False,
    mode="x",
    float_format="%.8f",
    na_rep="N/A"
)

# ---------------------------------------------------------------
# F. CREATE LATEX BOOKTABS VERSION
# ---------------------------------------------------------------

def latex_value2(value, best=False):

    if pd.isna(value):
        return r"\multicolumn{1}{c}{N/A}"

    result = f"{float(value):.4f}"

    return (
        rf"\textbf{{{result}}}"
        if best
        else result
    )

lines2 = [
    r"% Requires \usepackage{booktabs}",
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Forecasting performance across attack prediction horizons.}",
    r"\label{tab:forecasting_benchmark}",
    r"\small",
    r"\begin{tabular}{llrrrrrr}",
    r"\toprule",
    r"\multirow{2}{*}{Model} & "
    r"\multirow{2}{*}{Architecture} & "
    r"\multicolumn{2}{c}{H5 (5 min)} & "
    r"\multicolumn{2}{c}{H10 (10 min)} & "
    r"\multicolumn{2}{c}{H15 (15 min)} \\",
    r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}",
    r" & & AP & $\Delta$ (pp) "
    r"& AP & $\Delta$ (pp) "
    r"& AP & $\Delta$ (pp) \\",
    r"\midrule"
]

for i, row in TABLE2.iterrows():

    pieces = [
        row["Model"],
        row["Architecture"]
    ]

    for horizon in HORIZONS2:

        ap = row[f"{horizon}_AP"]
        delta = row[f"{horizon}_Difference_pp"]

        best = (
            pd.notna(ap)
            and np.isclose(
                ap, BEST2[horizon],
                atol=1e-12
            )
        )

        pieces.append(
            latex_value2(ap, best=best)
        )

        pieces.append(
            f"{delta:+.2f}"
            if pd.notna(delta)
            else "N/A"
        )

    lines2.append(
        " & ".join(pieces) + r" \\"
    )

    if i in [2, 4]:
        lines2.append(r"\midrule")

lines2.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\vspace{2pt}",
    r"\parbox{\textwidth}{\footnotesize "
    r"AP = average precision. "
    r"$\Delta$ is measured relative to the best AP "
    r"at the same horizon in percentage points. "
    r"Bold values indicate horizon leaders. "
    r"N/A means no saved prediction is available. "
    r"Results use 18,839 synthetic development-validation "
    r"windows; differences are descriptive.}",
    r"\end{table*}"
])

# Add multirow to the documented dependencies.
lines2[0] = (
    r"% Requires \usepackage{booktabs,multirow}"
)

with PATHS2["tex"].open(
    "x", encoding="utf-8"
) as f:

    f.write("\n".join(lines2) + "\n")

# ---------------------------------------------------------------
# G. VERIFY EXPORTS AND SAVE PROVENANCE
# ---------------------------------------------------------------

def hash2(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:

        for block in iter(
            lambda: f.read(1024*1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

generated2 = [
    PATHS2[ext]
    for ext in ["pdf", "svg", "png", "csv", "tex"]
]

for path in generated2:

    if not path.is_file():
        raise RuntimeError(f"Missing output: {path}")

    if path.stat().st_size == 0:
        raise RuntimeError(f"Empty output: {path}")

manifest2 = {
    "project": "NetGuard",
    "publication_table": 1,
    "title": (
        "Forecasting performance across "
        "attack prediction horizons"
    ),
    "dataset": "Synthetic NAF v2",
    "split": "development validation",
    "validation_windows": 18839,
    "model_count": 9,
    "prediction_horizons": HORIZONS2,
    "best_ap_by_horizon": {
        key: float(value)
        for key, value in BEST2.items()
    },
    "source_files": [
        {
            "path": str(p.relative_to(PROJECT2)),
            "sha256": hash2(p)
        }
        for p in [AP_SOURCE2, GRAPH_SOURCE2]
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(PROJECT2)),
            "sha256": hash2(p)
        }
        for p in generated2
    ],
    "notes": [
        "AP is not classification accuracy.",
        "XGBoost H15 is unavailable.",
        "Shared GCN and GAT AP metrics were cross-verified.",
        "No statistical significance is implied.",
        "No official TEST data was accessed."
    ],
    "model_training": False,
    "official_test_accessed": False
}

with PATHS2["json"].open(
    "x", encoding="utf-8"
) as f:

    json.dump(
        manifest2,
        f,
        indent=2,
        allow_nan=False
    )

print("\n" + "=" * 92)
print("CELL 2 COMPLETE — FINAL PAPER TABLE 1")
print("=" * 92)

print("Models compared:", len(TABLE2))
print("Horizons:", HORIZONS2)
print("T10/T14 model agreement: PASS")
print("Output directory:", DEST2)

print("\nSAVED FILES")
print("-" * 92)

for path in PATHS2.values():

    assert path.is_file()
    assert path.stat().st_size > 0

    print("PASS:", path.name)

print("\nOriginal research tables modified: NO")
print("Model checkpoints modified: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")
print("Existing paper assets overwritten: NO")

print("=" * 92)


In [ ]:

# ================================================================
# NETGUARD — RESEARCH PAPER
# CELL 3 — TABLE 2: SOURCE-SPECIFIC FORECASTING AND ETA
#
# Models: Competing-Risk LSTM and Competing-Risk TCN
# Groups: CIC-IDS2017, UNSW-NB15, Pooled validation
#
# Metrics:
# H5/H10/H15 AP, family accuracy, family macro F1,
# conditional ETA MAE, joint negative log-likelihood.
#
# Outputs: PDF, SVG, PNG, CSV, TEX, JSON.
# No training, no TEST data, no existing-file overwrites.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 95)
print("CELL 3 — FINAL PAPER TABLE 2: SOURCE-SPECIFIC PERFORMANCE")
print("=" * 95)

ROOT3 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)
AUDIT3 = (
    ROOT3 / "Synthetic_NAF_v2" / "models"
    / "FINAL_VALIDATION_AUDIT"
)
DEST3 = ROOT3 / "Research_Paper" / "02_Final_Tables"

SOURCE_FILE3 = AUDIT3 / "SOURCE_SPECIFIC_METRICS.csv"
POOLED_FILE3 = AUDIT3 / "COMPETING_RISK_RECOMPUTED_METRICS.csv"

for p in [SOURCE_FILE3, POOLED_FILE3]:
    if not p.is_file():
        raise FileNotFoundError(p)

if not DEST3.is_dir():
    raise FileNotFoundError(
        "Research_Paper/02_Final_Tables is missing"
    )

# ---------------------------------------------------------------
# A. LOAD AND VERIFY THE ORIGINAL METRICS
# ---------------------------------------------------------------

source3 = pd.read_csv(SOURCE_FILE3)
pooled3 = pd.read_csv(POOLED_FILE3)

METRICS3 = [
    "H5_AP", "H10_AP", "H15_AP",
    "Family_Accuracy", "Family_Macro_F1",
    "ETA_MAE", "Joint_NLL"
]

SOURCE_REQUIRED3 = (
    {"Model", "Source", "Event_Windows"} | set(METRICS3)
)
POOLED_REQUIRED3 = {"Model"} | set(METRICS3)

if not SOURCE_REQUIRED3.issubset(source3.columns):
    raise RuntimeError(
        f"Source report missing columns: "
        f"{SOURCE_REQUIRED3 - set(source3.columns)}"
    )

if not POOLED_REQUIRED3.issubset(pooled3.columns):
    raise RuntimeError(
        f"Pooled report missing columns: "
        f"{POOLED_REQUIRED3 - set(pooled3.columns)}"
    )

MODELS3 = [
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]
SOURCES3 = ["CIC_IDS2017", "UNSW_NB15"]

if len(source3) != 4 or len(pooled3) != 2:
    raise RuntimeError("Unexpected source-report row counts")

if source3.duplicated(["Model", "Source"]).any():
    raise RuntimeError("Duplicate source/model combinations")

if pooled3.duplicated("Model").any():
    raise RuntimeError("Duplicate pooled model records")

if set(source3["Model"]) != set(MODELS3):
    raise RuntimeError("Unexpected source-specific models")

if set(pooled3["Model"]) != set(MODELS3):
    raise RuntimeError("Unexpected pooled models")

if set(source3["Source"]) != set(SOURCES3):
    raise RuntimeError("Unexpected source groups")

for frame in [source3, pooled3]:
    for col in METRICS3:
        frame[col] = pd.to_numeric(
            frame[col], errors="raise"
        )

    if not np.isfinite(
        frame[METRICS3].to_numpy(dtype=float)
    ).all():
        raise RuntimeError("Nonfinite evaluation metric")

    probability_metrics = [
        "H5_AP", "H10_AP", "H15_AP",
        "Family_Accuracy", "Family_Macro_F1"
    ]
    if not (
        frame[probability_metrics].ge(0).all().all()
        and frame[probability_metrics].le(1).all().all()
    ):
        raise RuntimeError("Metric outside [0,1]")

    if not frame[["ETA_MAE", "Joint_NLL"]].ge(0).all().all():
        raise RuntimeError("Negative ETA MAE or joint NLL")

EXPECTED_EVENT_COUNTS3 = {
    "CIC_IDS2017": 1710,
    "UNSW_NB15": 1665
}

source3["Event_Windows"] = pd.to_numeric(
    source3["Event_Windows"], errors="raise"
)

for row in source3.itertuples():
    if int(row.Event_Windows) != EXPECTED_EVENT_COUNTS3[row.Source]:
        raise RuntimeError(
            f"Event support mismatch: {row.Model}/{row.Source}"
        )

# The source-specific event counts are true-event windows.
# The pooled AP values were measured over all 18,839
# aligned decision windows, not only the true-event subset.
assert sum(EXPECTED_EVENT_COUNTS3.values()) == 3375

# ---------------------------------------------------------------
# B. ASSEMBLE THE SIX PUBLICATION ROWS
# ---------------------------------------------------------------

SHORT_MODEL3 = {
    "Competing-Risk LSTM": "CR-LSTM",
    "Competing-Risk TCN": "CR-TCN"
}
DISPLAY_SOURCE3 = {
    "CIC_IDS2017": "CIC-IDS2017",
    "UNSW_NB15": "UNSW-NB15",
    "Pooled": "Pooled"
}

rows3 = []

for src in SOURCES3 + ["Pooled"]:
    for model in MODELS3:

        if src == "Pooled":
            match = pooled3.loc[
                pooled3["Model"] == model
            ]
            event_count = 3375
        else:
            match = source3.loc[
                (source3["Model"] == model)
                & (source3["Source"] == src)
            ]
            event_count = EXPECTED_EVENT_COUNTS3[src]

        if len(match) != 1:
            raise RuntimeError(f"Missing row: {src}/{model}")

        rec = match.iloc[0]

        result = {
            "Source": DISPLAY_SOURCE3[src],
            "Model": SHORT_MODEL3[model],
            "True_Event_Windows": event_count
        }
        for metric in METRICS3:
            result[metric] = float(rec[metric])

        rows3.append(result)

TABLE3 = pd.DataFrame(rows3)

print("\nA. VERIFIED SCIENTIFIC RESULTS")
print("-" * 95)
print(TABLE3.round(5).to_string(index=False))

# ---------------------------------------------------------------
# C. DETERMINE BEST MODEL WITHIN EACH SOURCE
#
# Higher is better: AP, family accuracy, macro F1
# Lower is better: ETA MAE, joint NLL
# ---------------------------------------------------------------

LOWER_BETTER3 = {"ETA_MAE", "Joint_NLL"}

BEST3 = {}

for src in TABLE3["Source"].unique():
    subset = TABLE3[TABLE3["Source"] == src]

    for metric in METRICS3:
        values = subset[metric]

        BEST3[(src, metric)] = (
            float(values.min())
            if metric in LOWER_BETTER3
            else float(values.max())
        )

def is_best3(source_name, metric, value):
    return bool(np.isclose(
        value, BEST3[(source_name, metric)],
        atol=1e-10, rtol=1e-10
    ))

# ---------------------------------------------------------------
# D. RENDER A GROUPED ACADEMIC TABLE
# ---------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 10,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig3, ax3 = plt.subplots(figsize=(16.8, 7.6))
fig3.patch.set_facecolor("white")
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis("off")

LEFT3, RIGHT3 = 0.055, 0.96
WIDTHS3 = [
    0.145,  # source
    0.110,  # model
    0.075,  # true-event support
    0.082,  # H5 AP
    0.082,  # H10 AP
    0.082,  # H15 AP
    0.085,  # family accuracy
    0.085,  # family macro F1
    0.100,  # ETA MAE
    0.087   # joint NLL
]

factor3 = (RIGHT3 - LEFT3) / sum(WIDTHS3)
bounds3 = [LEFT3]
for width in WIDTHS3:
    bounds3.append(bounds3[-1] + width*factor3)

centers3 = [
    (bounds3[i] + bounds3[i+1]) / 2
    for i in range(len(WIDTHS3))
]

def rule3(y, a=LEFT3, b=RIGHT3, width=0.8):
    ax3.plot(
        [a, b], [y, y],
        color="#292929", linewidth=width
    )

def label3(x, y, value, **kwargs):
    ax3.text(
        x, y, str(value),
        ha=kwargs.pop("ha", "center"),
        va="center",
        **kwargs
    )

label3(
    LEFT3, 0.957, "Table 2",
    ha="left", fontsize=13.5, fontweight="bold"
)

label3(
    LEFT3, 0.916,
    "Source-dependent forecasting, attack-family "
    "classification and time-to-attack performance",
    ha="left", fontsize=12.8
)

rule3(0.868, width=1.15)

# Group headings
label3(centers3[0], 0.829, "Dataset", fontweight="bold")
label3(centers3[1], 0.829, "Model", fontweight="bold")
label3(centers3[2], 0.829, "True-event\nwindows",
       fontweight="bold", fontsize=9.3)

for heading, first, last in [
    ("Forecast discrimination", 3, 5),
    ("Attack-family prediction", 6, 7)
]:
    group_center = (bounds3[first] + bounds3[last+1])/2
    label3(group_center, 0.835, heading,
           fontweight="bold", fontsize=10)
    rule3(
        0.796,
        bounds3[first] + 0.007,
        bounds3[last+1] - 0.007,
        width=0.6
    )

label3(centers3[8], 0.835, "Conditional ETA",
       fontweight="bold", fontsize=9.7)
label3(centers3[9], 0.835, "Joint fit",
       fontweight="bold", fontsize=9.7)

subheaders3 = {
    3: "H5 AP",
    4: "H10 AP",
    5: "H15 AP",
    6: "Accuracy",
    7: "Macro F1",
    8: "MAE (min)",
    9: "NLL"
}

for col, title in subheaders3.items():
    label3(
        centers3[col],
        0.752,
        title,
        fontsize=9.7
    )

rule3(0.714, width=1.0)

# Data rows: two models within each of three source groups
row_y3 = [0.665, 0.603, 0.519, 0.457, 0.373, 0.311]

# Dataset names appear once per two-row block, like
# multirow labels in published research papers.
for group_index, src in enumerate(
    ["CIC-IDS2017", "UNSW-NB15", "Pooled"]
):
    y_center = (
        row_y3[2*group_index]
        + row_y3[2*group_index+1]
    ) / 2

    label3(
        bounds3[0] + 0.007,
        y_center,
        src,
        ha="left",
        fontsize=10.2
    )

numeric_cols3 = [
    ("H5_AP", 3),
    ("H10_AP", 4),
    ("H15_AP", 5),
    ("Family_Accuracy", 6),
    ("Family_Macro_F1", 7),
    ("ETA_MAE", 8),
    ("Joint_NLL", 9)
]

for row_num, record in enumerate(
    TABLE3.to_dict(orient="records")
):
    y = row_y3[row_num]

    label3(
        centers3[1], y,
        record["Model"],
        fontsize=10.0
    )

    label3(
        centers3[2], y,
        f"{record['True_Event_Windows']:,}",
        fontsize=9.8
    )

    for metric, col_idx in numeric_cols3:
        value = record[metric]

        label3(
            centers3[col_idx],
            y,
            f"{value:.4f}" if metric != "ETA_MAE"
            else f"{value:.3f}",
            fontsize=9.9,
            fontweight=(
                "bold"
                if is_best3(record["Source"], metric, value)
                else "normal"
            )
        )

    if row_num in [1, 3]:
        rule3(y-0.043, width=0.55)

rule3(0.267, width=1.15)

# Essential methodology notes
ax3.text(
    LEFT3, 0.216,
    "Notes: AP = average precision; "
    "MAE = mean absolute error; "
    "NLL = negative log-likelihood. "
    "Bold indicates the better of the two models within each dataset.",
    fontsize=9.1,
    ha="left"
)
ax3.text(
    LEFT3, 0.181,
    "The event-window column counts true future-event windows; "
    "pooled horizon AP is evaluated over 18,839 aligned decision windows.",
    fontsize=9.1,
    ha="left"
)
ax3.text(
    LEFT3, 0.146,
    "Future attack-family and ETA metrics are conditional on "
    "a true future event. Source-specific support differs "
    "and some attack families are absent in individual sources.",
    fontsize=9.1,
    ha="left"
)
ax3.text(
    LEFT3, 0.111,
    "Synthetic NAF v2 development validation. "
    "Source-specific and pooled aggregate scores are reported "
    "as saved; no confidence intervals are implied.",
    fontsize=9.1,
    ha="left", color="#555555"
)

# ---------------------------------------------------------------
# E. SAVE PDF, SVG, PNG, CSV
# ---------------------------------------------------------------

stamp3 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)
stem3 = f"Table_02_Source_Family_ETA_{stamp3}"

paths3 = {
    ext: DEST3 / f"{stem3}.{ext}"
    for ext in ["pdf", "svg", "png", "csv", "tex", "json"]
}

for path in paths3.values():
    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:
    fig3.savefig(
        paths3[ext],
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig3)

TABLE3.to_csv(
    paths3["csv"],
    index=False,
    mode="x",
    float_format="%.8f"
)

# ---------------------------------------------------------------
# F. EXPORT EDITABLE LaTeX BOOKTABS TABLE
# ---------------------------------------------------------------

metric_cols3 = [
    "H5_AP", "H10_AP", "H15_AP",
    "Family_Accuracy", "Family_Macro_F1",
    "ETA_MAE", "Joint_NLL"
]

lines3 = [
    r"% Requires \usepackage{booktabs,multirow}",
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Source-dependent forecasting, "
    r"attack-family classification, and conditional "
    r"time-to-attack performance.}",
    r"\label{tab:source_family_eta}",
    r"\small",
    r"\begin{tabular}{llrrrrrrrr}",
    r"\toprule",
    r"\multirow{2}{*}{Dataset} & "
    r"\multirow{2}{*}{Model} & "
    r"\multirow{2}{*}{Events} & "
    r"\multicolumn{3}{c}{Forecast AP} & "
    r"\multicolumn{2}{c}{Attack family} & "
    r"\multicolumn{1}{c}{ETA} & "
    r"\multicolumn{1}{c}{Joint fit} \\",
    r"\cmidrule(lr){4-6}\cmidrule(lr){7-8}",
    r" & & & H5 & H10 & H15 & "
    r"Accuracy & Macro F1 & MAE (min) & NLL \\",
    r"\midrule"
]

display_groups3 = ["CIC-IDS2017", "UNSW-NB15", "Pooled"]

for group_number, src in enumerate(display_groups3):
    subset = TABLE3[TABLE3["Source"] == src]

    for model_number, row in enumerate(
        subset.to_dict(orient="records")
    ):
        cells = [
            (
                rf"\multirow{{2}}{{*}}{{{src}}}"
                if model_number == 0 else ""
            ),
            row["Model"],
            f"{row['True_Event_Windows']:,}"
        ]

        for metric in metric_cols3:
            value = row[metric]
            formatted = (
                f"{value:.3f}" if metric == "ETA_MAE"
                else f"{value:.4f}"
            )

            if is_best3(src, metric, value):
                formatted = rf"\textbf{{{formatted}}}"

            cells.append(formatted)

        lines3.append(" & ".join(cells) + r" \\")

    if group_number < len(display_groups3)-1:
        lines3.append(r"\midrule")

lines3.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\vspace{2pt}",
    r"\parbox{\textwidth}{\footnotesize "
    r"AP = average precision; "
    r"MAE = mean absolute error; "
    r"NLL = negative log-likelihood. "
    r"Events reports true future-event windows "
    r"(1,710 CIC-derived, 1,665 UNSW-derived). "
    r"Pooled horizon AP uses 18,839 decision windows. "
    r"Family and ETA metrics are conditional on "
    r"a true future event. Best values are bold "
    r"within each dataset. Results are from "
    r"development validation.}",
    r"\end{table*}"
])

with paths3["tex"].open("x", encoding="utf-8") as f:
    f.write("\n".join(lines3) + "\n")

# ---------------------------------------------------------------
# G. SAVE INPUT PROVENANCE AND CHECK ALL OUTPUTS
# ---------------------------------------------------------------

def sha3(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024*1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

generated3 = [
    paths3[ext]
    for ext in ["pdf", "svg", "png", "csv", "tex"]
]

manifest3 = {
    "project": "NetGuard",
    "publication_table": 2,
    "title": "Source-dependent forecasting, family and ETA",
    "source_groups": display_groups3,
    "models": list(SHORT_MODEL3.values()),
    "dataset": "Synthetic NAF v2",
    "evaluation": "development validation",
    "pooled_decision_windows": 18839,
    "pooled_true_event_windows": 3375,
    "source_files": [
        {
            "path": str(p.relative_to(ROOT3)),
            "sha256": sha3(p)
        }
        for p in [SOURCE_FILE3, POOLED_FILE3]
    ],
    "outputs": [
        {
            "path": str(p.relative_to(ROOT3)),
            "sha256": sha3(p)
        }
        for p in generated3
    ],
    "limitations": [
        "Family and ETA scores are event-conditional.",
        "Source-specific family support differs.",
        "Pooled and source-specific macro F1 need not average.",
        "Development validation, not an independent test.",
        "No uncertainty intervals computed for this table."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with paths3["json"].open("x", encoding="utf-8") as f:
    json.dump(manifest3, f, indent=2, allow_nan=False)

print("\n" + "=" * 95)
print("CELL 3 COMPLETE — FINAL PAPER TABLE 2")
print("=" * 95)

print("Models:", len(MODELS3))
print("Source groups including pooled:", 3)
print("Publication rows:", len(TABLE3))
print("Scientific metrics per row:", len(METRICS3))
print("\nSaved outputs:")

for ext, path in paths3.items():
    assert path.is_file() and path.stat().st_size > 0
    print(f"PASS [{ext.upper()}] {path.name}")

print("\nModel training: NONE")
print("Official TEST accessed: NO")
print("Original reports modified: NO")
print("Existing publication tables overwritten: NO")
print("=" * 95)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 4 — PUBLICATION TABLE 3: OPERATIONAL FORECASTING
#
# Models: Competing-Risk LSTM and Competing-Risk TCN
# Horizons: H5, H10, H15
# Selected thresholds: 0.20 and 0.50
#
# Metrics:
# Precision, recall, false-positive rate,
# false-positive windows, missed-event windows,
# and whole-benign-episode alerts.
#
# Outputs: PDF, SVG, PNG, CSV, TEX, JSON
#
# No retraining / No TEST / No overwriting
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 96)
print("CELL 4 — FINAL PAPER TABLE 3: OPERATIONAL FORECASTING")
print("=" * 96)

PROJECT4 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

SOURCE4 = (
    PROJECT4 / "Synthetic_NAF_v2"
    / "models" / "FINAL_VALIDATION_AUDIT"
    / "FIXED_THRESHOLD_ALERT_AUDIT.csv"
)

DEST4 = (
    PROJECT4 / "Research_Paper"
    / "02_Final_Tables"
)

if not SOURCE4.is_file():
    raise FileNotFoundError(SOURCE4)

if not DEST4.is_dir():
    raise FileNotFoundError(DEST4)

data4 = pd.read_csv(SOURCE4)

# ======================================================================
# A. VALIDATE ORIGINAL 18 OPERATING POINTS
# ======================================================================

REQUIRED4 = {
    "Model", "Horizon", "Threshold",
    "TP", "FP", "TN", "FN",
    "Precision", "Recall", "False_Positive_Rate",
    "Whole_Benign_Episodes",
    "Benign_Episodes_Alerted",
    "Benign_Episode_Alert_Fraction"
}

missing4 = REQUIRED4 - set(data4.columns)

if missing4:
    raise RuntimeError(
        f"Source report missing columns: {sorted(missing4)}"
    )

MODELS4 = [
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]

MODEL_SHORT4 = {
    "Competing-Risk LSTM": "CR-LSTM",
    "Competing-Risk TCN": "CR-TCN"
}

HORIZONS4 = ["H5", "H10", "H15"]
ALL_THRESHOLDS4 = [0.1, 0.2, 0.5]
PAPER_THRESHOLDS4 = [0.2, 0.5]

EXPECTED_POS4 = {
    "H5": 1125,
    "H10": 2250,
    "H15": 3375
}

if len(data4) != 18:
    raise RuntimeError(
        f"Expected 18 saved operating points, found {len(data4)}"
    )

if set(data4["Model"]) != set(MODELS4):
    raise RuntimeError("Unexpected forecasting models")

if set(data4["Horizon"]) != set(HORIZONS4):
    raise RuntimeError("Unexpected horizons")

numeric4 = [
    "Threshold", "TP", "FP", "TN", "FN",
    "Precision", "Recall", "False_Positive_Rate",
    "Whole_Benign_Episodes",
    "Benign_Episodes_Alerted",
    "Benign_Episode_Alert_Fraction"
]

for col in numeric4:
    data4[col] = pd.to_numeric(
        data4[col],
        errors="raise"
    )

if not np.isfinite(
    data4[numeric4].to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite source values")

if data4.duplicated(
    ["Model", "Horizon", "Threshold"]
).any():
    raise RuntimeError("Duplicate operating points")

if set(np.round(data4["Threshold"], 5)) != set(
    ALL_THRESHOLDS4
):
    raise RuntimeError("Unexpected saved thresholds")

for model in MODELS4:
    for horizon in HORIZONS4:
        for threshold in ALL_THRESHOLDS4:
            match = data4.loc[
                (data4["Model"] == model)
                & (data4["Horizon"] == horizon)
                & np.isclose(data4["Threshold"], threshold)
            ]
            if len(match) != 1:
                raise RuntimeError(
                    f"Missing operating point: "
                    f"{model}/{horizon}/{threshold}"
                )

count_cols4 = [
    "TP", "FP", "TN", "FN",
    "Whole_Benign_Episodes",
    "Benign_Episodes_Alerted"
]

if (data4[count_cols4] < 0).any().any():
    raise RuntimeError("Negative counts in audit")

if not np.allclose(
    data4[count_cols4],
    np.round(data4[count_cols4])
):
    raise RuntimeError("Noninteger counts in audit")

for row in data4.itertuples():

    tp, fp, tn, fn = (
        int(row.TP), int(row.FP),
        int(row.TN), int(row.FN)
    )

    total = tp + fp + tn + fn

    if total != 18839:
        raise RuntimeError(
            "Incorrect validation-window support"
        )

    if tp + fn != EXPECTED_POS4[row.Horizon]:
        raise RuntimeError(
            "Incorrect horizon-specific positive support"
        )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0

    whole_benign = int(row.Whole_Benign_Episodes)
    alerted = int(row.Benign_Episodes_Alerted)

    if whole_benign != 75:
        raise RuntimeError(
            "Unexpected count of whole-benign episodes"
        )

    if not 0 <= alerted <= whole_benign:
        raise RuntimeError("Invalid benign-episode alert count")

    episode_fraction = alerted / whole_benign

    np.testing.assert_allclose(
        [
            precision,
            recall,
            fpr,
            episode_fraction
        ],
        [
            row.Precision,
            row.Recall,
            row.False_Positive_Rate,
            row.Benign_Episode_Alert_Fraction
        ],
        rtol=1e-8,
        atol=1e-9,
        err_msg=(
            f"Saved metric disagreement: "
            f"{row.Model}/{row.Horizon}/{row.Threshold}"
        )
    )

print("\nA. SOURCE REPORT VALIDATION")
print("-" * 96)

print("Saved operating points:", len(data4))
print("Forecasting models:", len(MODELS4))
print("Horizons:", HORIZONS4)
print("Available thresholds:", ALL_THRESHOLDS4)
print("Validation windows per operating point: 18,839")
print("Whole-benign episodes per operating point: 75")
print("All confusion-derived metrics verified: PASS")

# ======================================================================
# B. BUILD CONCISE PUBLICATION TABLE
# ======================================================================

selected4 = data4.loc[
    np.isclose(data4["Threshold"], 0.2)
    | np.isclose(data4["Threshold"], 0.5)
].copy()

selected4["Model"] = selected4["Model"].map(
    MODEL_SHORT4
)

selected4["Horizon"] = pd.Categorical(
    selected4["Horizon"],
    categories=HORIZONS4,
    ordered=True
)

selected4["Model"] = pd.Categorical(
    selected4["Model"],
    categories=["CR-LSTM", "CR-TCN"],
    ordered=True
)

selected4 = (
    selected4
    .sort_values(["Horizon", "Model", "Threshold"])
    .reset_index(drop=True)
)

if len(selected4) != 12:
    raise RuntimeError(
        "Expected 12 selected publication operating points"
    )

TABLE4 = pd.DataFrame({
    "Horizon": selected4["Horizon"].astype(str),
    "Model": selected4["Model"].astype(str),
    "Threshold": selected4["Threshold"].astype(float),
    "Precision": selected4["Precision"].astype(float),
    "Recall": selected4["Recall"].astype(float),
    "False_Positive_Rate":
        selected4["False_Positive_Rate"].astype(float),
    "False_Positive_Windows": selected4["FP"].astype(int),
    "Missed_Event_Windows": selected4["FN"].astype(int),
    "Benign_Episodes_Alerted":
        selected4["Benign_Episodes_Alerted"].astype(int),
    "Total_Whole_Benign_Episodes":
        selected4["Whole_Benign_Episodes"].astype(int),
    "Benign_Episode_Alert_Fraction":
        selected4["Benign_Episode_Alert_Fraction"].astype(float)
})

print("\nB. SELECTED PUBLICATION OPERATING POINTS")
print("-" * 96)

print(
    TABLE4.round(5).to_string(index=False)
)

# ======================================================================
# C. ACADEMIC TABLE DESIGN
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 10,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig4, ax4 = plt.subplots(
    figsize=(16.2, 8.3),
    facecolor="white"
)

ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis("off")

LEFT4 = 0.055
RIGHT4 = 0.96

# Horizon | Model | Threshold | Precision | Recall |
# FPR | FP | FN | Benign episodes alerted
WIDTHS4 = [
    0.085,
    0.115,
    0.090,
    0.110,
    0.110,
    0.100,
    0.095,
    0.095,
    0.180
]

scale4 = (
    RIGHT4 - LEFT4
) / sum(WIDTHS4)

bounds4 = [LEFT4]

for width in WIDTHS4:
    bounds4.append(
        bounds4[-1] + width*scale4
    )

centers4 = [
    (bounds4[i] + bounds4[i+1]) / 2
    for i in range(len(WIDTHS4))
]

def rule4(y, left=LEFT4, right=RIGHT4, width=0.8):
    ax4.plot(
        [left, right],
        [y, y],
        color="#262626",
        linewidth=width
    )

def txt4(x, y, value, **kwargs):
    ax4.text(
        x, y, str(value),
        ha=kwargs.pop("ha", "center"),
        va="center",
        **kwargs
    )

# Caption
txt4(
    LEFT4, 0.963,
    "Table 3",
    ha="left",
    fontsize=13.5,
    fontweight="bold"
)

txt4(
    LEFT4, 0.919,
    "Operational performance of future-attack "
    "forecasting models at fixed decision thresholds",
    ha="left",
    fontsize=12.8
)

rule4(0.873, width=1.2)

# Grouped column headers
txt4(
    centers4[0], 0.835,
    "Horizon",
    fontweight="bold"
)

txt4(
    centers4[1], 0.835,
    "Model",
    fontweight="bold"
)

txt4(
    centers4[2], 0.835,
    "Threshold",
    fontweight="bold",
    fontsize=9.6
)

groups4 = [
    ("Window-level performance", 3, 5),
    ("Window errors", 6, 7),
    ("Episode-level alerts", 8, 8)
]

for heading, start, stop in groups4:

    group_x = (
        bounds4[start] + bounds4[stop+1]
    ) / 2

    txt4(
        group_x,
        0.836,
        heading,
        fontweight="bold",
        fontsize=10.0
    )

    rule4(
        0.802,
        left=bounds4[start] + 0.005,
        right=bounds4[stop+1] - 0.005,
        width=0.6
    )

headers4 = {
    3: "Precision (%)",
    4: "Recall (%)",
    5: "FPR (%)",
    6: "FP",
    7: "FN",
    8: "Benign episodes\nalerted"
}

for col, heading in headers4.items():
    txt4(
        centers4[col],
        0.758,
        heading,
        fontsize=9.7
    )

rule4(0.709, width=1.0)

# Twelve rows, four per horizon
START_Y4 = 0.680
ROW_GAP4 = 0.0374

for i, record in enumerate(
    TABLE4.to_dict(orient="records")
):
    y = START_Y4 - i*ROW_GAP4

    horizon_start = i % 4 == 0

    # Merge horizon labels visually across four rows
    if horizon_start:
        group_center_y = (
            y + (y - 3*ROW_GAP4)
        ) / 2

        txt4(
            centers4[0],
            group_center_y,
            record["Horizon"],
            fontweight="bold",
            fontsize=10.3
        )

    # Merge model names visually across two threshold rows
    if i % 2 == 0:
        model_center_y = y - ROW_GAP4/2

        txt4(
            centers4[1],
            model_center_y,
            record["Model"],
            fontsize=10.2
        )

    txt4(
        centers4[2],
        y,
        f"{record['Threshold']:.2f}",
        fontsize=10
    )

    txt4(
        centers4[3],
        y,
        f"{record['Precision']*100:.1f}",
        fontsize=10
    )

    txt4(
        centers4[4],
        y,
        f"{record['Recall']*100:.1f}",
        fontsize=10
    )

    txt4(
        centers4[5],
        y,
        f"{record['False_Positive_Rate']*100:.2f}",
        fontsize=10
    )

    txt4(
        centers4[6],
        y,
        f"{record['False_Positive_Windows']:,}",
        fontsize=10
    )

    txt4(
        centers4[7],
        y,
        f"{record['Missed_Event_Windows']:,}",
        fontsize=10
    )

    benign_alerts = record["Benign_Episodes_Alerted"]
    benign_total = record["Total_Whole_Benign_Episodes"]

    txt4(
        centers4[8],
        y,
        (
            f"{benign_alerts}/{benign_total} "
            f"({record['Benign_Episode_Alert_Fraction']*100:.1f}%)"
        ),
        fontsize=9.7
    )

    # Light division between models
    if i in [1, 5, 9]:
        rule4(
            y - ROW_GAP4/2,
            left=bounds4[1],
            right=RIGHT4,
            width=0.4
        )

    # Stronger division between horizons
    if i in [3, 7]:
        rule4(
            y - ROW_GAP4/2,
            width=0.7
        )

last_y4 = START_Y4 - 11*ROW_GAP4

rule4(
    last_y4 - 0.032,
    width=1.2
)

# Footnotes
ax4.text(
    LEFT4, 0.155,
    "Notes: Precision and recall use true-positive attack "
    "windows. FPR = false positives / all true-negative windows; "
    "FP and FN are window counts.",
    ha="left",
    fontsize=9.0
)

ax4.text(
    LEFT4, 0.127,
    "Benign episodes alerted = number of entirely benign "
    "episodes with at least one alert / 75; "
    "this is not the same quantity as window-level FPR.",
    ha="left",
    fontsize=9.0
)

ax4.text(
    LEFT4, 0.099,
    "Synthetic NAF v2 development validation, "
    "18,839 overlapping decision windows. "
    "Thresholds 0.20 and 0.50 are illustrative operating "
    "points, not optimized deployment settings.",
    ha="left",
    fontsize=9.0
)

ax4.text(
    LEFT4, 0.071,
    "The original audit also contains threshold 0.10; "
    "it remains preserved in Research_Outputs.",
    ha="left",
    fontsize=9.0,
    color="#555555"
)

# ======================================================================
# D. EXPORT PAPER TABLE
# ======================================================================

STAMP4 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM4 = (
    f"Table_03_Operational_Forecasting_{STAMP4}"
)

PATHS4 = {
    ext: DEST4 / f"{STEM4}.{ext}"
    for ext in [
        "pdf", "svg", "png", "csv", "tex", "json"
    ]
}

for path in PATHS4.values():
    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig4.savefig(
        PATHS4[ext],
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig4)

# Editable table containing unrounded values
TABLE4.to_csv(
    PATHS4["csv"],
    index=False,
    mode="x",
    float_format="%.10f"
)

# ======================================================================
# E. LaTeX BOOKTABS EXPORT
# ======================================================================

latex4 = [
    r"% Requires \usepackage{booktabs,multirow}",
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Operational performance of future-attack "
    r"forecasting models at fixed decision thresholds.}",
    r"\label{tab:operational_forecasting}",
    r"\small",
    r"\begin{tabular}{llrrrrrrr}",
    r"\toprule",
    r"\multirow{2}{*}{Horizon} & "
    r"\multirow{2}{*}{Model} & "
    r"\multirow{2}{*}{$\tau$} & "
    r"\multicolumn{3}{c}{Window-level performance} & "
    r"\multicolumn{2}{c}{Window errors} & "
    r"\multicolumn{1}{c}{Episode-level alerts} \\",
    r"\cmidrule(lr){4-6}\cmidrule(lr){7-8}",
    r" & & & Precision (\%) & Recall (\%) & "
    r"FPR (\%) & FP & FN & Benign episodes alerted \\",
    r"\midrule"
]

for group_index, horizon in enumerate(HORIZONS4):

    source_group = TABLE4[
        TABLE4["Horizon"] == horizon
    ]

    for model_index, model in enumerate([
        "CR-LSTM", "CR-TCN"
    ]):

        part = source_group[
            source_group["Model"] == model
        ].sort_values("Threshold")

        for threshold_index, row in enumerate(
            part.to_dict(orient="records")
        ):

            horizon_text = (
                rf"\multirow{{4}}{{*}}{{{horizon}}}"
                if model_index == 0 and threshold_index == 0
                else ""
            )

            model_text = (
                rf"\multirow{{2}}{{*}}{{{model}}}"
                if threshold_index == 0
                else ""
            )

            episode_text = (
                f"{row['Benign_Episodes_Alerted']}"
                f"/{row['Total_Whole_Benign_Episodes']}"
            )

            cells = [
                horizon_text,
                model_text,
                f"{row['Threshold']:.2f}",
                f"{row['Precision']*100:.1f}",
                f"{row['Recall']*100:.1f}",
                f"{row['False_Positive_Rate']*100:.2f}",
                f"{row['False_Positive_Windows']}",
                f"{row['Missed_Event_Windows']}",
                episode_text
            ]

            latex4.append(
                " & ".join(cells) + r" \\"
            )

        if model_index == 0:
            latex4.append(
                r"\cmidrule(lr){2-9}"
            )

    if group_index < 2:
        latex4.append(r"\midrule")

latex4.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\vspace{2pt}",
    r"\parbox{\textwidth}{\footnotesize "
    r"FPR = false-positive rate; FP = false-positive "
    r"windows; FN = false-negative windows. "
    r"Benign episode alerts count wholly benign "
    r"episodes with at least one alert (denominator 75). "
    r"Results use 18,839 synthetic development-validation "
    r"decision windows. Thresholds are illustrative, "
    r"not deployment-optimized.}",
    r"\end{table*}"
])

with PATHS4["tex"].open(
    "x", encoding="utf-8"
) as f:
    f.write("\n".join(latex4) + "\n")

# ======================================================================
# F. SCIENTIFIC PROVENANCE
# ======================================================================

def sha4(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

GENERATED4 = [
    PATHS4[ext]
    for ext in ["pdf", "svg", "png", "csv", "tex"]
]

MANIFEST4 = {
    "project": "NetGuard",
    "publication_table": 3,
    "title": "Operational forecasting performance",
    "models": MODELS4,
    "horizons": HORIZONS4,
    "selected_thresholds": PAPER_THRESHOLDS4,
    "original_audit_thresholds": ALL_THRESHOLDS4,
    "validation_windows_per_operating_point": 18839,
    "whole_benign_episodes": 75,
    "all_original_rows_verified": True,
    "source_file": {
        "path": str(SOURCE4.relative_to(PROJECT4)),
        "sha256": sha4(SOURCE4)
    },
    "generated_files": [
        {
            "path": str(path.relative_to(PROJECT4)),
            "sha256": sha4(path)
        }
        for path in GENERATED4
    ],
    "limitations": [
        "Thresholds are fixed illustrative diagnostics.",
        "Thresholds were not selected for deployment.",
        "Decision windows overlap in time.",
        "Whole-benign-episode alert fraction is not window FPR.",
        "Evaluation uses synthetic development validation.",
        "No official TEST evaluation was performed."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS4["json"].open(
    "x", encoding="utf-8"
) as f:
    json.dump(
        MANIFEST4,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# G. COMPLETE
# ======================================================================

print("\n" + "=" * 96)
print("CELL 4 COMPLETE — FINAL PAPER TABLE 3")
print("=" * 96)

print("Original source operating points verified:", len(data4))
print("Publication operating points:", len(TABLE4))
print("Forecast horizons:", HORIZONS4)
print("Paper thresholds:", PAPER_THRESHOLDS4)
print("Models:", len(MODELS4))

print("\nSAVED OUTPUTS")
print("-" * 96)

for ext, path in PATHS4.items():

    assert path.is_file()
    assert path.stat().st_size > 0

    print(f"PASS [{ext.upper()}] {path.name}")

print("\nOriginal research reports modified: NO")
print("Previous paper tables overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 96)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 5 — PUBLICATION TABLE 4: CURRENT ATTACK CLASSIFICATION
#
# Four 10-class classifiers:
#   1. Original XGBoost V1
#   2. Class-weighted XGBoost V2
#   3. Conflict-aware XGBoost
#   4. CatBoost
#
# Saved metadata supplies ALL aggregate metrics.
# Per-class metrics for V2/Conflict/CatBoost are explicitly
# transcribed from previously executed Cells 18, 22 and 25.
# V1 per-class metrics are independently recalculated
# from its saved confusion matrix.
#
# Outputs: PDF, SVG, PNG, CSV, TEX, JSON
# No retraining, no TEST access, no overwritten files.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 98)
print("CELL 5 — FINAL PAPER TABLE 4: CURRENT ATTACK CLASSIFICATION")
print("=" * 98)

# ======================================================================
# A. EXACT VERIFIED SOURCE PATHS
# ======================================================================

ROOT5 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

CURRENT5 = ROOT5 / "Current_Attack_Detection"

DEST5 = ROOT5 / "Research_Paper" / "02_Final_Tables"

if not DEST5.is_dir():
    raise FileNotFoundError(DEST5)

SOURCES5 = {
    "XGBoost V1": (
        CURRENT5 / "artifacts"
        / "xgboost_20261027_023158_885060"
        / "validation_metrics.json"
    ),
    "Weighted XGBoost V2": (
        CURRENT5 / "experiments"
        / "weighted_xgb_v2_20261027_035816_500107"
        / "metadata.json"
    ),
    "Conflict-aware XGBoost": (
        CURRENT5 / "experiments"
        / "conflict_aware_xgb_20261027_042501_508802"
        / "metadata.json"
    ),
    "CatBoost": (
        CURRENT5 / "experiments"
        / "catboost_20261027_034931_720016"
        / "metadata.json"
    )
}

saved5 = {}

for name, path in SOURCES5.items():

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {name}: {path}"
        )

    with path.open("r", encoding="utf-8") as f:
        saved5[name] = json.load(f)

    if saved5[name].get("official_test_used") is not False:
        raise RuntimeError(
            f"{name}: TEST-use status not verified"
        )

MODELS5 = list(SOURCES5)

CLASSES5 = saved5["XGBoost V1"]["class_order"]

for name, key in [
    ("Weighted XGBoost V2", "classes"),
    ("Conflict-aware XGBoost", "classes"),
    ("CatBoost", "class_names")
]:

    if saved5[name][key] != CLASSES5:
        raise RuntimeError(
            f"{name}: class-order inconsistency"
        )

if len(CLASSES5) != 10:
    raise RuntimeError("Expected ten attack/normal classes")

# ======================================================================
# B. RETRIEVE AND CHECK SAVED AGGREGATE METRICS
# ======================================================================

def aggregate_metrics5(model):

    data = saved5[model]

    if model == "XGBoost V1":

        return {
            "Accuracy": float(data["accuracy"]),
            "Macro_F1": float(data["macro_f1"]),
            "Balanced_Accuracy": float(
                data["balanced_accuracy"]
            ),
            "Weighted_F1": float(data["weighted_f1"])
        }

    container = (
        data["metrics"]
        if model == "CatBoost"
        else data["validation_metrics"]
    )

    return {
        "Accuracy": float(container["Accuracy"]),
        "Macro_F1": float(container["Macro F1"]),
        "Balanced_Accuracy": float(
            container["Balanced accuracy"]
        ),
        "Weighted_F1": float(
            container["Weighted F1"]
        )
    }

aggregate5 = {
    model: aggregate_metrics5(model)
    for model in MODELS5
}

# ======================================================================
# C. INDEPENDENT V1 PER-CLASS VERIFICATION
# ======================================================================

cm5 = np.asarray(
    saved5["XGBoost V1"]["confusion_matrix"],
    dtype=int
)

if cm5.shape != (10, 10):
    raise RuntimeError("Invalid V1 confusion matrix")

if cm5.sum() != 35167:
    raise RuntimeError("Unexpected V1 validation support")

if len(cm5) != len(CLASSES5):
    raise RuntimeError("Confusion matrix class mismatch")

def per_class5(class_name):

    idx = CLASSES5.index(class_name)

    tp = float(cm5[idx, idx])
    support = float(cm5[idx, :].sum())
    predicted = float(cm5[:, idx].sum())

    precision = tp/predicted if predicted else 0.0
    recall = tp/support if support else 0.0

    f1 = (
        2*precision*recall/(precision+recall)
        if precision+recall else 0.0
    )

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

dos_v1_5 = per_class5("DoS")
exploits_v1_5 = per_class5("Exploits")

# ======================================================================
# D. PREVIOUSLY REPORTED PER-CLASS RESULTS
#
# These values are 4-decimal summaries from:
#   V2:       Cell 22
#   Conflict: Cell 25
#   CatBoost: Cell 18
#
# They are NOT implied to be freshly recomputed here.
# ======================================================================

PRIOR_PER_CLASS5 = {
    "Weighted XGBoost V2": {
        "DoS_Recall": 0.6391,
        "DoS_F1": 0.4645,
        "Exploits_Recall": 0.6646,
        "source_cell": 22
    },
    "Conflict-aware XGBoost": {
        "DoS_Recall": 0.0823,
        "DoS_F1": 0.1443,
        "Exploits_Recall": 0.9422,
        "source_cell": 25
    },
    "CatBoost": {
        "DoS_Recall": 0.1046,
        "DoS_F1": 0.1704,
        "Exploits_Recall": 0.9227,
        "source_cell": 18
    }
}

# Verify the original report against V1's saved predictions.
np.testing.assert_allclose(
    [
        dos_v1_5["Recall"],
        dos_v1_5["F1"],
        exploits_v1_5["Recall"]
    ],
    [0.0603, 0.1113, 0.9497],
    atol=0.0001,
    rtol=0
)

rows5 = []

for model in MODELS5:

    rec = {
        "Model": model,
        **aggregate5[model]
    }

    if model == "XGBoost V1":

        rec.update({
            "DoS_Recall": dos_v1_5["Recall"],
            "DoS_F1": dos_v1_5["F1"],
            "Exploits_Recall": exploits_v1_5["Recall"],
            "Per_Class_Evidence": (
                "Recomputed from saved confusion matrix"
            )
        })

    else:

        prior = PRIOR_PER_CLASS5[model]

        rec.update({
            "DoS_Recall": prior["DoS_Recall"],
            "DoS_F1": prior["DoS_F1"],
            "Exploits_Recall": prior["Exploits_Recall"],
            "Per_Class_Evidence": (
                f"Rounded previously reported Cell "
                f"{prior['source_cell']} results"
            )
        })

    rows5.append(rec)

TABLE5 = pd.DataFrame(rows5)

NUMERIC5 = [
    "Accuracy",
    "Macro_F1",
    "Balanced_Accuracy",
    "Weighted_F1",
    "DoS_Recall",
    "DoS_F1",
    "Exploits_Recall"
]

if not np.isfinite(
    TABLE5[NUMERIC5].to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite classification metrics")

if not (
    TABLE5[NUMERIC5].ge(0).all().all()
    and TABLE5[NUMERIC5].le(1).all().all()
):
    raise RuntimeError("Metric outside [0,1]")

BEST5 = {
    metric: TABLE5[metric].max()
    for metric in NUMERIC5
}

print("\nA. FOUR-MODEL CLASSIFICATION BENCHMARK")
print("-" * 98)

print(
    TABLE5[
        ["Model"] + NUMERIC5
    ].round(5).to_string(index=False)
)

print("\nB. METRIC LEADERS")
print("-" * 98)

for metric in NUMERIC5:

    winners = TABLE5.loc[
        np.isclose(
            TABLE5[metric],
            BEST5[metric]
        ),
        "Model"
    ].tolist()

    print(
        f"{metric:<22}: "
        f"{', '.join(winners)} "
        f"({BEST5[metric]:.5f})"
    )

# ======================================================================
# E. PUBLICATION TABLE RENDERING
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 10,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig5, ax5 = plt.subplots(
    figsize=(16.6, 7.4),
    facecolor="white"
)

ax5.set_xlim(0, 1)
ax5.set_ylim(0, 1)
ax5.axis("off")

LEFT5, RIGHT5 = 0.055, 0.96

# Model + four overall metrics + two DoS + one Exploits
WIDTHS5 = [
    0.245,
    0.104, 0.104, 0.104, 0.104,
    0.098, 0.098,
    0.112
]

scale5 = (RIGHT5-LEFT5)/sum(WIDTHS5)

bounds5 = [LEFT5]

for w in WIDTHS5:
    bounds5.append(bounds5[-1] + w*scale5)

centers5 = [
    (bounds5[i] + bounds5[i+1])/2
    for i in range(len(WIDTHS5))
]

def line5(y, x1=LEFT5, x2=RIGHT5, lw=0.8):
    ax5.plot(
        [x1, x2], [y, y],
        color="#242424", lw=lw
    )

def text5(x, y, txt, **kwargs):
    ax5.text(
        x, y, str(txt),
        ha=kwargs.pop("ha", "center"),
        va="center",
        **kwargs
    )

text5(
    LEFT5, 0.960,
    "Table 4",
    ha="left",
    fontsize=14,
    fontweight="bold"
)

text5(
    LEFT5, 0.910,
    "Comparison of current network attack-type classifiers",
    ha="left",
    fontsize=13.3
)

line5(0.855, lw=1.2)

text5(
    centers5[0], 0.790,
    "Classifier",
    fontweight="bold",
    fontsize=10.5
)

for group_title, start, end in [
    ("Overall ten-class classification", 1, 4),
    ("DoS", 5, 6),
    ("Exploits", 7, 7)
]:

    midpoint = (
        bounds5[start] + bounds5[end+1]
    ) / 2

    text5(
        midpoint, 0.798,
        group_title,
        fontweight="bold",
        fontsize=10.5
    )

    line5(
        0.758,
        bounds5[start] + 0.005,
        bounds5[end+1] - 0.005,
        lw=0.65
    )

HEADERS5 = [
    "Accuracy",
    "Macro F1",
    "Balanced\naccuracy",
    "Weighted F1",
    "Recall",
    "F1",
    "Recall"
]

for idx, header in enumerate(HEADERS5, start=1):
    text5(
        centers5[idx], 0.708,
        header,
        fontsize=10.2
    )

line5(0.652, lw=1.0)

ROW_Y5 = [
    0.586,
    0.522,
    0.458,
    0.394
]

KEYS5 = NUMERIC5

for idx, record in enumerate(
    TABLE5.to_dict(orient="records")
):

    y = ROW_Y5[idx]

    text5(
        bounds5[0] + 0.007,
        y,
        record["Model"],
        ha="left",
        fontsize=10.4
    )

    for col_idx, metric in enumerate(KEYS5, start=1):

        value = float(record[metric])

        is_best = np.isclose(
            value,
            BEST5[metric],
            rtol=1e-10,
            atol=1e-10
        )

        # All metrics shown as fractions for consistency
        # with the forecasting tables.
        text5(
            centers5[col_idx],
            y,
            f"{value:.4f}",
            fontsize=10.5,
            fontweight="bold" if is_best else "normal"
        )

line5(0.341, lw=1.2)

# Research-paper notes — no tiny illegible footer.
ax5.text(
    LEFT5,
    0.285,
    "Notes: All metrics are fractions between 0 and 1. "
    "Bold values denote the largest observed value "
    "in each metric column.",
    ha="left",
    fontsize=9.5
)

ax5.text(
    LEFT5,
    0.241,
    "UNSW-NB15 development validation "
    "(35,167 records, ten classes). "
    "Macro F1 and balanced accuracy help expose "
    "poor performance on minority attack classes.",
    ha="left",
    fontsize=9.4
)

ax5.text(
    LEFT5,
    0.197,
    "Overall metrics come from saved model reports. "
    "Original XGBoost per-class scores were recalculated "
    "from its saved confusion matrix;",
    ha="left",
    fontsize=9.4
)

ax5.text(
    LEFT5,
    0.165,
    "other per-class values are rounded diagnostics "
    "transcribed from earlier experiment output. "
    "These values were not independently recomputed here.",
    ha="left",
    fontsize=9.4
)

ax5.text(
    LEFT5,
    0.120,
    "The development-validation fold was examined repeatedly; "
    "these comparisons do not represent an independent "
    "final TEST-set assessment.",
    ha="left",
    fontsize=9.3,
    color="#555555"
)

# ======================================================================
# F. SAVE VISUAL EXPORTS AND EXACT SOURCE DATA
# ======================================================================

stamp5 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

stem5 = (
    f"Table_04_Current_Attack_Classification_{stamp5}"
)

paths5 = {
    ext: DEST5 / f"{stem5}.{ext}"
    for ext in ["pdf", "svg", "png", "csv", "tex", "json"]
}

for path in paths5.values():
    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig5.savefig(
        paths5[ext],
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig5)

TABLE5.to_csv(
    paths5["csv"],
    index=False,
    mode="x",
    float_format="%.10f"
)

# ======================================================================
# G. EDITABLE LATEX BOOKTABS EXPORT
# ======================================================================

latex5 = [
    r"% Requires \usepackage{booktabs}",
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Comparison of current network "
    r"attack-type classifiers.}",
    r"\label{tab:current_attack_models}",
    r"\small",
    r"\begin{tabular}{lrrrrrrr}",
    r"\toprule",
    r"Classifier & "
    r"\multicolumn{4}{c}{Overall classification} & "
    r"\multicolumn{2}{c}{DoS} & "
    r"\multicolumn{1}{c}{Exploits} \\",
    r"\cmidrule(lr){2-5}"
    r"\cmidrule(lr){6-7}"
    r"\cmidrule(lr){8-8}",
    r" & Accuracy & Macro F1 & Balanced acc. & "
    r"Weighted F1 & Recall & F1 & Recall \\",
    r"\midrule"
]

for record in TABLE5.to_dict("records"):

    cells = [record["Model"]]

    for metric in NUMERIC5:

        value = float(record[metric])
        formatted = f"{value:.4f}"

        if np.isclose(
            value, BEST5[metric], atol=1e-10
        ):
            formatted = rf"\textbf{{{formatted}}}"

        cells.append(formatted)

    latex5.append(
        " & ".join(cells) + r" \\"
    )

latex5.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\vspace{2pt}",
    r"\parbox{\textwidth}{\footnotesize "
    r"All scores are fractions. "
    r"Bold entries are the largest observed "
    r"values in each column. "
    r"Overall results are from saved experimental "
    r"metadata. V1 class scores are recomputed "
    r"from its saved confusion matrix; "
    r"the remaining class scores are rounded "
    r"previously reported diagnostics. "
    r"UNSW-NB15 development validation: 35,167 records. "
    r"No independent TEST assessment is implied.}",
    r"\end{table*}"
])

with paths5["tex"].open(
    "x", encoding="utf-8"
) as f:
    f.write("\n".join(latex5) + "\n")

# ======================================================================
# H. SOURCE PROVENANCE AND OUTPUT HASHES
# ======================================================================

def sha5(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(1024*1024), b""
        ):
            h.update(block)

    return h.hexdigest()

generated5 = [
    paths5[ext]
    for ext in ["pdf", "svg", "png", "csv", "tex"]
]

manifest5 = {
    "project": "NetGuard",
    "publication_table": 4,
    "title": "Current network attack classification",
    "dataset": "UNSW-NB15 canonical TRAIN",
    "evaluation": "development validation",
    "validation_records": 35167,
    "classes": CLASSES5,
    "model_count": len(MODELS5),
    "aggregated_metrics": (
        "Directly loaded from saved metadata"
    ),
    "per_class_evidence": {
        "XGBoost V1": (
            "Calculated from saved confusion matrix"
        ),
        **{
            model: {
                "source_cell": info["source_cell"],
                "measurement_precision": (
                    "Rounded to four decimal places"
                ),
                "DoS_Recall": info["DoS_Recall"],
                "DoS_F1": info["DoS_F1"],
                "Exploits_Recall": info["Exploits_Recall"]
            }
            for model, info in PRIOR_PER_CLASS5.items()
        }
    },
    "sources": [
        {
            "model": model,
            "path": str(path.relative_to(ROOT5)),
            "sha256": sha5(path)
        }
        for model, path in SOURCES5.items()
    ],
    "generated_outputs": [
        {
            "path": str(path.relative_to(ROOT5)),
            "sha256": sha5(path)
        }
        for path in generated5
    ],
    "limitations": [
        "Repeatedly examined development validation.",
        "Per-class scores for experimental models "
        "were transcribed from earlier displayed results.",
        "Class scores are not all reproduced from saved "
        "prediction arrays in this table.",
        "Comparisons are descriptive.",
        "No independent official TEST assessment."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with paths5["json"].open(
    "x", encoding="utf-8"
) as f:
    json.dump(
        manifest5,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# I. FINAL VERIFICATION
# ======================================================================

print("\n" + "=" * 98)
print("CELL 5 COMPLETE — FINAL PAPER TABLE 4")
print("=" * 98)

print("Classifiers compared:", len(MODELS5))
print("Attack/normal classes:", len(CLASSES5))
print("Validation records:", 35167)
print("Overall metrics from saved metadata: PASS")
print("Original XGBoost class verification: PASS")
print("Other class metrics: PREVIOUSLY REPORTED / ROUNDED")

print("\nSAVED PUBLICATION ASSETS")
print("-" * 98)

for ext, path in paths5.items():

    assert path.is_file()
    assert path.stat().st_size > 0

    print(f"PASS [{ext.upper()}] {path.name}")

print("\nExisting research files overwritten: NO")
print("Existing publication files overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 98)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 6 — TABLE 5: BINARY CURRENT-ATTACK DETECTION
#
# Source: verified Research_Outputs from Cell 18
#
# A. Threshold-independent discrimination and calibration
# B. Five threshold-specific operating points
#
# Outputs:
#   PDF / SVG / PNG / CSV / TEX / JSON
#
# No model training.
# No dataset download.
# No official TEST access.
# No previous files overwritten.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 96)
print("CELL 6 — FINAL PAPER TABLE 5: BINARY ATTACK DETECTION")
print("=" * 96)

# ======================================================================
# A. EXACT SOURCE FILES
# ======================================================================

ROOT6 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DEST6 = (
    ROOT6 / "Research_Paper"
    / "02_Final_Tables"
)

SOURCES6 = {
    "metrics": (
        ROOT6 / "Research_Outputs"
        / "03_Metrics"
        / "binary_attack_detection_metrics_20261027_073004_429411.json"
    ),
    "thresholds": (
        ROOT6 / "Research_Outputs"
        / "02_Tables"
        / "T20_Binary_Threshold_Metrics_20261027_073004_429411.csv"
    ),
    "calibration": (
        ROOT6 / "Research_Outputs"
        / "02_Tables"
        / "T21_Binary_Calibration_Bins_20261027_073004_429411.csv"
    ),
    "original_v1": (
        ROOT6 / "Current_Attack_Detection"
        / "artifacts"
        / "xgboost_20261027_023158_885060"
        / "validation_metrics.json"
    )
}

if not DEST6.is_dir():
    raise FileNotFoundError(
        "Final tables folder missing. Run Cell 1 first."
    )

for key, path in SOURCES6.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing source {key}: {path}"
        )

with SOURCES6["metrics"].open(
    "r", encoding="utf-8"
) as f:
    metrics6 = json.load(f)

with SOURCES6["original_v1"].open(
    "r", encoding="utf-8"
) as f:
    original6 = json.load(f)

thresholds6 = pd.read_csv(
    SOURCES6["thresholds"]
)

calibration6 = pd.read_csv(
    SOURCES6["calibration"]
)

if metrics6.get("official_test_accessed") is not False:
    raise RuntimeError(
        "Saved F14 metrics do not verify TEST exclusion."
    )

if original6.get("official_test_used") is not False:
    raise RuntimeError(
        "Original V1 metrics do not verify TEST exclusion."
    )

# ======================================================================
# B. VALIDATE SOURCE SHAPES
# ======================================================================

REQUIRED_THRESH6 = {
    "Threshold",
    "Precision",
    "Recall",
    "FPR",
    "F1",
    "TP",
    "FP",
    "TN",
    "FN"
}

REQUIRED_CAL6 = {
    "Bin",
    "Lower",
    "Upper",
    "Count",
    "Mean_Predicted",
    "Observed_Attack_Fraction"
}

if not REQUIRED_THRESH6.issubset(thresholds6.columns):
    raise RuntimeError(
        "Saved threshold table has unexpected columns."
    )

if not REQUIRED_CAL6.issubset(calibration6.columns):
    raise RuntimeError(
        "Saved calibration table has unexpected columns."
    )

if len(thresholds6) != 5:
    raise RuntimeError("Expected five binary thresholds")

if len(calibration6) != 10:
    raise RuntimeError("Expected ten calibration bins")

thresholds6 = (
    thresholds6
    .sort_values("Threshold")
    .reset_index(drop=True)
)

calibration6 = (
    calibration6
    .sort_values("Bin")
    .reset_index(drop=True)
)

EXPECTED_THRESHOLDS6 = np.array([
    0.50, 0.70, 0.80, 0.90, 0.95
])

np.testing.assert_allclose(
    thresholds6["Threshold"].to_numpy(),
    EXPECTED_THRESHOLDS6,
    atol=1e-10,
    rtol=0
)

if list(calibration6["Bin"]) != list(range(10)):
    raise RuntimeError("Invalid calibration bin order")

if not np.isfinite(
    thresholds6[list(REQUIRED_THRESH6)]
    .to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite operating metrics")

if not np.isfinite(
    calibration6[list(REQUIRED_CAL6)]
    .to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite calibration metrics")

# ======================================================================
# C. VERIFY BINARY CLASS SUPPORT
# ======================================================================

N6 = int(metrics6["validation_rows"])
ATTACK6 = int(metrics6["positive_attack_rows"])
NORMAL6 = int(metrics6["normal_rows"])

if N6 != 35167:
    raise RuntimeError("Unexpected validation records")

if ATTACK6 != 23976:
    raise RuntimeError("Unexpected attack support")

if NORMAL6 != 11191:
    raise RuntimeError("Unexpected normal support")

if ATTACK6 + NORMAL6 != N6:
    raise RuntimeError("Binary class support mismatch")

original_cm6 = np.asarray(
    original6["confusion_matrix"],
    dtype=int
)

classes6 = original6["class_order"]

if original_cm6.shape != (10, 10):
    raise RuntimeError("Invalid original confusion matrix")

if original_cm6.sum() != N6:
    raise RuntimeError("Original validation size mismatch")

normal_idx6 = classes6.index("Normal")

if int(original_cm6[normal_idx6].sum()) != NORMAL6:
    raise RuntimeError(
        "Original normal-class support mismatch"
    )

np.testing.assert_allclose(
    ATTACK6 / N6,
    float(metrics6["attack_prevalence"]),
    atol=1e-10
)

# ======================================================================
# D. RECOMPUTE EVERY SAVED OPERATING METRIC
# ======================================================================

for row in thresholds6.itertuples():

    tp = int(row.TP)
    fp = int(row.FP)
    tn = int(row.TN)
    fn = int(row.FN)

    if min(tp, fp, tn, fn) < 0:
        raise RuntimeError("Negative confusion count")

    if tp + fp + tn + fn != N6:
        raise RuntimeError("Binary confusion size mismatch")

    if tp + fn != ATTACK6:
        raise RuntimeError("Positive-class support mismatch")

    if fp + tn != NORMAL6:
        raise RuntimeError("Normal-class support mismatch")

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    fpr = fp / (fp + tn)

    f1 = (
        2 * precision * recall / (precision + recall)
    )

    np.testing.assert_allclose(
        [precision, recall, fpr, f1],
        [
            row.Precision,
            row.Recall,
            row.FPR,
            row.F1
        ],
        atol=2e-9,
        rtol=0
    )

# Original Cell 10 threshold 0.50 operating point
first6 = thresholds6.iloc[0]

assert int(first6["TP"]) == 23355
assert int(first6["FP"]) == 973
assert int(first6["TN"]) == 10218
assert int(first6["FN"]) == 621

# Increasing the threshold should not generate
# more positive predictions or fewer missed attacks.
assert thresholds6["FP"].is_monotonic_decreasing
assert thresholds6["FN"].is_monotonic_increasing

# Also verify against the metrics JSON's own
# threshold records, independently of the CSV.
json_thresholds6 = metrics6.get("thresholds", [])

if len(json_thresholds6) != 5:
    raise RuntimeError(
        "Saved metric JSON missing five operating points"
    )

for csv_row, json_row in zip(
    thresholds6.to_dict("records"),
    sorted(
        json_thresholds6,
        key=lambda x: float(x["Threshold"])
    )
):

    for key in [
        "Threshold", "Precision", "Recall",
        "FPR", "F1", "TP", "FP", "TN", "FN"
    ]:

        np.testing.assert_allclose(
            float(csv_row[key]),
            float(json_row[key]),
            atol=2e-9,
            rtol=0
        )

print("\nA. BINARY VALIDATION VERIFICATION")
print("-" * 96)

print("Development validation records:", N6)
print("Attack records:", ATTACK6)
print("Normal records:", NORMAL6)
print("Thresholds:", list(EXPECTED_THRESHOLDS6))
print("All five binary confusion matrices: PASS")
print("Source CSV / metrics JSON agreement: PASS")

# ======================================================================
# E. VERIFY CALIBRATION METRICS
# ======================================================================

assert int(calibration6["Count"].sum()) == N6

ECE6 = float(
    np.sum(
        calibration6["Count"].to_numpy()
        / N6
        * np.abs(
            calibration6["Mean_Predicted"].to_numpy()
            - calibration6[
                "Observed_Attack_Fraction"
            ].to_numpy()
        )
    )
)

np.testing.assert_allclose(
    ECE6,
    float(metrics6["ece_10_bins"]),
    atol=2e-9,
    rtol=0
)

AP6 = float(metrics6["average_precision"])
AUC6 = float(metrics6["roc_auc"])
BRIER6 = float(metrics6["brier_score"])
PREVALENCE6 = float(metrics6["attack_prevalence"])

for value in [
    AP6, AUC6, BRIER6, ECE6, PREVALENCE6
]:
    if not 0 <= value <= 1:
        raise RuntimeError("Invalid saved binary metric")

np.testing.assert_allclose(
    [AP6, AUC6, BRIER6, ECE6],
    [0.996673, 0.992924, 0.030875, 0.005499],
    atol=0.000002,
    rtol=0
)

print("\nB. THRESHOLD-INDEPENDENT PERFORMANCE")
print("-" * 96)

print(f"Average precision: {AP6:.6f}")
print(f"ROC-AUC:           {AUC6:.6f}")
print(f"Brier score:       {BRIER6:.6f}")
print(f"10-bin ECE:        {ECE6:.6f}")
print(f"Attack prevalence: {PREVALENCE6:.6f}")
print("Calibration source verification: PASS")

# ======================================================================
# F. BUILD EDITABLE FINAL TABLE
# ======================================================================

TABLE6 = pd.DataFrame({
    "Threshold": thresholds6["Threshold"].astype(float),
    "Attack_Precision": thresholds6["Precision"].astype(float),
    "Attack_Recall": thresholds6["Recall"].astype(float),
    "Binary_F1": thresholds6["F1"].astype(float),
    "False_Positive_Rate": thresholds6["FPR"].astype(float),
    "False_Positive_Count": thresholds6["FP"].astype(int),
    "Missed_Attack_Count": thresholds6["FN"].astype(int),
    "True_Positive_Count": thresholds6["TP"].astype(int),
    "True_Negative_Count": thresholds6["TN"].astype(int)
})

print("\nC. PUBLICATION OPERATING POINTS")
print("-" * 96)

print(
    TABLE6.round(5).to_string(index=False)
)

# ======================================================================
# G. BLACK-AND-WHITE JOURNAL TABLE
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 10,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig6, ax6 = plt.subplots(
    figsize=(15, 7.5),
    facecolor="white"
)

ax6.set_xlim(0, 1)
ax6.set_ylim(0, 1)
ax6.axis("off")

LEFT6 = 0.06
RIGHT6 = 0.95

def rule6(y, x1=LEFT6, x2=RIGHT6, lw=0.8):

    ax6.plot(
        [x1, x2],
        [y, y],
        color="#242424",
        linewidth=lw
    )

def label6(x, y, value, **kwargs):

    ax6.text(
        x, y,
        str(value),
        ha=kwargs.pop("ha", "center"),
        va=kwargs.pop("va", "center"),
        **kwargs
    )

label6(
    LEFT6, 0.962,
    "Table 5",
    ha="left",
    fontsize=14,
    fontweight="bold"
)

label6(
    LEFT6, 0.916,
    "Binary attack-detection performance "
    "and decision-threshold characteristics",
    ha="left",
    fontsize=13.2
)

rule6(0.867, lw=1.2)

# ----------------------------------------------------------------------
# Upper section: threshold-independent metrics
# ----------------------------------------------------------------------

label6(
    LEFT6, 0.825,
    "A. Threshold-independent discrimination "
    "and probability calibration",
    ha="left",
    fontsize=10.7,
    fontweight="bold"
)

summary_headers6 = [
    "Attack AP",
    "ROC-AUC",
    "Brier score",
    "ECE (10 bins)",
    "Attack prevalence"
]

summary_values6 = [
    AP6, AUC6, BRIER6, ECE6, PREVALENCE6
]

summary_left6 = LEFT6 + 0.005
summary_right6 = RIGHT6 - 0.005

summary_centers6 = np.linspace(
    summary_left6 + 0.09,
    summary_right6 - 0.09,
    len(summary_headers6)
)

for x, header, value in zip(
    summary_centers6,
    summary_headers6,
    summary_values6
):

    label6(
        x, 0.763,
        header,
        fontsize=10.4
    )

    label6(
        x, 0.704,
        f"{value:.4f}",
        fontsize=11.6,
        fontweight="bold"
    )

rule6(0.656, lw=0.75)

# ----------------------------------------------------------------------
# Lower section: operating thresholds
# ----------------------------------------------------------------------

label6(
    LEFT6, 0.617,
    "B. Operating characteristics at fixed attack thresholds",
    ha="left",
    fontsize=10.7,
    fontweight="bold"
)

col_headers6 = [
    "Threshold",
    "Precision (%)",
    "Recall (%)",
    "Binary F1 (%)",
    "FPR (%)",
    "False positives",
    "Missed attacks"
]

WIDTHS6 = [
    0.125,
    0.145,
    0.145,
    0.155,
    0.115,
    0.155,
    0.160
]

scale6 = (
    RIGHT6 - LEFT6
) / sum(WIDTHS6)

bounds6 = [LEFT6]

for w in WIDTHS6:
    bounds6.append(
        bounds6[-1] + scale6*w
    )

centers6 = [
    (bounds6[i] + bounds6[i+1])/2
    for i in range(len(WIDTHS6))
]

for x, heading in zip(centers6, col_headers6):

    label6(
        x, 0.560,
        heading,
        fontsize=10
    )

rule6(0.521, lw=0.95)

row_positions6 = [
    0.473, 0.425, 0.377, 0.329, 0.281
]

for y, row in zip(
    row_positions6,
    TABLE6.to_dict("records")
):

    values = [
        f"{row['Threshold']:.2f}",
        f"{row['Attack_Precision']*100:.2f}",
        f"{row['Attack_Recall']*100:.2f}",
        f"{row['Binary_F1']*100:.2f}",
        f"{row['False_Positive_Rate']*100:.2f}",
        f"{row['False_Positive_Count']:,}",
        f"{row['Missed_Attack_Count']:,}"
    ]

    for x, value in zip(centers6, values):

        label6(
            x, y,
            value,
            fontsize=10.8
        )

rule6(0.245, lw=1.2)

# ----------------------------------------------------------------------
# Scientific notes
# ----------------------------------------------------------------------

ax6.text(
    LEFT6, 0.197,
    "Notes: AP = average precision; "
    "ROC-AUC = area under the ROC curve; "
    "ECE = expected calibration error; "
    "FPR = false-positive rate.",
    ha="left",
    fontsize=9.25
)

ax6.text(
    LEFT6, 0.164,
    "Threshold-specific measurements classify Attack "
    "when 1 - P(Normal) is at least the stated threshold. "
    "Counts refer to network-flow records.",
    ha="left",
    fontsize=9.25
)

ax6.text(
    LEFT6, 0.131,
    "UNSW-NB15 development validation: "
    "35,167 records (23,976 Attack; 11,191 Normal). "
    "Scores were reproduced from the saved XGBoost V1 classifier.",
    ha="left",
    fontsize=9.25
)

ax6.text(
    LEFT6, 0.098,
    "Development validation was repeatedly examined; "
    "the results do not estimate independent TEST-set "
    "performance or real-world deployment reliability.",
    ha="left",
    fontsize=9.2,
    color="#565656"
)

# ======================================================================
# H. SAVE ALL SIX PUBLICATION ASSETS
# ======================================================================

STAMP6 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM6 = (
    f"Table_05_Binary_Attack_Detection_{STAMP6}"
)

PATHS6 = {
    ext: DEST6 / f"{STEM6}.{ext}"
    for ext in [
        "pdf", "svg", "png", "csv", "tex", "json"
    ]
}

for path in PATHS6.values():

    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig6.savefig(
        PATHS6[ext],
        dpi=450 if ext == "png" else 160,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig6)

# The CSV includes exact underlying threshold values
# plus the threshold-independent summary as constant columns.

CSV6 = TABLE6.copy()

CSV6.insert(
    0, "Validation_Records", N6
)
CSV6["Average_Precision"] = AP6
CSV6["ROC_AUC"] = AUC6
CSV6["Brier_Score"] = BRIER6
CSV6["ECE_10_Bins"] = ECE6
CSV6["Attack_Prevalence"] = PREVALENCE6

CSV6.to_csv(
    PATHS6["csv"],
    index=False,
    mode="x",
    float_format="%.10f"
)

# ======================================================================
# I. LATEX BOOKTABS TABLE
# ======================================================================

latex6 = [
    r"% Requires \usepackage{booktabs}",
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Binary attack-detection performance "
    r"and decision-threshold characteristics.}",
    r"\label{tab:binary_attack_detection}",
    r"\small",
    r"\begin{tabular}{lrrrrrr}",
    r"\toprule",
    r"\multicolumn{7}{l}{\textbf{A. Threshold-independent "
    r"discrimination and calibration}} \\",
    r"\midrule",
    r"Metric & AP & ROC-AUC & Brier & ECE & "
    r"Attack prevalence & Records \\",
    (
        f"Value & {AP6:.4f} & {AUC6:.4f}"
        f" & {BRIER6:.4f} & {ECE6:.4f}"
        f" & {PREVALENCE6:.4f} & {N6:,}"
        + r" \\"
    ),
    r"\midrule",
    r"\multicolumn{7}{l}{\textbf{B. "
    r"Fixed decision thresholds}} \\",
    r"\midrule",
    r"Threshold & Precision (\%) & Recall (\%) & "
    r"F1 (\%) & FPR (\%) & FP & FN \\",
    r"\midrule"
]

for row in TABLE6.to_dict("records"):

    cells = [
        f"{row['Threshold']:.2f}",
        f"{row['Attack_Precision']*100:.2f}",
        f"{row['Attack_Recall']*100:.2f}",
        f"{row['Binary_F1']*100:.2f}",
        f"{row['False_Positive_Rate']*100:.2f}",
        str(row["False_Positive_Count"]),
        str(row["Missed_Attack_Count"])
    ]

    latex6.append(
        " & ".join(cells) + r" \\"
    )

latex6.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\vspace{3pt}",
    r"\parbox{\textwidth}{\footnotesize "
    r"AP = average precision; "
    r"ROC-AUC = area under the ROC curve; "
    r"ECE = expected calibration error; "
    r"FPR = false-positive rate; "
    r"FP = false positives; FN = false negatives. "
    r"Attack probability is $1-P(\mathrm{Normal})$. "
    r"Results reflect 35,167 UNSW-NB15 "
    r"development-validation records, "
    r"not independent final TEST performance.}",
    r"\end{table*}"
])

with PATHS6["tex"].open(
    "x", encoding="utf-8"
) as f:

    f.write("\n".join(latex6) + "\n")

# ======================================================================
# J. SAVE PROVENANCE MANIFEST
# ======================================================================

def sha6(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024*1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

GENERATED6 = [
    PATHS6[ext]
    for ext in ["pdf", "svg", "png", "csv", "tex"]
]

MANIFEST6 = {
    "project": "NetGuard",
    "publication_table": 5,
    "title": "Binary attack-detection reliability",
    "dataset": "UNSW-NB15 canonical TRAIN",
    "evaluation": "reproduced development validation",
    "classifier": "Saved XGBoost V1",
    "validation_records": N6,
    "attack_records": ATTACK6,
    "normal_records": NORMAL6,
    "summary_metrics": {
        "average_precision": AP6,
        "roc_auc": AUC6,
        "brier_score": BRIER6,
        "ece_10_bins": ECE6,
        "attack_prevalence": PREVALENCE6
    },
    "operating_thresholds": (
        TABLE6.to_dict("records")
    ),
    "all_confusion_counts_verified": True,
    "source_files": [
        {
            "key": key,
            "path": str(path.relative_to(ROOT6)),
            "sha256": sha6(path)
        }
        for key, path in SOURCES6.items()
    ],
    "generated_files": [
        {
            "path": str(path.relative_to(ROOT6)),
            "sha256": sha6(path)
        }
        for path in GENERATED6
    ],
    "limitations": [
        "Repeated use of development validation.",
        "No independent TEST-set evaluation.",
        "No live-network deployment.",
        "Reported ECE uses ten fixed probability bins.",
        "Binary attack detection differs from "
        "ten-class attack classification.",
        "No confidence intervals are calculated."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS6["json"].open(
    "x", encoding="utf-8"
) as f:

    json.dump(
        MANIFEST6,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# K. COMPLETE
# ======================================================================

print("\n" + "=" * 96)
print("CELL 6 COMPLETE — FINAL PAPER TABLE 5")
print("=" * 96)

print("Validation records:", N6)
print("Attack records:", ATTACK6)
print("Normal records:", NORMAL6)
print("Operating thresholds:", len(TABLE6))
print("Original source verification: PASS")
print("Calibration verification: PASS")

print("\nSAVED PUBLICATION ASSETS")
print("-" * 96)

for ext, path in PATHS6.items():

    assert path.is_file()
    assert path.stat().st_size > 0

    print(f"PASS [{ext.upper()}] {path.name}")

print("\nFINAL TABLE COLLECTION: 5 / 5")
print("Original research outputs modified: NO")
print("Previous paper tables overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 96)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 7 — FIGURE 1: CONDENSED TEMPORAL FORECASTING
#
# 3 attack families x 4 columns = 12 panels
#
# Columns:
#   1. Synthetic traffic rate
#   2. Stacked LSTM forecasts
#   3. Competing-Risk LSTM forecasts
#   4. Competing-Risk TCN forecasts
#
# Forecast horizons: H5, H10, H15
#
# Original source:
# Research_Outputs/02_Tables/
# T09_F08_Underlying_Plot_Data_20261027_063921_411841.csv
#
# Saves into:
# Research_Paper/01_Final_Figures/
#
# No model training
# No TEST access
# No old figures overwritten
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

print("=" * 100)
print("CELL 7 — PUBLICATION FIGURE 1: TEMPORAL ATTACK FORECASTING")
print("=" * 100)

# ======================================================================
# A. PROJECT AND OUTPUT DIRECTORIES
# ======================================================================

PROJECT7 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

SOURCE7 = (
    PROJECT7
    / "Research_Outputs"
    / "02_Tables"
    / "T09_F08_Underlying_Plot_Data_20261027_063921_411841.csv"
)

DEST7 = (
    PROJECT7
    / "Research_Paper"
    / "01_Final_Figures"
)

if not SOURCE7.is_file():
    raise FileNotFoundError(
        f"Validated F08 plot data missing: {SOURCE7}"
    )

if not DEST7.is_dir():
    raise FileNotFoundError(
        "Research_Paper/01_Final_Figures not found. "
        "Please run the directory setup cell."
    )

def sha256_7(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(block)
    return digest.hexdigest()


# Verify F08 plot-data integrity against its original manifest.
# This replaces the incorrectly hardcoded SHA256 value.

MANIFEST_SOURCE7 = (
    PROJECT7
    / "Research_Outputs"
    / "05_Manifests"
    / "traffic_forecasting_atlas_20261027_063921_411841.json"
)

if not MANIFEST_SOURCE7.is_file():
    raise FileNotFoundError(MANIFEST_SOURCE7)

with MANIFEST_SOURCE7.open("r", encoding="utf-8") as f:
    original_manifest7 = json.load(f)

matching_entries7 = [
    item
    for item in original_manifest7["generated_files"]
    if Path(item["path"]).name == SOURCE7.name
]

if len(matching_entries7) != 1:
    raise RuntimeError(
        "The original F08 manifest does not uniquely identify "
        "the underlying plot-data CSV."
    )

EXPECTED_SHA7 = matching_entries7[0]["sha256"]
actual_sha7 = sha256_7(SOURCE7)

print("Expected SHA256:", EXPECTED_SHA7)
print("Actual SHA256:  ", actual_sha7)

if actual_sha7 != EXPECTED_SHA7:
    raise RuntimeError(
        "F08 plot data genuinely differs from its saved manifest. "
        "Do not bypass this integrity check."
    )

print("F08 SOURCE INTEGRITY: PASS")


if actual_sha7 != EXPECTED_SHA7:
    raise RuntimeError(
        "F08 plot data does not match its original "
        "research manifest. Refusing to generate figure."
    )

print("\nA. ORIGINAL RESEARCH DATA")
print("-" * 100)

print("Source:", SOURCE7.name)
print("SHA256 verification: PASS")

# ======================================================================
# B. LOAD AND VALIDATE SAVED FIGURE DATA
# ======================================================================

data7 = pd.read_csv(SOURCE7)

EXPECTED_COLUMNS7 = [
    "Episode_ID",
    "Source",
    "Family",
    "Minute",
    "Packets_Per_Second",
    "Model",
    "Horizon",
    "Predicted_Probability",
    "True_Label"
]

missing7 = set(EXPECTED_COLUMNS7) - set(data7.columns)

if missing7:
    raise RuntimeError(
        f"Saved F08 data is missing: {sorted(missing7)}"
    )

if len(data7) != 3648:
    raise RuntimeError(
        f"Expected 3,648 F08 plot-data rows, "
        f"found {len(data7)}"
    )

for col in [
    "Minute",
    "Packets_Per_Second",
    "Predicted_Probability",
    "True_Label"
]:
    data7[col] = pd.to_numeric(
        data7[col],
        errors="raise"
    )

if not np.isfinite(
    data7[
        [
            "Minute",
            "Packets_Per_Second",
            "Predicted_Probability",
            "True_Label"
        ]
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite figure source values")

if not data7["Predicted_Probability"].between(0, 1).all():
    raise RuntimeError(
        "Forecast probability outside [0,1]"
    )

if not (data7["Packets_Per_Second"] > 0).all():
    raise RuntimeError(
        "Packet-rate values must be positive for log axis"
    )

if not data7["True_Label"].isin([0, 1]).all():
    raise RuntimeError("Invalid future-event target label")

if data7.duplicated([
    "Episode_ID",
    "Minute",
    "Model",
    "Horizon"
]).any():
    raise RuntimeError(
        "Duplicate episode/model/horizon/minute keys"
    )

# ======================================================================
# C. PRESELECTED VALIDATION EPISODES
#
# These are three previously audited positive episodes.
# We are not selecting episodes by model performance.
# ======================================================================

EPISODES7 = [
    {
        "id": "VALIDATION_A_BRUTE_AUTH_0000",
        "family": "BRUTE_AUTH",
        "source": "CIC_IDS2017",
        "display": "Brute force / authentication",
        "short": "BRUTE / AUTH",
        "windows": 72
    },
    {
        "id": "VALIDATION_A_DOS_DDOS_0001",
        "family": "DOS_DDOS",
        "source": "UNSW_NB15",
        "display": "Denial of service / DDoS",
        "short": "DoS / DDoS",
        "windows": 72
    },
    {
        "id": "VALIDATION_A_EXPLOIT_APP_0000",
        "family": "EXPLOIT_APP",
        "source": "CIC_IDS2017",
        "display": "Application exploitation",
        "short": "EXPLOIT / APP",
        "windows": 60
    }
]

MODELS7 = [
    "Stacked LSTM",
    "CR-LSTM",
    "CR-TCN"
]

HORIZONS7 = ["H5", "H10", "H15"]

EXPECTED_POSITIVES7 = {
    "H5": 5,
    "H10": 10,
    "H15": 15
}

selected_ids7 = [
    e["id"] for e in EPISODES7
]

selected7 = data7.loc[
    data7["Episode_ID"].isin(selected_ids7)
    & data7["Model"].isin(MODELS7)
].copy()

# Three models x three horizons:
# each decision minute occurs nine times.
expected_rows7 = (
    sum(e["windows"] for e in EPISODES7)
    * len(MODELS7)
    * len(HORIZONS7)
)

if len(selected7) != expected_rows7:
    raise RuntimeError(
        f"Expected {expected_rows7} selected rows; "
        f"found {len(selected7)}"
    )

TRAFFIC7 = {}
TARGETS7 = {}
PREDICTIONS7 = {}

print("\nB. EPISODE AND MODEL VALIDATION")
print("-" * 100)

for episode in EPISODES7:

    eid = episode["id"]

    frame = selected7.loc[
        selected7["Episode_ID"] == eid
    ].copy()

    if frame.empty:
        raise RuntimeError(
            f"Selected episode missing: {eid}"
        )

    if set(frame["Source"]) != {episode["source"]}:
        raise RuntimeError(
            f"Source mismatch for {eid}"
        )

    if set(frame["Family"]) != {episode["family"]}:
        raise RuntimeError(
            f"Family mismatch for {eid}"
        )

    minutes = np.sort(
        frame["Minute"].unique()
    )

    if len(minutes) != episode["windows"]:
        raise RuntimeError(
            f"Minute count mismatch for {eid}"
        )

    if not np.allclose(
        np.diff(minutes),
        1
    ):
        raise RuntimeError(
            f"Nonconsecutive decision minutes: {eid}"
        )

    # Validate that the traffic feature is identical
    # for every saved model/horizon at each minute.
    traffic_checks = frame.groupby(
        "Minute"
    )["Packets_Per_Second"].agg(
        ["min", "max", "size"]
    )

    if not (
        traffic_checks["size"] == 9
    ).all():
        raise RuntimeError(
            f"Traffic replication mismatch: {eid}"
        )

    if not np.allclose(
        traffic_checks["min"],
        traffic_checks["max"],
        rtol=1e-8,
        atol=1e-8
    ):
        raise RuntimeError(
            f"Traffic feature inconsistent across models: {eid}"
        )

    traffic = (
        frame[
            [
                "Minute",
                "Packets_Per_Second"
            ]
        ]
        .drop_duplicates("Minute")
        .sort_values("Minute")
    )

    if len(traffic) != episode["windows"]:
        raise RuntimeError(
            f"Traffic alignment failure: {eid}"
        )

    TRAFFIC7[eid] = traffic

    # Check every saved prediction sequence and
    # confirm ground-truth labels match across models.
    TARGETS7[eid] = {}
    PREDICTIONS7[eid] = {}

    for horizon in HORIZONS7:

        reference = None

        for model in MODELS7:

            subset = (
                frame.loc[
                    (frame["Model"] == model)
                    & (frame["Horizon"] == horizon)
                ]
                .sort_values("Minute")
            )

            if len(subset) != episode["windows"]:
                raise RuntimeError(
                    f"Incomplete series: {eid}/{model}/{horizon}"
                )

            np.testing.assert_array_equal(
                subset["Minute"].to_numpy(),
                minutes
            )

            y = subset["True_Label"].to_numpy(
                dtype=int
            )

            if reference is None:
                reference = y
            else:
                np.testing.assert_array_equal(
                    y, reference
                )

            PREDICTIONS7[eid][(model, horizon)] = (
                subset["Predicted_Probability"]
                .to_numpy(dtype=float)
            )

        if int(reference.sum()) != EXPECTED_POSITIVES7[horizon]:
            raise RuntimeError(
                f"Unexpected true-label support: {eid}/{horizon}"
            )

        TARGETS7[eid][horizon] = reference

    # Verify horizon targets are nested:
    # H5 positive => H10 positive => H15 positive.
    h5 = TARGETS7[eid]["H5"]
    h10 = TARGETS7[eid]["H10"]
    h15 = TARGETS7[eid]["H15"]

    if not (
        np.all(h5 <= h10)
        and np.all(h10 <= h15)
    ):
        raise RuntimeError(
            f"Future-event horizon labels inconsistent: {eid}"
        )

    print(
        f"PASS | {episode['family']:<13} | "
        f"{episode['source']:<12} | "
        f"{len(minutes):>3} minutes | "
        f"3 models × 3 horizons"
    )

print("\nSelected episodes:", len(EPISODES7))
print("Models:", len(MODELS7))
print("Forecast horizons:", HORIZONS7)
print("Aligned decision minutes:", sum(
    len(t) for t in TRAFFIC7.values()
))
print("Saved prediction records:", len(selected7))
print("Data alignment: PASS")

# ======================================================================
# D. PUBLICATION VISUAL DESIGN
# ======================================================================

COLORS7 = {
    "traffic": "#397FA8",
    "H5": "#5B76A6",
    "H10": "#168778",
    "H15": "#D46B3B",
    "ground_truth": "#E5BD71",
    "reference": "#A5ADB7"
}

STYLES7 = {
    "H5": (0, (2, 2)),
    "H10": "solid",
    "H15": (0, (6, 2.5))
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig7 = plt.figure(
    figsize=(17.0, 10.2),
    facecolor="white"
)

grid7 = fig7.add_gridspec(
    nrows=3,
    ncols=4,
    left=0.145,
    right=0.976,
    top=0.805,
    bottom=0.165,
    wspace=0.25,
    hspace=0.43,
    width_ratios=[1.20, 1, 1, 1]
)

AXES7 = np.empty((3, 4), dtype=object)

for r in range(3):
    for c in range(4):
        AXES7[r, c] = fig7.add_subplot(
            grid7[r, c]
        )

fig7.suptitle(
    "Network traffic dynamics and multi-horizon attack forecasting",
    x=0.055,
    y=0.977,
    ha="left",
    fontsize=18.5,
    fontweight="bold"
)

fig7.text(
    0.055,
    0.942,
    "Three preselected synthetic validation episodes  |  "
    "H5 / H10 / H15 forecasts  |  "
    "Sequence and competing-risk architectures",
    fontsize=10.8,
    color="#5B6878"
)

LEGEND7 = [
    Line2D(
        [0], [0],
        color=COLORS7["traffic"],
        linewidth=2,
        label="Synthetic packet rate"
    ),
    Line2D(
        [0], [0],
        color=COLORS7["H5"],
        linestyle=STYLES7["H5"],
        linewidth=2,
        label="H5: attack within 5 min"
    ),
    Line2D(
        [0], [0],
        color=COLORS7["H10"],
        linestyle=STYLES7["H10"],
        linewidth=2,
        label="H10: attack within 10 min"
    ),
    Line2D(
        [0], [0],
        color=COLORS7["H15"],
        linestyle=STYLES7["H15"],
        linewidth=2,
        label="H15: attack within 15 min"
    ),
    Patch(
        facecolor=COLORS7["ground_truth"],
        alpha=0.30,
        edgecolor="none",
        label="True H10-positive region"
    )
]

fig7.legend(
    handles=LEGEND7,
    loc="upper center",
    bbox_to_anchor=(0.51, 0.893),
    ncol=5,
    frameon=False,
    fontsize=9.1,
    handlelength=2.7,
    columnspacing=1.7
)

COLUMN_TITLES7 = [
    "Synthetic traffic rate",
    "Stacked LSTM",
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]

for col, title in enumerate(COLUMN_TITLES7):

    ax = AXES7[0, col]

    ax.set_title(
        f"{chr(65 + col)}   {title}",
        fontweight="bold",
        fontsize=11.3,
        pad=13,
        color="#202830"
    )

# ======================================================================
# E. DRAW ALL 12 PANELS
# ======================================================================

for row, episode in enumerate(EPISODES7):

    eid = episode["id"]

    traffic = TRAFFIC7[eid]

    x = traffic["Minute"].to_numpy(
        dtype=float
    )

    packets = traffic[
        "Packets_Per_Second"
    ].to_numpy(dtype=float)

    # H10 positive decision region is not the attack-onset
    # timestamp. We shade only where True_Label(H10)==1.
    h10 = TARGETS7[eid]["H10"]

    positive_x = x[h10 == 1]

    if len(positive_x) != 10:
        raise RuntimeError(
            f"Unexpected H10 positive count for {eid}"
        )

    region_start = float(positive_x.min()) - 0.5
    region_end = float(positive_x.max()) + 0.5

    # ------------------------------
    # Panel 1: Synthetic traffic
    # ------------------------------

    traffic_ax = AXES7[row, 0]

    traffic_ax.set_facecolor("#FAFBFC")

    traffic_ax.axvspan(
        region_start,
        region_end,
        color=COLORS7["ground_truth"],
        alpha=0.27,
        zorder=0
    )

    traffic_ax.plot(
        x,
        packets,
        color=COLORS7["traffic"],
        linewidth=1.9,
        zorder=3
    )

    traffic_ax.fill_between(
        x,
        packets,
        np.min(packets)*0.85,
        color=COLORS7["traffic"],
        alpha=0.09
    )

    traffic_ax.set_yscale("log")

    traffic_ax.set_ylabel(
        "Packets / second\n(log scale)",
        fontsize=9.4
    )

    traffic_ax.grid(
        linestyle=":",
        linewidth=0.55,
        color="#D5DAE2",
        alpha=0.8
    )

    traffic_ax.tick_params(length=3)

    traffic_ax.set_xlim(
        x.min() - 0.5,
        x.max() + 0.5
    )

    # ------------------------------
    # Panels 2–4: Forecasting models
    # ------------------------------

    for model_idx, model in enumerate(MODELS7):

        ax = AXES7[row, model_idx + 1]

        ax.set_facecolor("#FAFBFC")

        ax.axvspan(
            region_start,
            region_end,
            facecolor=COLORS7["ground_truth"],
            alpha=0.27,
            zorder=0
        )

        # A subtle reference at 50% predicted risk.
        # This is NOT a tuned deployment threshold.
        ax.axhline(
            0.5,
            color=COLORS7["reference"],
            linestyle=":",
            linewidth=0.8,
            alpha=0.65,
            zorder=1
        )

        for horizon in HORIZONS7:

            p = PREDICTIONS7[eid][
                (model, horizon)
            ]

            ax.plot(
                x,
                p,
                color=COLORS7[horizon],
                linestyle=STYLES7[horizon],
                linewidth=(
                    2.0 if horizon == "H10" else 1.7
                ),
                alpha=0.98,
                zorder=3 if horizon == "H10" else 2
            )

        ax.set_xlim(
            x.min() - 0.5,
            x.max() + 0.5
        )

        ax.set_ylim(-0.025, 1.035)

        ax.set_yticks([0, 0.5, 1])

        if model_idx == 0:
            ax.set_ylabel(
                "Forecast attack risk",
                fontsize=9.4
            )
        else:
            ax.tick_params(
                axis="y",
                labelleft=False
            )

        ax.grid(
            linestyle=":",
            linewidth=0.55,
            color="#D5DAE2",
            alpha=0.75
        )

        ax.tick_params(length=3)

    # Row labels outside plots, so titles do not overlap.
    reference_ax = AXES7[row, 0]
    center_y = (
        reference_ax.get_position().y0
        + reference_ax.get_position().height/2
    )

    fig7.text(
        0.012,
        center_y + 0.022,
        episode["short"],
        fontsize=11.5,
        fontweight="bold",
        ha="left",
        va="center",
        color="#263747"
    )

    source_label = (
        "CIC-IDS2017-derived"
        if episode["source"] == "CIC_IDS2017"
        else "UNSW-NB15-derived"
    )

    fig7.text(
        0.012,
        center_y - 0.010,
        source_label,
        fontsize=8.8,
        ha="left",
        va="center",
        color="#607083"
    )

    fig7.text(
        0.012,
        center_y - 0.035,
        f"n = {len(x)} minutes",
        fontsize=8.8,
        ha="left",
        va="center",
        color="#607083"
    )

    if row == 2:
        for col in range(4):
            AXES7[row, col].set_xlabel(
                "Decision minute",
                fontsize=9.5
            )

# ======================================================================
# F. SCIENTIFIC CAPTION AND LIMITATIONS
# ======================================================================

fig7.text(
    0.055,
    0.095,
    "Figure 1. Synthetic packet-rate trajectories and saved "
    "future-attack probability forecasts for three "
    "preselected validation episodes.",
    fontsize=9.2,
    color="#445266"
)

fig7.text(
    0.055,
    0.074,
    "Gold shading identifies decision minutes with a positive "
    "H10 forecast target; it does not identify exact attack "
    "onset. The dotted 0.5 risk line is illustrative.",
    fontsize=9.2,
    color="#445266"
)

fig7.text(
    0.055,
    0.053,
    "All curves are from aligned saved validation predictions. "
    "Examples are illustrative, not aggregate model "
    "performance or evidence of real-network deployment.",
    fontsize=9.2,
    color="#586779"
)

# ======================================================================
# G. SAVE VECTOR AND HIGH-RESOLUTION EXPORTS
# ======================================================================

STAMP7 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM7 = (
    f"Figure_01_Condensed_Temporal_Forecasting_{STAMP7}"
)

PATHS7 = {
    ext: DEST7 / f"{STEM7}.{ext}"
    for ext in ["pdf", "svg", "png", "csv", "json"]
}

for path in PATHS7.values():
    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig7.savefig(
        PATHS7[ext],
        dpi=450 if ext == "png" else 180,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig7)

# Save exact selected underlying plot data.
selected7.to_csv(
    PATHS7["csv"],
    index=False,
    mode="x",
    float_format="%.10f"
)

# ======================================================================
# H. SAVE REPRODUCIBILITY MANIFEST
# ======================================================================

GENERATED7 = [
    PATHS7[ext]
    for ext in ["pdf", "svg", "png", "csv"]
]

MANIFEST7 = {
    "project": "NetGuard",
    "publication_figure": 1,
    "title": (
        "Network traffic dynamics and "
        "multi-horizon attack forecasting"
    ),
    "dataset": "Synthetic NAF v2",
    "evaluation": "development validation",
    "figure_layout": "3 rows x 4 columns",
    "panels": 12,
    "models": MODELS7,
    "horizons": HORIZONS7,
    "episode_selection": (
        "Preselected family examples from earlier F08. "
        "Not selected by model performance."
    ),
    "episodes": [
        {
            "episode_id": item["id"],
            "family": item["family"],
            "source": item["source"],
            "aligned_minutes": item["windows"]
        }
        for item in EPISODES7
    ],
    "selected_decision_minutes": int(
        sum(len(t) for t in TRAFFIC7.values())
    ),
    "selected_prediction_rows": int(
        len(selected7)
    ),
    "source_file": {
        "path": str(
            SOURCE7.relative_to(PROJECT7)
        ),
        "sha256": actual_sha7
    },
    "generated_files": [
        {
            "path": str(
                p.relative_to(PROJECT7)
            ),
            "sha256": sha256_7(p)
        }
        for p in GENERATED7
    ],
    "limitations": [
        "Synthetic traffic signals, not real captured traces.",
        "Illustrative episodes, not aggregate evaluation.",
        "H10 ground-truth shading is not exact event onset.",
        "The 0.5 guide is not an optimized alert threshold.",
        "No independent TEST data was accessed.",
        "No forecasting models were retrained."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS7["json"].open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        MANIFEST7,
        handle,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# I. FINAL AUDIT
# ======================================================================

print("\n" + "=" * 100)
print("CELL 7 COMPLETE — FINAL PAPER FIGURE 1")
print("=" * 100)

print("Attack families:", len(EPISODES7))
print("Forecasting architectures:", len(MODELS7))
print("Forecast horizons:", len(HORIZONS7))
print("Figure panels:", 12)
print(
    "Aligned decision minutes:",
    sum(len(t) for t in TRAFFIC7.values())
)
print("Underlying prediction rows:", len(selected7))
print("Original F08 source SHA256: PASS")

print("\nSAVED PUBLICATION ASSETS")
print("-" * 100)

for ext, path in PATHS7.items():

    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        f"PASS [{ext.upper()}] {path.name}"
    )

print("\nFinal paper figures completed: 1 / 5")
print("Original research outputs modified: NO")
print("Existing publication assets overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 100)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 8 — FIGURE 2: MULTI-HORIZON PRECISION–RECALL BENCHMARK
#
# Three large panels: H5 / H10 / H15
# Seven architectures, 20 available PR curves
#
# Emphasis:
#   Stacked LSTM
#   Competing-Risk LSTM
#   GCN-LSTM (representative graph architecture)
#
# Other models remain visible as comparison curves.
#
# Saved F09 curve points are used without model inference.
# Every PR curve is checked against its saved AP.
#
# Saves PDF / SVG / PNG / CSV.GZ / JSON into:
# Research_Paper/01_Final_Figures/
#
# No retraining, no TEST access, no overwriting.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
from matplotlib.lines import Line2D
import hashlib
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 8 — FINAL PAPER FIGURE 2: PRECISION–RECALL COMPARISON")
print("=" * 100)

# ======================================================================
# A. LOCATE ORIGINAL VERIFIED RESEARCH OUTPUTS
# ======================================================================

ROOT8 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

RESEARCH8 = ROOT8 / "Research_Outputs"

DEST8 = (
    ROOT8 / "Research_Paper"
    / "01_Final_Figures"
)

CURVES_FILE8 = (
    RESEARCH8 / "02_Tables"
    / "T11_Precision_Recall_Curve_Points_"
      "20261027_064511_790770.csv.gz"
)

BENCHMARK_FILE8 = (
    RESEARCH8 / "02_Tables"
    / "T10_Seven_Model_AP_Benchmark_"
      "20261027_064511_790770.csv"
)

MANIFEST_FILE8 = (
    RESEARCH8 / "05_Manifests"
    / "precision_recall_manifest_"
      "20261027_064511_790770.json"
)

for path in [
    CURVES_FILE8,
    BENCHMARK_FILE8,
    MANIFEST_FILE8
]:
    if not path.is_file():
        raise FileNotFoundError(path)

if not DEST8.is_dir():
    raise FileNotFoundError(
        "Research_Paper/01_Final_Figures is missing"
    )

def sha256_8(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(block)
    return digest.hexdigest()

# Verify source files using their ORIGINAL F09 manifest.
with MANIFEST_FILE8.open("r", encoding="utf-8") as f:
    original_manifest8 = json.load(f)

if original_manifest8.get("official_test_accessed") is not False:
    raise RuntimeError("Original TEST-exclusion status missing")

manifest_entries8 = original_manifest8["generated_files"]

for path in [CURVES_FILE8, BENCHMARK_FILE8]:

    matches = [
        item for item in manifest_entries8
        if Path(item["path"]).name == path.name
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Missing or duplicate F09 manifest entry: {path.name}"
        )

    expected = matches[0]["sha256"]
    actual = sha256_8(path)

    if actual != expected:
        raise RuntimeError(
            f"Source integrity failure: {path.name}"
        )

print("\nA. ORIGINAL F09 SOURCE VERIFICATION")
print("-" * 100)
print("Curve-point CSV.GZ checksum: PASS")
print("AP benchmark CSV checksum: PASS")
print("Original manifest: VERIFIED")

# ======================================================================
# B. LOAD EXACT PR CURVES AND AP BENCHMARK
# ======================================================================

curves8 = pd.read_csv(
    CURVES_FILE8,
    compression="gzip"
)

benchmark8 = pd.read_csv(
    BENCHMARK_FILE8
)

EXPECTED_CURVE_COLUMNS8 = {
    "Model",
    "Horizon",
    "Point_Index",
    "Recall",
    "Precision"
}

EXPECTED_AP_COLUMNS8 = {
    "Model",
    "Horizon",
    "Average_Precision",
    "Positive_Windows",
    "Negative_Windows",
    "Prevalence",
    "Available"
}

if not EXPECTED_CURVE_COLUMNS8.issubset(curves8.columns):
    raise RuntimeError("Unexpected PR curve schema")

if not EXPECTED_AP_COLUMNS8.issubset(benchmark8.columns):
    raise RuntimeError("Unexpected AP benchmark schema")

if len(curves8) != 300281:
    raise RuntimeError(
        f"Expected 300,281 PR points; found {len(curves8)}"
    )

if len(benchmark8) != 21:
    raise RuntimeError("Expected 21 benchmark records")

for col in ["Point_Index", "Recall", "Precision"]:
    curves8[col] = pd.to_numeric(
        curves8[col],
        errors="raise"
    )

if not np.isfinite(
    curves8[["Recall", "Precision"]]
    .to_numpy(dtype=float)
).all():
    raise RuntimeError("Nonfinite PR curve values")

if not (
    curves8["Recall"].between(0, 1).all()
    and curves8["Precision"].between(0, 1).all()
):
    raise RuntimeError("PR curve value outside [0,1]")

MODELS8 = [
    "Single LSTM",
    "Stacked LSTM",
    "XGBoost V2",
    "CR-LSTM",
    "CR-TCN",
    "GCN-LSTM",
    "GAT-LSTM"
]

HORIZONS8 = ["H5", "H10", "H15"]

EXPECTED_SUPPORT8 = {
    "H5": 1125,
    "H10": 2250,
    "H15": 3375
}

TOTAL_WINDOWS8 = 18839

if set(curves8["Model"]) != set(MODELS8):
    raise RuntimeError("Unexpected model in curve archive")

if set(curves8["Horizon"]) != set(HORIZONS8):
    raise RuntimeError("Unexpected horizon in curve archive")

if benchmark8.duplicated(["Model", "Horizon"]).any():
    raise RuntimeError("Duplicate AP benchmark record")

# ======================================================================
# C. VERIFY EVERY CURVE AND RECOMPUTE ITS AP
#
# Saved curves have recall in ascending order.
# For that orientation, sklearn-equivalent AP is:
#
# AP = sum[(recall[i+1] - recall[i]) * precision[i+1]]
#
# This is NOT trapezoidal PR-AUC.
# ======================================================================

CURVE_DATA8 = {}
AP_VALUES8 = {}
PREVALENCE8 = {}

validation_rows8 = []

for horizon in HORIZONS8:

    expected_pos = EXPECTED_SUPPORT8[horizon]

    PREVALENCE8[horizon] = (
        expected_pos / TOTAL_WINDOWS8
    )

    for model in MODELS8:

        record = benchmark8.loc[
            (benchmark8["Model"] == model)
            & (benchmark8["Horizon"] == horizon)
        ]

        if len(record) != 1:
            raise RuntimeError(
                f"Missing benchmark: {model}/{horizon}"
            )

        record = record.iloc[0]

        if int(record["Positive_Windows"]) != expected_pos:
            raise RuntimeError("Positive support mismatch")

        if (
            int(record["Positive_Windows"])
            + int(record["Negative_Windows"])
            != TOTAL_WINDOWS8
        ):
            raise RuntimeError("Validation count mismatch")

        np.testing.assert_allclose(
            float(record["Prevalence"]),
            PREVALENCE8[horizon],
            atol=1e-7,
            rtol=0
        )

        available = (
            str(record["Available"]).lower() == "true"
        )

        subset = curves8.loc[
            (curves8["Model"] == model)
            & (curves8["Horizon"] == horizon)
        ].sort_values("Point_Index")

        if not available:

            if model != "XGBoost V2" or horizon != "H15":
                raise RuntimeError(
                    f"Unexpected missing curve: {model}/{horizon}"
                )

            if not subset.empty:
                raise RuntimeError(
                    "Unavailable XGBoost H15 has saved points"
                )

            AP_VALUES8[(model, horizon)] = np.nan
            continue

        if len(subset) < 2:
            raise RuntimeError(
                f"Incomplete curve: {model}/{horizon}"
            )

        if subset["Point_Index"].duplicated().any():
            raise RuntimeError("Duplicate curve point indices")

        recall = subset["Recall"].to_numpy(
            dtype=float
        )

        precision = subset["Precision"].to_numpy(
            dtype=float
        )

        if np.any(np.diff(recall) < -1e-9):
            raise RuntimeError(
                f"Nonmonotonic recall: {model}/{horizon}"
            )

        if not (
            np.isclose(recall[0], 0, atol=1e-8)
            and np.isclose(recall[-1], 1, atol=1e-8)
        ):
            raise RuntimeError(
                f"Curve endpoint mismatch: {model}/{horizon}"
            )

        np.testing.assert_allclose(
            precision[-1],
            PREVALENCE8[horizon],
            atol=1e-6,
            rtol=0
        )

        reconstructed_ap = float(
            np.sum(
                np.diff(recall) * precision[1:]
            )
        )

        reported_ap = float(
            record["Average_Precision"]
        )

        np.testing.assert_allclose(
            reconstructed_ap,
            reported_ap,
            atol=2e-6,
            rtol=0,
            err_msg=(
                f"AP mismatch for {model}/{horizon}"
            )
        )

        CURVE_DATA8[(model, horizon)] = (
            recall, precision
        )

        AP_VALUES8[(model, horizon)] = reported_ap

        validation_rows8.append({
            "Model": model,
            "Horizon": horizon,
            "Saved_AP": reported_ap,
            "Reconstructed_AP": reconstructed_ap,
            "Curve_Points": int(len(subset))
        })

VALIDATION8 = pd.DataFrame(validation_rows8)

if len(CURVE_DATA8) != 20:
    raise RuntimeError(
        "Expected exactly 20 available model/horizon curves"
    )

if sum(len(r) for r, p in CURVE_DATA8.values()) != len(curves8):
    raise RuntimeError("Curve-point accounting mismatch")

print("\nB. FULL PR CURVE VERIFICATION")
print("-" * 100)

print("Aligned validation windows:", TOTAL_WINDOWS8)
print("Models:", len(MODELS8))
print("Available PR curves:", len(CURVE_DATA8))
print("Total PR points:", len(curves8))
print("All saved AP values reconstructed: PASS")
print("XGBoost H15: N/A")

print("\nHORIZON-WISE AVERAGE PRECISION")
print("-" * 100)

print(
    benchmark8.pivot(
        index="Model",
        columns="Horizon",
        values="Average_Precision"
    ).reindex(MODELS8).round(4).to_string()
)

# ======================================================================
# D. PUBLICATION FIGURE STYLE
# ======================================================================

COLORS8 = {
    "Stacked LSTM": "#264F91",
    "CR-LSTM": "#168878",
    "GCN-LSTM": "#708B43",

    "Single LSTM": "#6684A6",
    "XGBoost V2": "#C98542",
    "CR-TCN": "#9562A7",
    "GAT-LSTM": "#CB6A81"
}

HIGHLIGHT8 = [
    "Stacked LSTM",
    "CR-LSTM",
    "GCN-LSTM"
]

BACKGROUND8 = [
    "Single LSTM",
    "XGBoost V2",
    "CR-TCN",
    "GAT-LSTM"
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white",
    "path.simplify": True,
    "path.simplify_threshold": 0.1
})

fig8, axes8 = plt.subplots(
    1, 3,
    figsize=(17.2, 7.0),
    sharex=True,
    sharey=True,
    facecolor="white"
)

fig8.subplots_adjust(
    left=0.075,
    right=0.975,
    top=0.73,
    bottom=0.24,
    wspace=0.17
)

fig8.suptitle(
    "Precision–recall discrimination across forecasting horizons",
    x=0.075,
    y=0.978,
    ha="left",
    fontsize=18.0,
    fontweight="bold"
)

fig8.text(
    0.075,
    0.929,
    "Seven forecasting architectures  |  "
    "18,839 aligned synthetic validation windows  |  "
    "Exact saved PR curves",
    fontsize=10.7,
    color="#607084"
)

# Shared legend; prominent lines are the focus models.
legend8 = []

for model in HIGHLIGHT8 + BACKGROUND8:

    legend8.append(
        Line2D(
            [0], [0],
            color=COLORS8[model],
            linewidth=(
                2.7 if model in HIGHLIGHT8 else 1.4
            ),
            alpha=(
                1.0 if model in HIGHLIGHT8 else 0.60
            ),
            label=model
        )
    )

legend8.append(
    Line2D(
        [0], [0],
        color="#697889",
        linestyle="--",
        linewidth=1.2,
        label="Class prevalence"
    )
)

fig8.legend(
    handles=legend8,
    loc="upper center",
    bbox_to_anchor=(0.51, 0.865),
    ncol=4,
    frameon=False,
    fontsize=9.4,
    columnspacing=1.65,
    handlelength=2.7
)

# ======================================================================
# E. RENDER THREE FULL PR PANELS
# ======================================================================

for idx, horizon in enumerate(HORIZONS8):

    ax = axes8[idx]

    ax.set_facecolor("#FAFBFC")
    ax.set_axisbelow(True)

    ax.grid(
        color="#CAD3DC",
        linestyle=":",
        linewidth=0.65,
        alpha=0.55
    )

    prevalence = PREVALENCE8[horizon]

    ax.axhline(
        prevalence,
        color="#697889",
        linestyle="--",
        linewidth=1.2,
        alpha=0.90,
        zorder=1
    )

    # Plot background references first.
    for model in BACKGROUND8:

        key = (model, horizon)

        if key not in CURVE_DATA8:
            continue

        recall, precision = CURVE_DATA8[key]

        ax.plot(
            recall,
            precision,
            color=COLORS8[model],
            linewidth=1.2,
            alpha=0.48,
            zorder=2,
            rasterized=False
        )

    # Overlay the three interpretable focus models.
    for model in HIGHLIGHT8:

        recall, precision = CURVE_DATA8[
            (model, horizon)
        ]

        ax.plot(
            recall,
            precision,
            color=COLORS8[model],
            linewidth=2.2,
            alpha=0.98,
            zorder=4
        )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.025)

    ax.set_xticks(
        np.linspace(0, 1, 6)
    )

    ax.set_yticks(
        np.linspace(0, 1, 6)
    )

    ax.set_xlabel(
        "Recall",
        labelpad=8
    )

    if idx == 0:
        ax.set_ylabel(
            "Precision",
            labelpad=8
        )

    positive = EXPECTED_SUPPORT8[horizon]

    ax.set_title(
        f"{chr(65 + idx)}    "
        f"{horizon} · {horizon[1:]}-minute prediction",
        loc="left",
        fontweight="bold",
        pad=15
    )

    # Display actual class support, avoiding any implication
    # that AP scores have identical positive prevalence.
    ax.text(
        0.03,
        0.045,
        (
            f"Positive windows: {positive:,}\n"
            f"Prevalence: {prevalence:.3f}"
        ),
        transform=ax.transAxes,
        fontsize=9.0,
        color="#4E5E70",
        va="bottom",
        ha="left",
        bbox={
            "facecolor": "white",
            "edgecolor": "#D7DEE6",
            "boxstyle": "round,pad=0.42",
            "alpha": 0.94
        },
        zorder=8
    )

    # Highlight horizon leader without pretending that
    # small differences imply statistical significance.
    available_models = [
        model
        for model in MODELS8
        if np.isfinite(
            AP_VALUES8[(model, horizon)]
        )
    ]

    leader = max(
        available_models,
        key=lambda model: AP_VALUES8[
            (model, horizon)
        ]
    )

    leader_ap = AP_VALUES8[
        (leader, horizon)
    ]

    ax.text(
        0.97,
        0.19,
        (
            f"Highest AP\n"
            f"{leader}: {leader_ap:.4f}"
        ),
        transform=ax.transAxes,
        fontsize=9.0,
        va="bottom",
        ha="right",
        color="#26384A",
        bbox={
            "facecolor": "white",
            "edgecolor": "#D7DEE6",
            "boxstyle": "round,pad=0.45",
            "alpha": 0.96
        },
        zorder=8
    )

    if horizon == "H15":

        ax.text(
            0.97,
            0.043,
            "XGBoost V2: H15 unavailable",
            ha="right",
            va="bottom",
            transform=ax.transAxes,
            fontsize=8.8,
            color="#A06334"
        )

# ======================================================================
# F. SCIENTIFIC FOOTNOTES
# ======================================================================

fig8.text(
    0.075,
    0.144,
    "Figure 2. Precision–recall curves for 5-, 10-, "
    "and 15-minute future-attack prediction. "
    "Stacked LSTM, CR-LSTM and GCN-LSTM are emphasized; "
    "other evaluated architectures remain visible.",
    fontsize=9.1,
    color="#4F5D70"
)

fig8.text(
    0.075,
    0.112,
    "Average precision is computed from the exact saved "
    "curves and differs from overall classification accuracy. "
    "Positive prevalence increases with forecast horizon.",
    fontsize=9.1,
    color="#4F5D70"
)

fig8.text(
    0.075,
    0.080,
    "Synthetic NAF v2 development validation. "
    "The other Graph LSTM variants are compared in Table 1 "
    "but have no curves in the F09 archive. "
    "No independent-window confidence intervals are implied.",
    fontsize=9.0,
    color="#5F6E80"
)

# ======================================================================
# G. EXPORT PUBLICATION FILES
# ======================================================================

STAMP8 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM8 = (
    f"Figure_02_Multi_Horizon_Precision_Recall_{STAMP8}"
)

PATHS8 = {
    "pdf": DEST8 / f"{STEM8}.pdf",
    "svg": DEST8 / f"{STEM8}.svg",
    "png": DEST8 / f"{STEM8}.png",
    "curves": DEST8 / f"{STEM8}_Curve_Data.csv.gz",
    "json": DEST8 / f"{STEM8}.json"
}

for path in PATHS8.values():

    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig8.savefig(
        PATHS8[ext],
        dpi=450 if ext == "png" else 170,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig8)

# Copy the original compressed curve points unchanged,
# retaining their complete precision and point ordering.
shutil.copyfile(
    CURVES_FILE8,
    PATHS8["curves"]
)

if sha256_8(PATHS8["curves"]) != sha256_8(CURVES_FILE8):

    raise RuntimeError(
        "Exported curve-data checksum mismatch"
    )

# ======================================================================
# H. RECORD REPRODUCIBILITY AND PROVENANCE
# ======================================================================

EXPORTS8 = [
    PATHS8["pdf"],
    PATHS8["svg"],
    PATHS8["png"],
    PATHS8["curves"]
]

manifest8 = {
    "project": "NetGuard",
    "publication_figure": 2,
    "title": (
        "Precision–recall discrimination "
        "across forecasting horizons"
    ),
    "evaluation": (
        "Synthetic NAF v2 development validation"
    ),
    "validation_windows": TOTAL_WINDOWS8,
    "model_count": len(MODELS8),
    "models": MODELS8,
    "emphasized_models": HIGHLIGHT8,
    "horizons": HORIZONS8,
    "available_curves": len(CURVE_DATA8),
    "total_curve_points": len(curves8),
    "missing_result": "XGBoost V2 H15",
    "all_curve_aps_verified": True,
    "average_precision": {
        model: {
            horizon: (
                float(AP_VALUES8[(model, horizon)])
                if np.isfinite(
                    AP_VALUES8[(model, horizon)]
                )
                else None
            )
            for horizon in HORIZONS8
        }
        for model in MODELS8
    },
    "source_files": [
        {
            "path": str(path.relative_to(ROOT8)),
            "sha256": sha256_8(path)
        }
        for path in [
            CURVES_FILE8,
            BENCHMARK_FILE8,
            MANIFEST_FILE8
        ]
    ],
    "generated_files": [
        {
            "path": str(path.relative_to(ROOT8)),
            "sha256": sha256_8(path)
        }
        for path in EXPORTS8
    ],
    "limitations": [
        "Synthetic development-validation data.",
        "Repeated, overlapping temporal windows.",
        "Seven available F09 architectures only.",
        "Original and optimized Graph LSTM curves "
        "are not included in F09.",
        "XGBoost H15 unavailable.",
        "No independent-window confidence intervals.",
        "No official TEST evaluation.",
        "No forecasting model retraining."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS8["json"].open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest8,
        handle,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# I. FINAL VERIFICATION
# ======================================================================

print("\n" + "=" * 100)
print("CELL 8 COMPLETE — FINAL PAPER FIGURE 2")
print("=" * 100)

print("Models evaluated:", len(MODELS8))
print("Horizons:", HORIZONS8)
print("PR curves:", len(CURVE_DATA8))
print("Total original PR points:", len(curves8))
print("Saved AP reconstruction: PASS")
print("Original F09 source hashes: PASS")

print("\nBEST MODEL PER HORIZON")
print("-" * 100)

for horizon in HORIZONS8:

    ranked = sorted(
        [
            (model, AP_VALUES8[(model, horizon)])
            for model in MODELS8
            if np.isfinite(
                AP_VALUES8[(model, horizon)]
            )
        ],
        key=lambda item: item[1],
        reverse=True
    )

    name, ap = ranked[0]

    print(
        f"{horizon:<4} | "
        f"{name:<16} | AP={ap:.6f}"
    )

print("\nSAVED PUBLICATION ASSETS")
print("-" * 100)

for label, path in PATHS8.items():

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Missing or empty output: {path}"
        )

    print(
        f"PASS [{label.upper()}] {path.name}"
    )

print("\nFinal paper figures completed: 2 / 5")
print("Original research outputs modified: NO")
print("Previous publication figures overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 100)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 9 — FIGURE 3: SOURCE-DEPENDENT ATTACK-FAMILY CONFUSION
#
# 2 rows (CIC-IDS2017 / UNSW-NB15)
# x 2 columns (CR-LSTM / CR-TCN)
#
# Uses verified Cell 8 source-specific confusion counts
# and corrected Cell 8B source-support rules.
#
# Outputs: PDF, SVG, PNG, CSV, JSON
#
# No training.
# No official TEST.
# No existing files overwritten.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

print("=" * 100)
print("CELL 9 — FINAL PAPER FIGURE 3: ATTACK-FAMILY CONFUSION")
print("=" * 100)

# ======================================================================
# A. SOURCE PATHS
# ======================================================================

ROOT9 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUTPUTS9 = ROOT9 / "Research_Outputs"

DEST9 = (
    ROOT9 / "Research_Paper"
    / "01_Final_Figures"
)

CONFUSION_FILE9 = (
    OUTPUTS9 / "02_Tables"
    / "T04_Source_Family_Confusion_20261027_061113_471823.csv"
)

RECALL_FILE9 = (
    OUTPUTS9 / "02_Tables"
    / "T05_Source_Family_Recall_20261027_061113_471823.csv"
)

SOURCE_MANIFEST9 = (
    OUTPUTS9 / "05_Manifests"
    / "family_confusion_research_manifest_20261027_061113_471823.json"
)

CORRECTED_MANIFEST9 = (
    OUTPUTS9 / "05_Manifests"
    / "corrected_confusion_manifest_20261027_061555_018961.json"
)

for path in [
    CONFUSION_FILE9,
    RECALL_FILE9,
    SOURCE_MANIFEST9,
    CORRECTED_MANIFEST9
]:
    if not path.is_file():
        raise FileNotFoundError(path)

if not DEST9.is_dir():
    raise FileNotFoundError(
        "Research_Paper/01_Final_Figures missing"
    )

def sha9(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

with SOURCE_MANIFEST9.open("r", encoding="utf-8") as f:
    source_manifest9 = json.load(f)

with CORRECTED_MANIFEST9.open("r", encoding="utf-8") as f:
    corrected_manifest9 = json.load(f)

if source_manifest9.get("official_test_used") is not False:
    raise RuntimeError("Original TEST status unverified")

if corrected_manifest9.get("official_test_used") is not False:
    raise RuntimeError("Corrected figure TEST status unverified")

# Validate exact source data against Cell 8 manifest.
for path in [CONFUSION_FILE9, RECALL_FILE9]:

    matches = [
        item
        for item in source_manifest9["generated_files"]
        if Path(item["path"]).name == path.name
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Missing source-manifest entry: {path.name}"
        )

    if sha9(path) != matches[0]["sha256"]:
        raise RuntimeError(
            f"Source checksum mismatch: {path.name}"
        )

print("\nA. ORIGINAL RESEARCH SOURCE VERIFICATION")
print("-" * 100)
print("Confusion data checksum: PASS")
print("Class-recall data checksum: PASS")
print("Corrected F04 manifest: FOUND")

# ======================================================================
# B. LOAD DATA
# ======================================================================

confusion9 = pd.read_csv(CONFUSION_FILE9)
recall9 = pd.read_csv(RECALL_FILE9)

REQUIRED_CM9 = {
    "Model",
    "Source",
    "True_Family",
    "Predicted_Family",
    "Count",
    "True_Class_Support",
    "Row_Fraction"
}

REQUIRED_REC9 = {
    "Source",
    "Family",
    "Support",
    "LSTM_Recall",
    "TCN_Recall",
    "TCN_Minus_LSTM"
}

if not REQUIRED_CM9.issubset(confusion9.columns):
    raise RuntimeError(
        "Unexpected saved confusion-data schema"
    )

if not REQUIRED_REC9.issubset(recall9.columns):
    raise RuntimeError(
        "Unexpected saved class-recall schema"
    )

MODELS9 = [
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
]

SOURCES9 = [
    "CIC_IDS2017",
    "UNSW_NB15"
]

FAMILIES9 = [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

DISPLAY_FAMILY9 = [
    "Brute/Auth",
    "DoS/DDoS",
    "Exploit/App",
    "Fuzzing",
    "Recon/Scan"
]

DISPLAY_SOURCE9 = {
    "CIC_IDS2017": "CIC-IDS2017-derived",
    "UNSW_NB15": "UNSW-NB15-derived"
}

DISPLAY_MODEL9 = {
    "Competing-Risk LSTM": "CR-LSTM",
    "Competing-Risk TCN": "CR-TCN"
}

EXPECTED_SOURCE_TOTALS9 = {
    "CIC_IDS2017": 1710,
    "UNSW_NB15": 1665
}

if len(confusion9) != 100:
    raise RuntimeError(
        "Expected four complete 5x5 confusion matrices"
    )

if len(recall9) != 8:
    raise RuntimeError(
        "Expected eight nonzero-support family recall rows"
    )

if set(confusion9["Model"]) != set(MODELS9):
    raise RuntimeError("Unexpected model identifiers")

if set(confusion9["Source"]) != set(SOURCES9):
    raise RuntimeError("Unexpected source identifiers")

if set(confusion9["True_Family"]) != set(FAMILIES9):
    raise RuntimeError("Unexpected true attack families")

if set(confusion9["Predicted_Family"]) != set(FAMILIES9):
    raise RuntimeError("Unexpected predicted attack families")

if confusion9.duplicated(
    ["Model", "Source", "True_Family", "Predicted_Family"]
).any():
    raise RuntimeError("Duplicate confusion-matrix cell")

for col in ["Count", "True_Class_Support"]:
    confusion9[col] = pd.to_numeric(
        confusion9[col],
        errors="raise"
    )

confusion9["Row_Fraction"] = pd.to_numeric(
    confusion9["Row_Fraction"],
    errors="coerce"
)

if (
    confusion9["Count"].isna().any()
    or confusion9["True_Class_Support"].isna().any()
):
    raise RuntimeError("Missing count or support")

if (
    (confusion9["Count"] < 0).any()
    or (confusion9["True_Class_Support"] < 0).any()
):
    raise RuntimeError("Negative confusion count")

if not np.allclose(
    confusion9["Count"],
    np.round(confusion9["Count"])
):
    raise RuntimeError("Noninteger confusion count")

# ======================================================================
# C. RECONSTRUCT AND VERIFY EACH MATRIX
# ======================================================================

EXPECTED_SUPPORT9 = corrected_manifest9[
    "source_specific_supports"
]

COUNTS9 = {}
FRACTIONS9 = {}
SUPPORTS9 = {}
RESULT_ROWS9 = []

for source in SOURCES9:

    manifest_support = np.asarray(
        EXPECTED_SUPPORT9[source],
        dtype=int
    )

    if len(manifest_support) != 5:
        raise RuntimeError("Invalid corrected support metadata")

    for model in MODELS9:

        subset = confusion9.loc[
            (confusion9["Source"] == source)
            & (confusion9["Model"] == model)
        ]

        if len(subset) != 25:
            raise RuntimeError(
                f"Incomplete matrix: {model}/{source}"
            )

        counts = (
            subset.pivot(
                index="True_Family",
                columns="Predicted_Family",
                values="Count"
            )
            .reindex(
                index=FAMILIES9,
                columns=FAMILIES9
            )
        )

        if counts.isna().any().any():
            raise RuntimeError("Missing confusion cell")

        counts = counts.to_numpy(dtype=int)

        support = counts.sum(axis=1)

        np.testing.assert_array_equal(
            support,
            manifest_support
        )

        if counts.sum() != EXPECTED_SOURCE_TOTALS9[source]:
            raise RuntimeError(
                f"Wrong source total: {model}/{source}"
            )

        fractions = np.full(
            (5, 5),
            np.nan,
            dtype=float
        )

        for i, family in enumerate(FAMILIES9):

            row = subset.loc[
                subset["True_Family"] == family
            ]

            if len(row) != 5:
                raise RuntimeError(
                    f"Incomplete class row: {family}"
                )

            if not (
                row["True_Class_Support"].to_numpy()
                == support[i]
            ).all():
                raise RuntimeError(
                    f"Source support mismatch: {family}"
                )

            if support[i] == 0:

                if counts[i].sum() != 0:
                    raise RuntimeError(
                        "Zero-support class has predictions"
                    )

                if row["Row_Fraction"].notna().any():
                    raise RuntimeError(
                        "Zero-support recall must be N/A"
                    )

            else:

                fractions[i] = counts[i] / support[i]

                for j, pred_family in enumerate(FAMILIES9):

                    cell = row.loc[
                        row["Predicted_Family"] == pred_family
                    ]

                    if len(cell) != 1:
                        raise RuntimeError(
                            "Duplicate or missing prediction cell"
                        )

                    np.testing.assert_allclose(
                        float(cell["Row_Fraction"].iloc[0]),
                        fractions[i, j],
                        atol=1e-12,
                        rtol=0
                    )

                recall_match = recall9.loc[
                    (recall9["Source"] == source)
                    & (recall9["Family"] == family)
                ]

                if len(recall_match) != 1:
                    raise RuntimeError(
                        f"Missing recall evidence: {source}/{family}"
                    )

                key = (
                    "LSTM_Recall"
                    if model == MODELS9[0]
                    else "TCN_Recall"
                )

                np.testing.assert_allclose(
                    fractions[i, i],
                    float(recall_match[key].iloc[0]),
                    atol=1e-12,
                    rtol=0
                )

        COUNTS9[(source, model)] = counts
        FRACTIONS9[(source, model)] = fractions
        SUPPORTS9[(source, model)] = support

        print(
            f"PASS | {DISPLAY_SOURCE9[source]:<22} "
            f"| {DISPLAY_MODEL9[model]:<7} "
            f"| n={counts.sum():,} "
            f"| zero-support rows={(support == 0).sum()}"
        )

print("\nB. MATRIX RECONSTRUCTION")
print("-" * 100)
print("Four source/model matrices: PASS")
print("All class supports: PASS")
print("All saved row fractions: PASS")
print("Independent class-recall table: PASS")

# ======================================================================
# D. VERIFY THE KEY EXPLOIT/APP FAILURE
# ======================================================================

EXPLOIT_INDEX9 = FAMILIES9.index("EXPLOIT_APP")
FUZZING_INDEX9 = FAMILIES9.index("FUZZING")

unsw_lstm9 = COUNTS9[
    ("UNSW_NB15", "Competing-Risk LSTM")
]

unsw_tcn9 = COUNTS9[
    ("UNSW_NB15", "Competing-Risk TCN")
]

exploit_support9 = int(
    unsw_tcn9[EXPLOIT_INDEX9].sum()
)

lstm_correct9 = int(
    unsw_lstm9[EXPLOIT_INDEX9, EXPLOIT_INDEX9]
)

tcn_correct9 = int(
    unsw_tcn9[EXPLOIT_INDEX9, EXPLOIT_INDEX9]
)

tcn_as_fuzzing9 = int(
    unsw_tcn9[EXPLOIT_INDEX9, FUZZING_INDEX9]
)

if (
    exploit_support9 != 330
    or lstm_correct9 != 48
    or tcn_correct9 != 4
    or tcn_as_fuzzing9 != 297
):
    raise RuntimeError(
        "UNSW Exploit/App diagnostic does not match Cell 8"
    )

print("\nC. KEY SOURCE-DEPENDENT FAILURE")
print("-" * 100)

print("UNSW Exploit/App true windows:", exploit_support9)
print("CR-LSTM correctly predicted:", lstm_correct9)
print("CR-TCN correctly predicted:", tcn_correct9)
print("CR-TCN predicted Fuzzing:", tcn_as_fuzzing9)

# ======================================================================
# E. CREATE THE FOUR-PANEL PUBLICATION FIGURE
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig9, axes9 = plt.subplots(
    2, 2,
    figsize=(15.2, 12.0),
    facecolor="white"
)

fig9.subplots_adjust(
    left=0.135,
    right=0.875,
    top=0.825,
    bottom=0.145,
    wspace=0.33,
    hspace=0.36
)

fig9.suptitle(
    "Source-dependent future attack-family classification",
    x=0.055,
    y=0.975,
    ha="left",
    fontsize=18.5,
    fontweight="bold"
)

fig9.text(
    0.055,
    0.940,
    "Conditional future-event classification  |  "
    "Two competing-risk models  |  "
    "Two synthetic validation source groups",
    fontsize=10.4,
    color="#617185"
)

# Shared 0-100% heatmap scale.
cmap9 = plt.get_cmap("YlGnBu").copy()
cmap9.set_bad("#E4E8EE")

norm9 = Normalize(vmin=0, vmax=1)

panel_letters9 = ["A", "B", "C", "D"]

for row, source in enumerate(SOURCES9):

    for col, model in enumerate(MODELS9):

        ax = axes9[row, col]

        counts = COUNTS9[(source, model)]
        fractions = FRACTIONS9[(source, model)]
        support = SUPPORTS9[(source, model)]

        masked = np.ma.masked_invalid(fractions)

        im9 = ax.imshow(
            masked,
            cmap=cmap9,
            norm=norm9,
            interpolation="nearest",
            aspect="equal"
        )

        panel_letter = panel_letters9[row*2 + col]

        ax.set_title(
            f"{panel_letter}    "
            f"{DISPLAY_MODEL9[model]}  |  "
            f"{DISPLAY_SOURCE9[source]}",
            loc="left",
            fontsize=11.0,
            fontweight="bold",
            pad=14
        )

        ax.set_xticks(range(5))
        ax.set_yticks(range(5))

        ax.set_xticklabels(
            DISPLAY_FAMILY9,
            rotation=35,
            ha="right",
            rotation_mode="anchor",
            fontsize=9.1
        )

        # Every subplot has its own class-support labels.
        # This avoids the original F04 y-axis support bug.
        ax.set_yticklabels([
            f"{family} (n={int(support[i])})"
            for i, family in enumerate(DISPLAY_FAMILY9)
        ], fontsize=9.0)

        ax.set_xlabel(
            "Predicted future attack family",
            fontsize=9.8,
            labelpad=7
        )

        ax.set_ylabel(
            "True future attack family",
            fontsize=9.8,
            labelpad=7
        )

        # Cell grid borders, without obscuring annotations.
        ax.set_xticks(
            np.arange(-0.5, 5, 1),
            minor=True
        )
        ax.set_yticks(
            np.arange(-0.5, 5, 1),
            minor=True
        )

        ax.grid(
            which="minor",
            color="white",
            linewidth=0.9,
            alpha=0.65
        )

        ax.tick_params(
            which="minor",
            bottom=False,
            left=False
        )

        ax.tick_params(length=0)

        # Annotate all observed nonzero cells with
        # true-row percentage and raw count.
        for i in range(5):
            for j in range(5):

                if support[i] == 0:

                    ax.text(
                        j, i, "N/A",
                        ha="center",
                        va="center",
                        fontsize=9.0,
                        color="#6F7C8B"
                    )

                    continue

                value = fractions[i, j]
                count = counts[i, j]

                if count == 0:
                    text = "·"
                    weight = "normal"
                else:
                    text = (
                        f"{value*100:.1f}%\n"
                        f"({count:,})"
                    )
                    weight = (
                        "bold" if i == j else "normal"
                    )

                color = (
                    "white"
                    if value >= 0.58
                    else "#223344"
                )

                ax.text(
                    j, i,
                    text,
                    ha="center",
                    va="center",
                    fontsize=8.9,
                    color=color,
                    fontweight=weight,
                    linespacing=1.15
                )

        # Outline measured correct-class recall.
        # Do not highlight zero-support diagonal cells.
        for i in range(5):

            if support[i] == 0:
                continue

            ax.add_patch(
                Rectangle(
                    (i-0.48, i-0.48),
                    0.96,
                    0.96,
                    fill=False,
                    linewidth=1.5,
                    edgecolor="#CB8143"
                )
            )

        # Highlight the major actual misclassification
        # in the UNSW TCN matrix: Exploit/App -> Fuzzing.
        if (
            source == "UNSW_NB15"
            and model == "Competing-Risk TCN"
        ):

            ax.add_patch(
                Rectangle(
                    (
                        FUZZING_INDEX9 - 0.47,
                        EXPLOIT_INDEX9 - 0.47
                    ),
                    0.94,
                    0.94,
                    fill=False,
                    linewidth=2.5,
                    edgecolor="#C24848"
                )
            )

# Shared colorbar.
cbar_ax9 = fig9.add_axes(
    [0.905, 0.305, 0.018, 0.415]
)

cbar9 = fig9.colorbar(
    im9,
    cax=cbar_ax9
)

cbar9.set_label(
    "Fraction of true-class windows",
    fontsize=10.3,
    labelpad=12
)

cbar9.set_ticks(
    np.linspace(0, 1, 6)
)

# ======================================================================
# F. SOURCE-DEPENDENT FAILURE SUMMARY
# ======================================================================

fig9.text(
    0.055,
    0.097,
    "KEY FINDING",
    fontsize=10.8,
    fontweight="bold",
    color="#263647"
)

fig9.text(
    0.163,
    0.097,
    (
        f"On UNSW-NB15-derived Exploit/App windows, "
        f"CR-LSTM correctly classified "
        f"{lstm_correct9}/{exploit_support9} "
        f"({lstm_correct9/exploit_support9*100:.1f}%), "
        f"versus {tcn_correct9}/{exploit_support9} "
        f"({tcn_correct9/exploit_support9*100:.1f}%) for CR-TCN."
    ),
    fontsize=9.6,
    color="#324459"
)

fig9.text(
    0.055,
    0.072,
    (
        f"CR-TCN assigned {tcn_as_fuzzing9}/{exploit_support9} "
        f"({tcn_as_fuzzing9/exploit_support9*100:.1f}%) "
        f"of these windows to Fuzzing "
        f"(red outlined cell). "
        f"Orange diagonal outlines identify "
        f"correct-family predictions."
    ),
    fontsize=9.3,
    color="#4F5D70"
)

fig9.text(
    0.055,
    0.047,
    "Synthetic NAF v2 development validation. "
    "Rows are normalized by source-specific true-class support; "
    "gray rows denote absent classes, not zero recall. "
    "Family classification is conditional on a true future event.",
    fontsize=9.0,
    color="#5E6B7B"
)

# ======================================================================
# G. SAVE PDF, SVG, PNG AND UNDERLYING DATA
# ======================================================================

STAMP9 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM9 = (
    f"Figure_03_Source_Dependent_Family_Confusion_{STAMP9}"
)

PATHS9 = {
    "pdf": DEST9 / f"{STEM9}.pdf",
    "svg": DEST9 / f"{STEM9}.svg",
    "png": DEST9 / f"{STEM9}.png",
    "csv": DEST9 / f"{STEM9}_Confusion_Data.csv",
    "json": DEST9 / f"{STEM9}.json"
}

for path in PATHS9.values():
    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig9.savefig(
        PATHS9[ext],
        dpi=450 if ext == "png" else 180,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig9)

# Preserve all 100 underlying confusion-matrix cells.
confusion9.to_csv(
    PATHS9["csv"],
    index=False,
    mode="x",
    float_format="%.12f"
)

# ======================================================================
# H. PROVENANCE MANIFEST
# ======================================================================

GENERATED9 = [
    PATHS9[ext]
    for ext in ["pdf", "svg", "png", "csv"]
]

MANIFEST9 = {
    "project": "NetGuard",
    "publication_figure": 3,
    "title": (
        "Source-dependent future "
        "attack-family classification"
    ),
    "dataset": "Synthetic NAF v2",
    "evaluation": "development validation",
    "models": MODELS9,
    "sources": SOURCES9,
    "class_order": FAMILIES9,
    "panels": 4,
    "event_windows_per_source": EXPECTED_SOURCE_TOTALS9,
    "all_source_supports_verified": True,
    "all_row_fractions_verified": True,
    "class_recalls_verified": True,
    "key_failure": {
        "source": "UNSW_NB15",
        "true_family": "EXPLOIT_APP",
        "true_support": exploit_support9,
        "lstm_correct": lstm_correct9,
        "tcn_correct": tcn_correct9,
        "tcn_as_fuzzing": tcn_as_fuzzing9
    },
    "source_files": [
        {
            "path": str(p.relative_to(ROOT9)),
            "sha256": sha9(p)
        }
        for p in [
            CONFUSION_FILE9,
            RECALL_FILE9,
            SOURCE_MANIFEST9,
            CORRECTED_MANIFEST9
        ]
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(ROOT9)),
            "sha256": sha9(p)
        }
        for p in GENERATED9
    ],
    "limitations": [
        "Future attack-family classification, "
        "not current traffic classification.",
        "Evaluation conditional on true future events.",
        "Synthetic validation data.",
        "Source-specific true-class support differs.",
        "Zero-support classes represented as N/A.",
        "Source comparisons are descriptive.",
        "No transfer-learning claim is made.",
        "No official TEST dataset accessed.",
        "No models retrained."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS9["json"].open(
    "x",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST9,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# I. FINAL VERIFICATION
# ======================================================================

print("\n" + "=" * 100)
print("CELL 9 COMPLETE — FINAL PAPER FIGURE 3")
print("=" * 100)

print("Matrices:", len(COUNTS9))
print("Attack families:", len(FAMILIES9))
print("Total matrix cells:", len(confusion9))
print("Source-specific support verification: PASS")
print("Recall verification: PASS")
print("Key Exploit/App failure verified: PASS")

print("\nSAVED PUBLICATION ASSETS")
print("-" * 100)

for ext, path in PATHS9.items():

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Missing/empty output: {path}"
        )

    print(
        f"PASS [{ext.upper()}] {path.name}"
    )

print("\nFinal research-paper figures completed: 3 / 5")
print("Original research outputs modified: NO")
print("Previous final figures overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 100)


In [ ]:

# ================================================================
# CELL 9B — PUBLICATION FIGURE 3 LAYOUT CORRECTION
#
# Fix: bottom x-axis labels overlap key findings / footnotes.
# Original Figure 3 remains untouched.
# No data reload, model inference or retraining.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import shutil

print("=" * 85)
print("CELL 9B — CORRECTED FINAL PAPER FIGURE 3")
print("=" * 85)

# Requires Cell 9 to have been run in this Colab session.
required9b = [
    "fig9", "axes9", "DEST9", "PATHS9",
    "CONFUSION_FILE9", "sha9"
]

missing9b = [
    name for name in required9b
    if name not in globals()
]

if missing9b:
    raise RuntimeError(
        "Run Cell 9 first. Missing variables: "
        + ", ".join(missing9b)
    )

# 1. Increase the height and reserve footer space.
fig9.set_size_inches(16.2, 13.8, forward=True)

fig9.subplots_adjust(
    left=0.135,
    right=0.875,
    top=0.825,
    bottom=0.265,
    wspace=0.33,
    hspace=0.47
)

# 2. Move the findings and caption beneath the x labels.
footer_rules9b = [
    ("KEY FINDING", 0.133, 0.055),
    ("On UNSW-NB15-derived", 0.133, 0.163),
    ("CR-TCN assigned", 0.102, 0.055),
    ("Synthetic NAF v2 development validation", 0.072, 0.055)
]

for prefix, new_y, new_x in footer_rules9b:

    matches = [
        t for t in fig9.texts
        if t.get_text().startswith(prefix)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely find footer: {prefix}"
        )

    matches[0].set_position((new_x, new_y))

# 3. Force a fresh drawing before export.
fig9.canvas.draw()

# 4. Choose new filenames; never overwrite originals.
stamp9b = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

stem9b = (
    "Figure_03_Source_Dependent_Family_Confusion_"
    f"FINAL_V2_{stamp9b}"
)

paths9b = {
    "pdf": DEST9 / f"{stem9b}.pdf",
    "svg": DEST9 / f"{stem9b}.svg",
    "png": DEST9 / f"{stem9b}.png",
    "csv": DEST9 / f"{stem9b}_Confusion_Data.csv",
    "json": DEST9 / f"{stem9b}.json"
}

for p in paths9b.values():
    if p.exists():
        raise FileExistsError(p)

# 5. Export corrected vector and raster figures.
for ext in ["pdf", "svg", "png"]:
    fig9.savefig(
        paths9b[ext],
        dpi=450 if ext == "png" else 180,
        bbox_inches="tight",
        facecolor="white"
    )

# Preserve exact underlying confusion data.
shutil.copyfile(
    PATHS9["csv"],
    paths9b["csv"]
)

assert sha9(paths9b["csv"]) == sha9(PATHS9["csv"])

# 6. Record provenance and exact original references.
manifest9b = {
    "project": "NetGuard",
    "publication_figure": 3,
    "version": "FINAL_V2",
    "change": (
        "Enlarged bottom margin and repositioned "
        "findings to prevent text overlap."
    ),
    "scientific_data_changed": False,
    "confusion_source_sha256": sha9(CONFUSION_FILE9),
    "previous_figure": {
        "path": str(PATHS9["png"]),
        "sha256": sha9(PATHS9["png"])
    },
    "new_outputs": [
        {
            "path": str(paths9b[ext]),
            "sha256": sha9(paths9b[ext])
        }
        for ext in ["pdf", "svg", "png", "csv"]
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with paths9b["json"].open(
    "x", encoding="utf-8"
) as f:
    json.dump(
        manifest9b,
        f,
        indent=2,
        allow_nan=False
    )

# Re-display the corrected version.
from IPython.display import display
display(fig9)

print("\nCORRECTED OUTPUTS")
print("-" * 85)

for ext, path in paths9b.items():
    assert path.exists() and path.stat().st_size > 0
    print(f"PASS [{ext.upper()}] {path.name}")

print("\nOriginal Figure 3 preserved: YES")
print("Scientific data changed: NO")
print("Confusion matrices recomputed: NO")
print("Official TEST accessed: NO")
print("Models retrained: 0")
print("=" * 85)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 10 — FIGURE 4: CONDITIONAL TIME-TO-ATTACK ESTIMATION
#
# Four panels:
# A. CR-LSTM: true versus predicted ETA density
# B. CR-TCN: true versus predicted ETA density
# C. Cumulative distribution of absolute ETA error
# D. Signed ETA prediction bias by actual time remaining
#
# Source: saved F10 official development-validation predictions
# Independent validation: saved F10 per-minute/source reports
#
# Outputs: PDF, SVG, PNG, CSV.GZ, JSON
#
# No model training, no official TEST, no overwriting.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

print("=" * 100)
print("CELL 10 — FINAL PAPER FIGURE 4: CONDITIONAL ETA ANALYSIS")
print("=" * 100)

# ======================================================================
# A. EXACT SOURCE FILES
# ======================================================================

ROOT10 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

MODEL_ROOT10 = (
    ROOT10 / "Synthetic_NAF_v2" / "models"
)

RESEARCH10 = ROOT10 / "Research_Outputs"

DEST10 = (
    ROOT10 / "Research_Paper" / "01_Final_Figures"
)

PREDICTION_FILES10 = {
    "CR-LSTM": (
        MODEL_ROOT10 / "LSTM_V2_CompetingRisk"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.csv"
    ),
    "CR-TCN": (
        MODEL_ROOT10 / "TCN_V2_CompetingRisk"
        / "official_validation"
        / "VALIDATION_PREDICTIONS.csv"
    )
}

MINUTE_FILE10 = (
    RESEARCH10 / "02_Tables"
    / "T12_ETA_By_True_Minute_20261027_065243_370117.csv"
)

SOURCE_FILE10 = (
    RESEARCH10 / "02_Tables"
    / "T13_ETA_By_Source_20261027_065243_370117.csv"
)

ORIGINAL_MANIFEST10 = (
    RESEARCH10 / "05_Manifests"
    / "eta_analysis_manifest_20261027_065243_370117.json"
)

SOURCE_FILES10 = (
    list(PREDICTION_FILES10.values())
    + [MINUTE_FILE10, SOURCE_FILE10]
)

for path in SOURCE_FILES10 + [ORIGINAL_MANIFEST10]:
    if not path.is_file():
        raise FileNotFoundError(path)

if not DEST10.is_dir():
    raise FileNotFoundError(
        "Missing Research_Paper/01_Final_Figures"
    )

def sha10(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()

# ======================================================================
# B. VERIFY F10 ORIGINAL MANIFEST AND SOURCE CHECKSUMS
# ======================================================================

with ORIGINAL_MANIFEST10.open(
    "r", encoding="utf-8"
) as f:
    original_manifest10 = json.load(f)

if original_manifest10.get("official_test_accessed") is not False:
    raise RuntimeError(
        "Original F10 manifest does not confirm TEST exclusion"
    )

for path in SOURCE_FILES10:

    relative = path.relative_to(ROOT10).as_posix()

    manifest_entries = (
        original_manifest10["data_inputs"]
        + original_manifest10["generated_files"]
    )

    matches = [
        item
        for item in manifest_entries
        if item["path"] == relative
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"F10 manifest missing source: {relative}"
        )

    expected_hash = matches[0]["sha256"]
    actual_hash = sha10(path)

    if expected_hash != actual_hash:
        raise RuntimeError(
            f"Source integrity failure: {relative}"
        )

print("\nA. SAVED RESEARCH DATA INTEGRITY")
print("-" * 100)

for path in SOURCE_FILES10:
    print("PASS |", path.name, "| SHA256 verified")

# ======================================================================
# C. LOAD ALIGNED SAVED PREDICTIONS
# ======================================================================

USE_COLUMNS10 = [
    "Episode_ID",
    "Minute",
    "Source",
    "True_H15",
    "True_Onset_Minute",
    "Expected_ETA_If_Event"
]

FRAMES10 = {}

for model, path in PREDICTION_FILES10.items():

    frame = pd.read_csv(
        path,
        usecols=USE_COLUMNS10
    )

    if len(frame) != 18839:
        raise RuntimeError(
            f"{model}: expected 18,839 validation records"
        )

    for col in [
        "Minute",
        "True_H15",
        "True_Onset_Minute",
        "Expected_ETA_If_Event"
    ]:
        frame[col] = pd.to_numeric(
            frame[col],
            errors="raise"
        )

    if frame[USE_COLUMNS10].isna().any().any():
        raise RuntimeError(
            f"{model}: missing prediction data"
        )

    if frame.duplicated(
        ["Episode_ID", "Minute", "Source"]
    ).any():
        raise RuntimeError(
            f"{model}: duplicate validation records"
        )

    if not frame["True_H15"].isin([0, 1]).all():
        raise RuntimeError(
            f"{model}: invalid H15 target"
        )

    if not np.isfinite(
        frame[[
            "True_Onset_Minute",
            "Expected_ETA_If_Event"
        ]].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{model}: nonfinite ETA values"
        )

    frame = frame.sort_values(
        ["Source", "Episode_ID", "Minute"]
    ).reset_index(drop=True)

    FRAMES10[model] = frame

# Verify that the two models were evaluated on exactly
# the same records with the same future-event labels.
reference10 = FRAMES10["CR-LSTM"]
comparison10 = FRAMES10["CR-TCN"]

for col in [
    "Episode_ID",
    "Minute",
    "Source",
    "True_H15",
    "True_Onset_Minute"
]:

    np.testing.assert_array_equal(
        reference10[col].to_numpy(),
        comparison10[col].to_numpy(),
        err_msg=f"Validation alignment mismatch: {col}"
    )

print("\nB. FULL VALIDATION ALIGNMENT")
print("-" * 100)

print("CR-LSTM records:", len(reference10))
print("CR-TCN records:", len(comparison10))
print("Record IDs and true-event labels aligned: PASS")

# ======================================================================
# D. SELECT ONLY TRUE FUTURE-EVENT WINDOWS
#
# Conditional ETA is evaluated only where an attack
# truly occurs within the next 15 minutes.
# ======================================================================

EVENTS10 = {}
ALL_EVENT_ROWS10 = []

EXPECTED_SOURCE_EVENTS10 = {
    "CIC_IDS2017": 1710,
    "UNSW_NB15": 1665
}

for model, frame in FRAMES10.items():

    event = (
        frame.loc[frame["True_H15"] == 1]
        .copy()
    )

    if len(event) != 3375:
        raise RuntimeError(
            f"{model}: expected 3,375 true-event windows"
        )

    event["True_ETA"] = event[
        "True_Onset_Minute"
    ].astype(float)

    event["Predicted_ETA"] = event[
        "Expected_ETA_If_Event"
    ].astype(float)

    if not event["True_ETA"].between(1, 15).all():
        raise RuntimeError(
            f"{model}: true ETA outside 1–15 minutes"
        )

    if not event["Predicted_ETA"].between(
        1 - 1e-6,
        15 + 1e-6
    ).all():
        raise RuntimeError(
            f"{model}: predicted ETA outside 1–15 minutes"
        )

    event["Signed_Error"] = (
        event["Predicted_ETA"] - event["True_ETA"]
    )

    event["Absolute_Error"] = (
        event["Signed_Error"].abs()
    )

    event["Model"] = model

    source_counts = (
        event["Source"].value_counts().to_dict()
    )

    if source_counts != EXPECTED_SOURCE_EVENTS10:
        raise RuntimeError(
            f"{model}: unexpected source distribution "
            f"{source_counts}"
        )

    minute_counts = (
        event["True_ETA"]
        .value_counts()
        .sort_index()
    )

    if len(minute_counts) != 15:
        raise RuntimeError(
            f"{model}: incomplete 1–15 minute coverage"
        )

    if not (minute_counts == 225).all():
        raise RuntimeError(
            f"{model}: expected 225 windows per minute"
        )

    EVENTS10[model] = event

    ALL_EVENT_ROWS10.append(
        event[
            [
                "Episode_ID",
                "Minute",
                "Source",
                "Model",
                "True_ETA",
                "Predicted_ETA",
                "Signed_Error",
                "Absolute_Error"
            ]
        ]
    )

PLOT_DATA10 = pd.concat(
    ALL_EVENT_ROWS10,
    ignore_index=True
)

print("\nC. TRUE-EVENT VALIDATION RECORDS")
print("-" * 100)

print("Events per model:", 3375)
print("Total model-event observations:", len(PLOT_DATA10))
print("CIC-IDS2017 windows per model:", 1710)
print("UNSW-NB15 windows per model:", 1665)
print("Per-minute event support:", 225)
print("Conditional ETA evaluation rules: PASS")

# ======================================================================
# E. INDEPENDENTLY RECOMPUTE METRICS
# ======================================================================

minute_report10 = pd.read_csv(MINUTE_FILE10)
source_report10 = pd.read_csv(SOURCE_FILE10)

MINUTE_REQUIRED10 = {
    "True_Onset_Minute",
    "Windows",
    "ETA_MAE",
    "Mean_Signed_Error",
    "Mean_Predicted_ETA",
    "Model"
}

SOURCE_REQUIRED10 = {
    "Source",
    "Event_Windows",
    "ETA_MAE",
    "Model"
}

if not MINUTE_REQUIRED10.issubset(
    minute_report10.columns
):
    raise RuntimeError(
        "Unexpected F10 per-minute report schema"
    )

if not SOURCE_REQUIRED10.issubset(
    source_report10.columns
):
    raise RuntimeError(
        "Unexpected F10 source report schema"
    )

METRICS10 = {}

for model, event in EVENTS10.items():

    true = event["True_ETA"].to_numpy()
    pred = event["Predicted_ETA"].to_numpy()

    error = pred - true
    absolute = np.abs(error)

    mae = float(absolute.mean())
    median_ae = float(np.median(absolute))
    bias = float(error.mean())

    # Independently recomputed per-minute comparisons
    for minute in range(1, 16):

        subset = event.loc[
            event["True_ETA"] == minute
        ]

        match = minute_report10.loc[
            (minute_report10["Model"] == model)
            & (
                minute_report10["True_Onset_Minute"]
                == minute
            )
        ]

        if len(match) != 1:
            raise RuntimeError(
                f"Missing minute report: {model}/{minute}"
            )

        reported = match.iloc[0]

        if int(reported["Windows"]) != len(subset):
            raise RuntimeError(
                "Per-minute support mismatch"
            )

        np.testing.assert_allclose(
            [
                subset["Absolute_Error"].mean(),
                subset["Signed_Error"].mean(),
                subset["Predicted_ETA"].mean()
            ],
            [
                reported["ETA_MAE"],
                reported["Mean_Signed_Error"],
                reported["Mean_Predicted_ETA"]
            ],
            rtol=0,
            atol=2e-5
        )

    # Independently recompute both source-specific MAEs.
    for source in EXPECTED_SOURCE_EVENTS10:

        subset = event.loc[
            event["Source"] == source
        ]

        match = source_report10.loc[
            (source_report10["Model"] == model)
            & (source_report10["Source"] == source)
        ]

        if len(match) != 1:
            raise RuntimeError(
                f"Missing source report: {model}/{source}"
            )

        saved = match.iloc[0]

        if int(saved["Event_Windows"]) != len(subset):
            raise RuntimeError(
                "Source event support mismatch"
            )

        np.testing.assert_allclose(
            subset["Absolute_Error"].mean(),
            saved["ETA_MAE"],
            rtol=0,
            atol=2e-5
        )

    METRICS10[model] = {
        "event_windows": int(len(event)),
        "eta_mae": mae,
        "median_absolute_error": median_ae,
        "mean_signed_error": bias,
        "within_2_min_fraction": float(
            (absolute <= 2).mean()
        ),
        "within_3_min_fraction": float(
            (absolute <= 3).mean()
        )
    }

np.testing.assert_allclose(
    [
        METRICS10["CR-LSTM"]["eta_mae"],
        METRICS10["CR-TCN"]["eta_mae"]
    ],
    [2.53653, 2.48162],
    rtol=0,
    atol=2e-5
)

print("\nD. RECOMPUTED ETA METRICS")
print("-" * 100)

for model, vals in METRICS10.items():

    print(
        f"{model:<9} "
        f"MAE={vals['eta_mae']:.5f} min | "
        f"Median AE={vals['median_absolute_error']:.5f} | "
        f"Signed bias={vals['mean_signed_error']:+.5f}"
    )

print("Original F10 per-minute report: MATCH")
print("Original F10 source-specific report: MATCH")
print("Original pooled MAE benchmarks: MATCH")

# ======================================================================
# F. FIGURE STYLE AND LAYOUT
# ======================================================================

COLORS10 = {
    "CR-LSTM": "#147F83",
    "CR-TCN": "#8854A0"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig10 = plt.figure(
    figsize=(15.8, 10.6),
    facecolor="white"
)

gs10 = fig10.add_gridspec(
    2, 2,
    left=0.085,
    right=0.895,
    top=0.855,
    bottom=0.183,
    hspace=0.43,
    wspace=0.31
)

ax_a10 = fig10.add_subplot(gs10[0, 0])
ax_b10 = fig10.add_subplot(gs10[0, 1])
ax_c10 = fig10.add_subplot(gs10[1, 0])
ax_d10 = fig10.add_subplot(gs10[1, 1])

fig10.suptitle(
    "Conditional time-to-attack estimation accuracy",
    x=0.065,
    y=0.976,
    ha="left",
    fontsize=18.5,
    fontweight="bold"
)

fig10.text(
    0.065,
    0.939,
    "Competing-risk LSTM versus TCN  |  "
    "3,375 true-event windows per model  |  "
    "Prediction horizon: 1–15 minutes",
    fontsize=10.6,
    color="#5B6B7F"
)

# ======================================================================
# G. PANELS A/B: ACTUAL VS PREDICTED ETA DENSITY
# ======================================================================

HEX_COLLECTIONS10 = []

for ax, model, letter in [
    (ax_a10, "CR-LSTM", "A"),
    (ax_b10, "CR-TCN", "B")
]:

    event = EVENTS10[model]

    hb = ax.hexbin(
        event["True_ETA"],
        event["Predicted_ETA"],
        gridsize=(30, 28),
        extent=(0.5, 15.5, 0.5, 15.5),
        mincnt=1,
        cmap="YlGnBu",
        norm=LogNorm(vmin=1)
    )

    HEX_COLLECTIONS10.append(hb)

    # Ideal prediction diagonal
    ax.plot(
        [1, 15], [1, 15],
        linestyle="--",
        linewidth=1.5,
        color="#CD7940",
        label="Perfect prediction",
        zorder=5
    )

    ax.set_xlim(0.5, 15.5)
    ax.set_ylim(0.5, 15.5)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xticks([1, 3, 5, 7, 9, 11, 13, 15])
    ax.set_yticks([1, 3, 5, 7, 9, 11, 13, 15])

    ax.set_xlabel(
        "True time until attack (minutes)"
    )

    ax.set_ylabel(
        "Predicted time until attack (minutes)"
    )

    ax.set_title(
        f"{letter}    {model}: observed vs predicted ETA",
        loc="left",
        pad=13,
        fontweight="bold"
    )

    ax.text(
        0.04, 0.965,
        (
            f"N = 3,375\n"
            f"MAE = {METRICS10[model]['eta_mae']:.3f} min"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9.1,
        bbox={
            "facecolor": "white",
            "edgecolor": "#D7E0E7",
            "boxstyle": "round,pad=0.42",
            "alpha": 0.95
        }
    )

    ax.grid(
        linestyle=":",
        color="#D0D9E2",
        linewidth=0.5,
        alpha=0.45
    )

# Use a consistent density scale for both models.
max_hex_count10 = max(
    float(hb.get_array().max())
    for hb in HEX_COLLECTIONS10
)

for hb in HEX_COLLECTIONS10:
    hb.set_clim(1, max_hex_count10)

cax10 = fig10.add_axes([
    0.925, 0.546, 0.016, 0.255
])

cbar10 = fig10.colorbar(
    HEX_COLLECTIONS10[0],
    cax=cax10
)

cbar10.set_label(
    "Window density (log scale)",
    fontsize=9.6,
    labelpad=10
)

# ======================================================================
# H. PANEL C: ABSOLUTE ETA ERROR CDF
# ======================================================================

for model in ["CR-LSTM", "CR-TCN"]:

    errors = np.sort(
        EVENTS10[model][
            "Absolute_Error"
        ].to_numpy(dtype=float)
    )

    cumulative = (
        np.arange(1, len(errors)+1)
        / len(errors)
    )

    ax_c10.plot(
        errors,
        cumulative,
        linewidth=2.4,
        color=COLORS10[model],
        label=(
            f"{model}  "
            f"(MAE {METRICS10[model]['eta_mae']:.3f} min)"
        )
    )

ax_c10.set_title(
    "C    Absolute ETA error distribution",
    loc="left",
    pad=13,
    fontweight="bold"
)

ax_c10.axvline(
    2,
    linestyle=":",
    color="#7A8797",
    linewidth=1.2
)

ax_c10.axvline(
    3,
    linestyle=":",
    color="#A5ADB9",
    linewidth=1.2
)

ax_c10.set_xlim(
    0,
    np.ceil(
        PLOT_DATA10["Absolute_Error"].max()
    )
)

ax_c10.set_ylim(0, 1.015)

ax_c10.set_xlabel(
    "Absolute time-to-attack error (minutes)"
)

ax_c10.set_ylabel(
    "Fraction of true-event windows"
)

ax_c10.legend(
    loc="lower right",
    frameon=True,
    fontsize=9.1,
    edgecolor="#D8DEE6"
)

ax_c10.grid(
    linestyle=":",
    linewidth=0.65,
    color="#CFD6DE",
    alpha=0.7
)

# ======================================================================
# I. PANEL D: EARLY/LATE ETA BIAS
#
# Signed error = predicted ETA - true ETA
# Positive: overestimates time remaining.
# Negative: underestimates time remaining.
# ======================================================================

for model in ["CR-LSTM", "CR-TCN"]:

    event = EVENTS10[model]

    grouped = (
        event.groupby("True_ETA", sort=True)
        ["Signed_Error"]
        .mean()
    )

    x = grouped.index.to_numpy(dtype=float)
    y = grouped.to_numpy(dtype=float)

    ax_d10.plot(
        x,
        y,
        linewidth=2.2,
        color=COLORS10[model],
        marker="o",
        markersize=4.5,
        label=model
    )

ax_d10.axhline(
    0,
    color="#778596",
    linestyle="--",
    linewidth=1.2
)

ax_d10.set_xticks(
    np.arange(1, 16, 2)
)

ax_d10.set_xlim(0.5, 15.5)

ax_d10.set_xlabel(
    "True time remaining before attack (minutes)"
)

ax_d10.set_ylabel(
    "Mean signed ETA error (minutes)"
)

ax_d10.set_title(
    "D    Systematic early/late prediction bias",
    loc="left",
    pad=13,
    fontweight="bold"
)

ax_d10.legend(
    loc="upper right",
    frameon=True,
    fontsize=9.4,
    edgecolor="#D8DEE6"
)

ax_d10.grid(
    linestyle=":",
    linewidth=0.65,
    color="#CFD6DE",
    alpha=0.7
)

# ======================================================================
# J. SCIENTIFIC FIGURE NOTES
# ======================================================================

fig10.text(
    0.065,
    0.126,
    "Figure 4. Conditional expected time-to-attack "
    "predictions from saved CR-LSTM and CR-TCN "
    "development-validation results.",
    fontsize=9.3,
    color="#47586A"
)

fig10.text(
    0.065,
    0.101,
    "Density panels use a common logarithmic count scale. "
    "The dashed diagonal represents perfect ETA prediction; "
    "the CDF shows absolute error.",
    fontsize=9.3,
    color="#47586A"
)

fig10.text(
    0.065,
    0.076,
    "Positive signed error means the model predicts more "
    "time remaining than actually exists; negative error "
    "means it predicts less.",
    fontsize=9.3,
    color="#47586A"
)

fig10.text(
    0.065,
    0.051,
    "Synthetic NAF v2 development validation. "
    "ETA is evaluated conditional on a true future attack. "
    "Windows overlap in time; confidence intervals "
    "and end-to-end onset accuracy are not implied.",
    fontsize=9.0,
    color="#5C6D80"
)

# ======================================================================
# K. SAVE RESEARCH-PAPER EXPORTS
# ======================================================================

STAMP10 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM10 = (
    f"Figure_04_Conditional_Time_To_Attack_{STAMP10}"
)

PATHS10 = {
    "pdf": DEST10 / f"{STEM10}.pdf",
    "svg": DEST10 / f"{STEM10}.svg",
    "png": DEST10 / f"{STEM10}.png",
    "data": DEST10 / f"{STEM10}_ETA_Data.csv.gz",
    "json": DEST10 / f"{STEM10}.json"
}

for path in PATHS10.values():

    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig10.savefig(
        PATHS10[ext],
        dpi=450 if ext == "png" else 180,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig10)

PLOT_DATA10.to_csv(
    PATHS10["data"],
    index=False,
    compression="gzip",
    float_format="%.12g"
)

# ======================================================================
# L. REPRODUCIBILITY MANIFEST
# ======================================================================

GENERATED10 = [
    PATHS10[ext]
    for ext in ["pdf", "svg", "png", "data"]
]

MANIFEST10 = {
    "project": "NetGuard",
    "publication_figure": 4,
    "title": (
        "Conditional time-to-attack estimation accuracy"
    ),
    "evaluation": "Synthetic NAF v2 development validation",
    "models": ["CR-LSTM", "CR-TCN"],
    "validation_windows_per_model": 18839,
    "true_event_windows_per_model": 3375,
    "panels": 4,
    "metrics": METRICS10,
    "original_f10_metrics_reproduced": True,
    "source_files": [
        {
            "path": str(p.relative_to(ROOT10)),
            "sha256": sha10(p)
        }
        for p in SOURCE_FILES10 + [ORIGINAL_MANIFEST10]
    ],
    "generated_files": [
        {
            "path": str(p.relative_to(ROOT10)),
            "sha256": sha10(p)
        }
        for p in GENERATED10
    ],
    "limitations": [
        "ETA conditional on a true event within 15 minutes.",
        "Not end-to-end attack-onset detection accuracy.",
        "Synthetic development-validation data.",
        "Overlapping, temporally dependent windows.",
        "No independent-window confidence intervals.",
        "No official TEST evaluation.",
        "No models retrained."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS10["json"].open(
    "x", encoding="utf-8"
) as handle:

    json.dump(
        MANIFEST10,
        handle,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# M. FINAL CHECK
# ======================================================================

print("\n" + "=" * 100)
print("CELL 10 COMPLETE — FINAL PAPER FIGURE 4")
print("=" * 100)

print("Models:", len(EVENTS10))
print("Figure panels:", 4)
print("True-event windows per model:", 3375)
print("Total plotted event-model records:", len(PLOT_DATA10))
print("Source checksum verification: PASS")
print("Per-minute ETA verification: PASS")
print("Source-specific ETA verification: PASS")
print("Pooled ETA MAE verification: PASS")

print("\nSAVED PUBLICATION ASSETS")
print("-" * 100)

for ext, path in PATHS10.items():

    if not path.is_file():
        raise RuntimeError(f"Missing output: {path}")

    if path.stat().st_size == 0:
        raise RuntimeError(f"Empty output: {path}")

    print(f"PASS [{ext.upper()}] {path.name}")

print("\nFinal paper figures completed: 4 / 5")
print("Original F10 outputs modified: NO")
print("Previous publication figures overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 100)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 11 — FIGURE 5: CURRENT ATTACK DETECTION & CLASSIFICATION
#
# A. Binary attack precision–recall curve
# B. Binary ROC curve (with low-FPR detail)
# C. Ten-class XGBoost V1 confusion matrix
# D. Class-wise recall: XGBoost V1 vs weighted XGBoost V2
#
# Sources: Verified Research_Outputs F12 and F14.
# V2 per-class recall: previously reported Cell 22 values,
# checked for consistency with saved balanced accuracy.
#
# Outputs: PDF / SVG / PNG / CSV / JSON
#
# No model training, no TEST access, no overwritten files.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

print("=" * 100)
print("CELL 11 — FINAL PAPER FIGURE 5: CURRENT ATTACK DETECTION")
print("=" * 100)

# ======================================================================
# A. ORIGINAL SOURCE PATHS
# ======================================================================

ROOT11 = Path("/content/drive/MyDrive/network-attack-forecasting")
RESEARCH11 = ROOT11 / "Research_Outputs"

DEST11 = ROOT11 / "Research_Paper" / "01_Final_Figures"

SOURCES11 = {
    "pr": RESEARCH11 / "02_Tables"
          / "T22_Binary_Precision_Recall_Curve_20261027_073004_429411.csv",
    "roc": RESEARCH11 / "02_Tables"
           / "T23_Binary_ROC_Curve_20261027_073004_429411.csv",
    "confusion": RESEARCH11 / "02_Tables"
                 / "T17_Current_Attack_Confusion_Details_20261027_071414_062617.csv",
    "class_metrics": RESEARCH11 / "02_Tables"
                     / "T16_Current_Attack_Class_Performance_20261027_071414_062617.csv",
    "model_comparison": RESEARCH11 / "02_Tables"
                        / "T15_Current_Attack_Model_Comparison_20261027_071414_062617.csv",
    "binary_metrics": RESEARCH11 / "03_Metrics"
                      / "binary_attack_detection_metrics_20261027_073004_429411.json",
    "multiclass_metrics": RESEARCH11 / "03_Metrics"
                          / "current_attack_multiclass_metrics_20261027_071414_062617.json",
    "weighted_metadata": ROOT11 / "Current_Attack_Detection"
                         / "experiments"
                         / "weighted_xgb_v2_20261027_035816_500107"
                         / "metadata.json"
}

MANIFESTS11 = {
    "binary": RESEARCH11 / "05_Manifests"
              / "binary_detection_manifest_20261027_073004_429411.json",
    "multiclass": RESEARCH11 / "05_Manifests"
                  / "current_attack_multiclass_manifest_20261027_071414_062617.json"
}

if not DEST11.is_dir():
    raise FileNotFoundError(DEST11)

for path in list(SOURCES11.values()) + list(MANIFESTS11.values()):
    if not path.is_file():
        raise FileNotFoundError(path)

def hash11(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

with MANIFESTS11["binary"].open("r") as f:
    binary_manifest11 = json.load(f)

with MANIFESTS11["multiclass"].open("r") as f:
    multi_manifest11 = json.load(f)

if binary_manifest11.get("official_test_accessed") is not False:
    raise RuntimeError("Binary TEST exclusion unverified")

if multi_manifest11.get("official_test_accessed") is not False:
    raise RuntimeError("Multiclass TEST exclusion unverified")

# Original file-integrity verification
for key, path in SOURCES11.items():

    if key == "weighted_metadata":
        continue

    manifest = (
        binary_manifest11
        if key in ["pr", "roc", "binary_metrics"]
        else multi_manifest11
    )

    entries = (
        manifest["outputs"]
        if manifest is binary_manifest11
        else manifest["generated_files"]
    )

    relative = path.relative_to(ROOT11).as_posix()

    match = [
        item for item in entries
        if item["path"] == relative
    ]

    if len(match) != 1:
        raise RuntimeError(
            f"Source not uniquely recorded: {relative}"
        )

    if hash11(path) != match[0]["sha256"]:
        raise RuntimeError(
            f"Checksum mismatch: {relative}"
        )

print("\nA. SOURCE INTEGRITY")
print("-" * 100)
print("F12 multiclass research files: VERIFIED")
print("F14 binary detection research files: VERIFIED")
print("Original checksums: PASS")

# ======================================================================
# B. LOAD RESEARCH RESULTS
# ======================================================================

pr11 = pd.read_csv(SOURCES11["pr"])
roc11 = pd.read_csv(SOURCES11["roc"])
cm_raw11 = pd.read_csv(SOURCES11["confusion"])
class_raw11 = pd.read_csv(SOURCES11["class_metrics"])
model_raw11 = pd.read_csv(SOURCES11["model_comparison"])

with SOURCES11["binary_metrics"].open("r") as f:
    binary11 = json.load(f)

with SOURCES11["multiclass_metrics"].open("r") as f:
    multiclass11 = json.load(f)

with SOURCES11["weighted_metadata"].open("r") as f:
    weighted11 = json.load(f)

if weighted11.get("official_test_used") is not False:
    raise RuntimeError("Weighted model TEST exclusion unverified")

CLASSES11 = multiclass11["classes"]

if weighted11["classes"] != CLASSES11:
    raise RuntimeError("Weighted model class order mismatch")

if len(CLASSES11) != 10:
    raise RuntimeError("Expected ten classes")

N11 = 35167
ATTACK11 = 23976
NORMAL11 = 11191

assert binary11["validation_rows"] == N11
assert multiclass11["validation_rows"] == N11
assert binary11["positive_attack_rows"] == ATTACK11
assert binary11["normal_rows"] == NORMAL11

required_pr11 = {"Recall", "Precision"}
required_roc11 = {"False_Positive_Rate", "True_Positive_Rate"}
required_cm11 = {
    "True_Class", "Predicted_Class", "Count",
    "True_Class_Support", "Fraction_Of_True_Class"
}

assert required_pr11.issubset(pr11.columns)
assert required_roc11.issubset(roc11.columns)
assert required_cm11.issubset(cm_raw11.columns)

# ======================================================================
# C. VERIFY ORIGINAL BINARY CURVES
# ======================================================================

recall11 = pr11["Recall"].to_numpy(dtype=float)
precision11 = pr11["Precision"].to_numpy(dtype=float)

fpr11 = roc11["False_Positive_Rate"].to_numpy(dtype=float)
tpr11 = roc11["True_Positive_Rate"].to_numpy(dtype=float)

for name, arr in {
    "PR recall": recall11,
    "PR precision": precision11,
    "ROC FPR": fpr11,
    "ROC TPR": tpr11
}.items():

    if len(arr) < 2 or not np.isfinite(arr).all():
        raise RuntimeError(f"Invalid curve: {name}")

    if not ((arr >= -1e-10) & (arr <= 1+1e-10)).all():
        raise RuntimeError(f"Curve outside [0,1]: {name}")

# sklearn precision_recall_curve typically stores
# recall from 1 down to 0.
if np.all(np.diff(recall11) <= 1e-8):

    ap_recomputed11 = float(
        -np.sum(
            np.diff(recall11) * precision11[:-1]
        )
    )

elif np.all(np.diff(recall11) >= -1e-8):

    ap_recomputed11 = float(
        np.sum(
            np.diff(recall11) * precision11[1:]
        )
    )

else:
    raise RuntimeError("PR recall order is not monotonic")

if np.any(np.diff(fpr11) < -1e-8):
    raise RuntimeError("ROC FPR is not monotonic")

auc_recomputed11 = float(
    np.trapezoid(tpr11, fpr11)
)

AP11 = float(binary11["average_precision"])
AUC11 = float(binary11["roc_auc"])

np.testing.assert_allclose(
    [ap_recomputed11, auc_recomputed11],
    [AP11, AUC11],
    atol=2e-6,
    rtol=0
)

print("\nB. BINARY CURVE VERIFICATION")
print("-" * 100)
print(f"Attack AP: saved={AP11:.6f} recomputed={ap_recomputed11:.6f}")
print(f"ROC-AUC:   saved={AUC11:.6f} recomputed={auc_recomputed11:.6f}")
print("Binary PR / ROC consistency: PASS")

# ======================================================================
# D. RECONSTRUCT TEN-CLASS CONFUSION MATRIX
# ======================================================================

if len(cm_raw11) != 100:
    raise RuntimeError("Expected 100 confusion matrix cells")

matrix11 = (
    cm_raw11.pivot(
        index="True_Class",
        columns="Predicted_Class",
        values="Count"
    )
    .reindex(index=CLASSES11, columns=CLASSES11)
)

if matrix11.isna().any().any():
    raise RuntimeError("Incomplete multiclass matrix")

CM11 = matrix11.to_numpy(dtype=int)

if CM11.sum() != N11:
    raise RuntimeError("Confusion validation total mismatch")

support11 = CM11.sum(axis=1)
predicted11 = CM11.sum(axis=0)

fraction11 = (
    CM11 / support11[:, None]
)

# Verify recorded row percentages.
raw_fraction11 = (
    cm_raw11.pivot(
        index="True_Class",
        columns="Predicted_Class",
        values="Fraction_Of_True_Class"
    )
    .reindex(index=CLASSES11, columns=CLASSES11)
    .to_numpy(dtype=float)
)

np.testing.assert_allclose(
    raw_fraction11,
    fraction11,
    atol=1e-7,
    rtol=0
)

V1_RECALL11 = (
    np.diag(CM11) / support11
)

V1_PRECISION11 = np.divide(
    np.diag(CM11),
    predicted11,
    out=np.zeros(10, dtype=float),
    where=predicted11 > 0
)

V1_F1_11 = np.divide(
    2 * V1_PRECISION11 * V1_RECALL11,
    V1_PRECISION11 + V1_RECALL11,
    out=np.zeros(10, dtype=float),
    where=(V1_PRECISION11 + V1_RECALL11) > 0
)

V1_ACCURACY11 = np.trace(CM11) / N11
V1_MACRO_F1_11 = V1_F1_11.mean()
V1_BAL_ACC11 = V1_RECALL11.mean()

for key, calculated in {
    "accuracy": V1_ACCURACY11,
    "macro_f1": V1_MACRO_F1_11,
    "balanced_accuracy": V1_BAL_ACC11
}.items():

    np.testing.assert_allclose(
        calculated,
        multiclass11["baseline_metrics_recomputed"][key],
        atol=2e-8,
        rtol=0
    )

# Independently check original saved class metrics.
class_check11 = class_raw11.set_index("Class").reindex(CLASSES11)

if class_check11.isna().any().any():
    raise RuntimeError("Missing saved class metrics")

np.testing.assert_array_equal(
    class_check11["Support"].to_numpy(dtype=int),
    support11
)

np.testing.assert_allclose(
    class_check11["Recall"].to_numpy(dtype=float),
    V1_RECALL11,
    atol=1e-7,
    rtol=0
)

np.testing.assert_allclose(
    class_check11["F1"].to_numpy(dtype=float),
    V1_F1_11,
    atol=1e-7,
    rtol=0
)

# Cross-check Normal support against binary detector.
normal_idx11 = CLASSES11.index("Normal")
assert support11[normal_idx11] == NORMAL11

dos_idx11 = CLASSES11.index("DoS")
exploit_idx11 = CLASSES11.index("Exploits")

DOS_AS_EXPLOITS11 = int(
    CM11[dos_idx11, exploit_idx11]
)

assert DOS_AS_EXPLOITS11 == 2309

print("\nC. MULTICLASS VERIFICATION")
print("-" * 100)
print("Validation rows:", N11)
print("Classes:", len(CLASSES11))
print(f"V1 accuracy: {V1_ACCURACY11:.6f}")
print(f"V1 macro F1: {V1_MACRO_F1_11:.6f}")
print("DoS -> Exploits errors:", DOS_AS_EXPLOITS11)
print("Confusion / class-metrics agreement: PASS")

# ======================================================================
# E. WEIGHTED XGBOOST V2 CLASS RECALL
#
# These 4-decimal values were printed in Cell 22.
# They are not independently recomputed prediction metrics.
# We validate their mean against saved balanced accuracy.
# ======================================================================

V2_REPORTED_RECALL11 = {
    "Analysis": 0.1797,
    "Backdoor": 0.1574,
    "DoS": 0.6391,
    "Exploits": 0.6646,
    "Fuzzers": 0.8204,
    "Generic": 0.9815,
    "Normal": 0.8864,
    "Reconnaissance": 0.7318,
    "Shellcode": 0.8996,
    "Worms": 0.5909
}

V2_RECALL11 = np.array(
    [
        V2_REPORTED_RECALL11[name]
        for name in CLASSES11
    ],
    dtype=float
)

weighted_balanced11 = float(
    weighted11["validation_metrics"]["Balanced accuracy"]
)

np.testing.assert_allclose(
    V2_RECALL11.mean(),
    weighted_balanced11,
    atol=6e-5,
    rtol=0
)

RECALL_TABLE11 = pd.DataFrame({
    "Class": CLASSES11,
    "Support": support11,
    "XGBoost_V1_Recall": V1_RECALL11,
    "Weighted_XGBoost_V2_Recall_Reported": V2_RECALL11,
    "V2_Minus_V1_Recall": V2_RECALL11 - V1_RECALL11
})

print("\nD. CLASS-WEIGHTING TRADE-OFF")
print("-" * 100)
print(
    RECALL_TABLE11[
        [
            "Class",
            "Support",
            "XGBoost_V1_Recall",
            "Weighted_XGBoost_V2_Recall_Reported",
            "V2_Minus_V1_Recall"
        ]
    ].round(4).to_string(index=False)
)
print("V2 reported mean recall / saved balanced accuracy: MATCH")
print("Note: V2 class-wise scores are previously reported.")

# ======================================================================
# F. FIGURE CONFIGURATION
# ======================================================================

C11 = {
    "binary": "#087D8F",
    "roc": "#6E5A9D",
    "v1": "#3B6798",
    "v2": "#C66D49",
    "neutral": "#68798A",
    "danger": "#B74E49"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig11 = plt.figure(
    figsize=(18, 13.7),
    facecolor="white"
)

gs11 = fig11.add_gridspec(
    2, 2,
    left=0.085,
    right=0.963,
    top=0.833,
    bottom=0.175,
    width_ratios=[1.14, 1.0],
    height_ratios=[1.0, 1.38],
    hspace=0.42,
    wspace=0.34
)

ax_a11 = fig11.add_subplot(gs11[0, 0])
ax_b11 = fig11.add_subplot(gs11[0, 1])
ax_c11 = fig11.add_subplot(gs11[1, 0])
ax_d11 = fig11.add_subplot(gs11[1, 1])

fig11.suptitle(
    "Current network attack detection and attack-type classification",
    x=0.075,
    y=0.98,
    ha="left",
    fontsize=18.5,
    fontweight="bold"
)

fig11.text(
    0.075,
    0.948,
    "UNSW-NB15 development validation  |  "
    "35,167 records  |  "
    "Binary detection versus ten-class classification",
    fontsize=10.7,
    color="#617082"
)

# ======================================================================
# G. PANEL A — BINARY PRECISION–RECALL
# ======================================================================

ax_a11.set_facecolor("#FAFBFC")

ax_a11.plot(
    recall11,
    precision11,
    color=C11["binary"],
    lw=2.25,
    label=f"Attack AP = {AP11:.4f}"
)

ax_a11.axhline(
    ATTACK11 / N11,
    color="#A97B4C",
    lw=1.25,
    linestyle="--",
    label=f"Attack prevalence = {ATTACK11/N11:.3f}"
)

ax_a11.set_xlim(0, 1)
ax_a11.set_ylim(0.65, 1.01)

ax_a11.set_xlabel("Attack recall")
ax_a11.set_ylabel("Attack precision")

ax_a11.set_title(
    "A    Binary detection: precision–recall",
    loc="left",
    fontweight="bold",
    pad=13
)

ax_a11.legend(
    loc="lower left",
    fontsize=9,
    frameon=True,
    edgecolor="#D6DEE5"
)

ax_a11.text(
    0.98, 0.05,
    "Precision axis starts at 0.65",
    transform=ax_a11.transAxes,
    ha="right",
    va="bottom",
    fontsize=8.4,
    color="#667688"
)

ax_a11.grid(
    linestyle=":",
    linewidth=0.6,
    alpha=0.45
)

# ======================================================================
# H. PANEL B — BINARY ROC AND LOW-FPR INSET
# ======================================================================

ax_b11.set_facecolor("#FAFBFC")

ax_b11.plot(
    fpr11,
    tpr11,
    color=C11["roc"],
    lw=2.2,
    label=f"ROC-AUC = {AUC11:.4f}"
)

ax_b11.plot(
    [0, 1], [0, 1],
    color="#A3AFBB",
    linestyle="--",
    linewidth=1.1,
    label="Random ranking"
)

ax_b11.set_xlim(0, 1)
ax_b11.set_ylim(0, 1.015)

ax_b11.set_xlabel("False-positive rate")
ax_b11.set_ylabel("True-positive rate")

ax_b11.set_title(
    "B    Binary detection: ROC",
    loc="left",
    fontweight="bold",
    pad=13
)

ax_b11.legend(
    loc="lower right",
    fontsize=9,
    frameon=True,
    edgecolor="#D6DEE5"
)

ax_b11.grid(
    linestyle=":",
    linewidth=0.6,
    alpha=0.45
)

# Inset zoom, using actual saved points.
ax_zoom11 = ax_b11.inset_axes(
    [0.40, 0.19, 0.54, 0.42]
)

ax_zoom11.plot(
    fpr11,
    tpr11,
    color=C11["roc"],
    lw=1.55
)

ax_zoom11.set_xlim(0, 0.12)
ax_zoom11.set_ylim(0.75, 1.005)

ax_zoom11.set_title(
    "Low-FPR region",
    fontsize=9.0,
    fontweight="bold"
)

ax_zoom11.tick_params(labelsize=7.2)

ax_zoom11.grid(
    linestyle=":",
    linewidth=0.5,
    alpha=0.35
)

# ======================================================================
# I. PANEL C — TEN-CLASS CONFUSION MATRIX
# ======================================================================

ax_c11.set_facecolor("#FAFBFC")

heat11 = ax_c11.imshow(
    fraction11,
    vmin=0,
    vmax=1,
    cmap="YlGnBu",
    interpolation="nearest",
    aspect="equal"
)

ax_c11.set_xticks(np.arange(10))
ax_c11.set_yticks(np.arange(10))

ax_c11.set_xticklabels(
    CLASSES11,
    rotation=52,
    ha="right",
    rotation_mode="anchor",
    fontsize=8.5
)

ax_c11.set_yticklabels([
    f"{name} (n={support11[i]:,})"
    for i, name in enumerate(CLASSES11)
], fontsize=8.4)

ax_c11.set_xlabel("Predicted attack class")
ax_c11.set_ylabel("True attack class")

ax_c11.set_title(
    "C    XGBoost V1: ten-class confusion",
    loc="left",
    fontweight="bold",
    pad=13
)

ax_c11.set_xticks(
    np.arange(-0.5, 10, 1),
    minor=True
)
ax_c11.set_yticks(
    np.arange(-0.5, 10, 1),
    minor=True
)

ax_c11.grid(
    which="minor",
    color="white",
    linewidth=0.7,
    alpha=0.65
)

ax_c11.tick_params(
    which="both",
    length=0
)

for i in range(10):

    for j in range(10):

        value = fraction11[i, j]
        count = int(CM11[i, j])

        # Show important confusion pathways and
        # correctly predicted class percentages.
        if i == j or value >= 0.065:

            text_value = (
                f"{value*100:.1f}%\n"
                f"({count:,})"
            )

            ax_c11.text(
                j, i,
                text_value,
                ha="center",
                va="center",
                fontsize=7.7,
                fontweight="bold" if i == j else "normal",
                color=(
                    "white" if value >= 0.55 else "#203344"
                )
            )

        elif count > 0:
            ax_c11.text(
                j, i,
                "·",
                ha="center",
                va="center",
                fontsize=7.2,
                color="#68798A"
            )

# Highlight the confirmed DoS -> Exploits error.
ax_c11.add_patch(
    Rectangle(
        (exploit_idx11 - 0.48, dos_idx11 - 0.48),
        0.96,
        0.96,
        fill=False,
        linewidth=2.5,
        edgecolor=C11["danger"]
    )
)

cbar11 = fig11.colorbar(
    heat11,
    ax=ax_c11,
    fraction=0.045,
    pad=0.025,
    shrink=0.88
)

cbar11.set_label(
    "Share of true-class records",
    fontsize=8.5
)

# ======================================================================
# J. PANEL D — V1 VERSUS WEIGHTED V2 RECALL
# ======================================================================

# Order classes by recall change, largest gains at top.
ordered11 = RECALL_TABLE11.sort_values(
    "V2_Minus_V1_Recall",
    ascending=False
).reset_index(drop=True)

y11 = np.arange(len(ordered11))

v1_vals11 = ordered11[
    "XGBoost_V1_Recall"
].to_numpy()

v2_vals11 = ordered11[
    "Weighted_XGBoost_V2_Recall_Reported"
].to_numpy()

for i in range(len(ordered11)):

    ax_d11.plot(
        [v1_vals11[i], v2_vals11[i]],
        [y11[i], y11[i]],
        color="#B9C2CB",
        linewidth=2.0,
        zorder=1
    )

ax_d11.scatter(
    v1_vals11,
    y11,
    color=C11["v1"],
    marker="o",
    s=46,
    label="XGBoost V1",
    zorder=3
)

ax_d11.scatter(
    v2_vals11,
    y11,
    color=C11["v2"],
    marker="D",
    s=38,
    label="Weighted XGBoost V2",
    zorder=4
)

ax_d11.set_yticks(y11)

ax_d11.set_yticklabels([
    f"{row.Class}  (n={int(row.Support):,})"
    for row in ordered11.itertuples()
], fontsize=8.8)

ax_d11.invert_yaxis()

ax_d11.set_xlim(-0.025, 1.07)

ax_d11.set_xticks(np.linspace(0, 1, 6))

ax_d11.set_xlabel(
    "Recall (fraction of true-class records)"
)

ax_d11.set_title(
    "D    Class-weighting changes attack-class recall",
    loc="left",
    fontweight="bold",
    pad=13
)

ax_d11.set_facecolor("#FAFBFC")

ax_d11.grid(
    axis="x",
    linestyle=":",
    linewidth=0.65,
    alpha=0.55
)

ax_d11.legend(
    loc="lower right",
    frameon=True,
    fontsize=8.7,
    edgecolor="#D6DEE5"
)

for name in ["DoS", "Exploits"]:

    row = ordered11.index[
        ordered11["Class"] == name
    ][0]

    diff_pp = (
        ordered11.loc[row, "V2_Minus_V1_Recall"] * 100
    )

    ax_d11.text(
        1.045,
        row,
        f"{diff_pp:+.1f} pp",
        ha="right",
        va="center",
        fontsize=8.7,
        color=(
            "#168478" if diff_pp > 0
            else "#B74E49"
        ),
        fontweight="bold"
    )

# ======================================================================
# K. SCIENTIFIC CAPTION
# ======================================================================

fig11.text(
    0.075,
    0.105,
    "Figure 5. Current attack detection is evaluated separately "
    "from ten-class attack identification. Binary scores use "
    "the saved XGBoost V1 attack probability, 1 - P(Normal).",
    fontsize=9.2,
    color="#48586A"
)

fig11.text(
    0.075,
    0.081,
    "The red-outlined confusion cell denotes DoS misclassified "
    "as Exploits (2,309 / 2,552). Weighted V2 class recalls "
    "are rounded values previously reported in Cell 22.",
    fontsize=9.2,
    color="#48586A"
)

fig11.text(
    0.075,
    0.057,
    "UNSW-NB15 development validation; repeatedly examined "
    "during model selection. These results do not establish "
    "independent TEST-set or live-network performance.",
    fontsize=9.0,
    color="#5F6E80"
)

# ======================================================================
# L. SAVE FIGURE AND UNDERLYING RESEARCH DATA
# ======================================================================

STAMP11 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM11 = (
    f"Figure_05_Current_Attack_Detection_{STAMP11}"
)

PATHS11 = {
    "pdf": DEST11 / f"{STEM11}.pdf",
    "svg": DEST11 / f"{STEM11}.svg",
    "png": DEST11 / f"{STEM11}.png",
    "recall": DEST11 / f"{STEM11}_Class_Recall.csv",
    "curves": DEST11 / f"{STEM11}_Binary_Curves.csv.gz",
    "json": DEST11 / f"{STEM11}.json"
}

for path in PATHS11.values():
    if path.exists():
        raise FileExistsError(path)

for ext in ["pdf", "svg", "png"]:

    fig11.savefig(
        PATHS11[ext],
        dpi=450 if ext == "png" else 180,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig11)

RECALL_TABLE11.to_csv(
    PATHS11["recall"],
    index=False,
    mode="x",
    float_format="%.10f"
)

# Preserve the exact plotted binary curve values.
pr_export11 = pd.DataFrame({
    "Curve": "PR",
    "X": recall11,
    "Y": precision11
})

roc_export11 = pd.DataFrame({
    "Curve": "ROC",
    "X": fpr11,
    "Y": tpr11
})

curve_export11 = pd.concat(
    [pr_export11, roc_export11],
    ignore_index=True
)

curve_export11.to_csv(
    PATHS11["curves"],
    index=False,
    compression="gzip",
    float_format="%.12g"
)

# ======================================================================
# M. PROVENANCE MANIFEST
# ======================================================================

GENERATED11 = [
    PATHS11[key]
    for key in ["pdf", "svg", "png", "recall", "curves"]
]

MANIFEST11 = {
    "project": "NetGuard",
    "publication_figure": 5,
    "title": (
        "Current attack detection and attack-type classification"
    ),
    "dataset": "UNSW-NB15 canonical TRAIN",
    "evaluation": "development validation",
    "validation_records": N11,
    "attack_records": ATTACK11,
    "normal_records": NORMAL11,
    "classes": CLASSES11,
    "binary": {
        "model": "XGBoost V1",
        "average_precision": AP11,
        "roc_auc": AUC11,
        "source": "Saved F14 PR and ROC curve records"
    },
    "multiclass": {
        "model": "XGBoost V1",
        "accuracy": float(V1_ACCURACY11),
        "macro_f1": float(V1_MACRO_F1_11),
        "balanced_accuracy": float(V1_BAL_ACC11),
        "dos_as_exploits": DOS_AS_EXPLOITS11,
        "source": "Saved F12 confusion and class metrics"
    },
    "weighted_v2": {
        "evidence_type": (
            "Previously reported rounded per-class "
            "recalls from experiment Cell 22"
        ),
        "independently_recomputed_here": False,
        "mean_recall_matches_saved_balanced_accuracy": True,
        "saved_balanced_accuracy": weighted_balanced11,
        "reported_class_recalls": V2_REPORTED_RECALL11
    },
    "source_files": [
        {
            "path": str(path.relative_to(ROOT11)),
            "sha256": hash11(path)
        }
        for path in (
            list(SOURCES11.values())
            + list(MANIFESTS11.values())
        )
    ],
    "generated_files": [
        {
            "path": str(path.relative_to(ROOT11)),
            "sha256": hash11(path)
        }
        for path in GENERATED11
    ],
    "limitations": [
        "Repeated development-validation evaluation.",
        "Binary detection is not ten-class accuracy.",
        "Weighted V2 per-class values were transcribed "
        "from earlier experiment output.",
        "V2 per-class values were not independently "
        "recomputed in this cell.",
        "No official TEST evaluation.",
        "No live-network performance estimate."
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS11["json"].open("x", encoding="utf-8") as f:
    json.dump(
        MANIFEST11,
        f,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# N. FINAL VERIFICATION
# ======================================================================

print("\n" + "=" * 100)
print("CELL 11 COMPLETE — FINAL PAPER FIGURE 5")
print("=" * 100)

print("Figure panels:", 4)
print("Validation records:", N11)
print("Attack classes including Normal:", len(CLASSES11))
print("Original source hashes: PASS")
print("Binary curve recomputation: PASS")
print("Original multiclass metrics: PASS")
print("V2 balanced-accuracy consistency: PASS")

print("\nKEY FINDINGS")
print("-" * 100)

print(f"Binary AP:       {AP11:.6f}")
print(f"Binary ROC-AUC:  {AUC11:.6f}")
print(f"Ten-class accuracy: {V1_ACCURACY11:.6f}")
print(f"Ten-class macro F1: {V1_MACRO_F1_11:.6f}")
print(f"DoS -> Exploits confusion: {DOS_AS_EXPLOITS11:,}")

print("\nSAVED PUBLICATION ASSETS")
print("-" * 100)

for ext, path in PATHS11.items():

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing/empty output: {path}")

    print(f"PASS [{ext.upper()}] {path.name}")

print("\nFINAL PAPER FIGURES: 5 / 5")
print("FINAL PAPER TABLES: 5 / 5")
print("Remaining paper architecture diagram: 1")
print("Existing research outputs overwritten: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 100)


In [ ]:

# ======================================================================
# CELL 11C — FIGURE 5 FINAL PUBLICATION LAYOUT
#
# Permanent solution:
# - Remove long footnotes from the image itself
# - Preserve exact footnotes in a companion caption file
# - Give rotated class labels additional clearance
# - Automatically validate label positions
# - Save FINAL_V3 without touching earlier exports
#
# Requires the existing Cell 11 figure in memory.
# No training, no TEST access, no model modifications.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib
import json
import shutil

print("=" * 95)
print("CELL 11C — FINAL FIGURE 5 PUBLICATION LAYOUT")
print("=" * 95)

# ======================================================================
# A. VALIDATE EXISTING FIGURE
# ======================================================================

needed = [
    "fig11", "gs11",
    "ax_a11", "ax_b11",
    "ax_c11", "ax_d11",
    "DEST11", "PATHS11", "ROOT11"
]

missing = [
    item for item in needed
    if item not in globals()
]

if missing:
    raise RuntimeError(
        "Run original Cell 11 first. Missing: "
        + ", ".join(missing)
    )

for key in [
    "pdf", "svg", "png",
    "recall", "curves", "json"
]:
    if not PATHS11[key].is_file():
        raise FileNotFoundError(PATHS11[key])

# ======================================================================
# B. PRESERVE ORIGINAL SCIENTIFIC CAPTION
# ======================================================================

# The last three fig.text objects in Cell 11 contain
# the scientific explanation, data provenance and caveats.

if len(fig11.texts) < 5:
    raise RuntimeError(
        "Could not identify the three original caption notes."
    )

caption_artists = list(fig11.texts[-3:])

original_captions = [
    item.get_text()
    for item in caption_artists
]

if not original_captions[0].startswith("Figure 5."):
    raise RuntimeError(
        "Unexpected caption ordering. "
        "Do not hide unidentified figure text."
    )

# Hide only the three explanatory notes.
# Keep title and subtitle fully visible.
for artist in caption_artists:
    artist.set_visible(False)

print("\nA. CAPTION EXTRACTION")
print("-" * 95)

for i, caption in enumerate(original_captions, 1):
    print(f"{i}. {caption}")

# ======================================================================
# C. RESET FIGURE GEOMETRY
#
# Important: earlier failed 11B runs mutated the
# GridSpec in memory. This resets it explicitly.
# ======================================================================

fig11.set_size_inches(
    18.0,
    14.8,
    forward=True
)

gs11.update(
    left=0.085,
    right=0.963,
    top=0.845,
    bottom=0.26,
    hspace=0.47,
    wspace=0.34
)

ax_c11.set_xlabel(
    "Predicted attack class",
    labelpad=15
)

ax_d11.set_xlabel(
    "Recall (fraction of true-class records)",
    labelpad=12
)

# ======================================================================
# D. MEASURE X-AXIS TEXT POSITIONS
#
# Instead of comparing axes against captions,
# check that all bottom labels fit within the figure.
# ======================================================================

def check_axis_layout():

    fig11.canvas.draw()

    renderer = fig11.canvas.get_renderer()

    labels = [
        ax_c11.xaxis.label,
        ax_d11.xaxis.label
    ]

    labels += list(ax_c11.get_xticklabels())
    labels += list(ax_d11.get_xticklabels())

    measured = []

    for label in labels:

        if not label.get_visible():
            continue

        box = (
            label
            .get_window_extent(renderer=renderer)
            .transformed(
                fig11.transFigure.inverted()
            )
        )

        measured.append(box.y0)

    if not measured:
        raise RuntimeError(
            "No lower-axis labels found"
        )

    return min(measured)

# If labels are outside the figure, increase bottom
# panel clearance. Unlike Cell 11B, no footnotes
# occupy the space directly underneath them.

axis_floor = check_axis_layout()

for bottom in [
    0.29, 0.32, 0.35, 0.38, 0.41, 0.45
]:

    if axis_floor >= 0.045:
        break

    gs11.update(bottom=bottom)
    axis_floor = check_axis_layout()

print("\nB. AXIS-LABEL VERIFICATION")
print("-" * 95)

print(
    "Lowest class-label position:",
    f"{axis_floor:.4f}"
)

# Scientific caption is outside the actual image,
# so it cannot overlap the confusion matrix.
assert all(
    not item.get_visible()
    for item in caption_artists
)

print("Long footnotes removed from image: PASS")

if axis_floor < 0:
    raise RuntimeError(
        "An axis label remains outside the figure canvas."
    )

print("Axis labels remain inside figure: PASS")

# ======================================================================
# E. CREATE NEW OUTPUT NAMES
# ======================================================================

stamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

stem = (
    "Figure_05_Current_Attack_Detection_"
    f"FINAL_V3_{stamp}"
)

paths = {
    "pdf": DEST11 / f"{stem}.pdf",
    "svg": DEST11 / f"{stem}.svg",
    "png": DEST11 / f"{stem}.png",
    "recall": DEST11 / f"{stem}_Class_Recall.csv",
    "curves": DEST11 / f"{stem}_Binary_Curves.csv.gz",
    "caption": DEST11 / f"{stem}_Caption.txt",
    "json": DEST11 / f"{stem}.json"
}

for path in paths.values():

    if path.exists():
        raise FileExistsError(path)

# ======================================================================
# F. EXPORT FINAL PUBLICATION FIGURE
# ======================================================================

for ext in ["pdf", "svg", "png"]:

    fig11.savefig(
        paths[ext],
        dpi=450 if ext == "png" else 180,
        bbox_inches="tight",
        pad_inches=0.18,
        facecolor="white"
    )

# Copy the original evidence exactly.
shutil.copyfile(
    PATHS11["recall"],
    paths["recall"]
)

shutil.copyfile(
    PATHS11["curves"],
    paths["curves"]
)

# Save caption independently for later LaTeX use.
with paths["caption"].open(
    "x",
    encoding="utf-8"
) as handle:

    handle.write(
        "FIGURE 5 — CURRENT ATTACK DETECTION\n\n"
    )

    for paragraph in original_captions:
        handle.write(paragraph + "\n\n")

def sha256_file(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

for key in ["recall", "curves"]:

    if sha256_file(paths[key]) != sha256_file(PATHS11[key]):
        raise RuntimeError(
            f"Research data copy mismatch: {key}"
        )

# ======================================================================
# G. NEW PROVENANCE MANIFEST
# ======================================================================

manifest = {
    "project": "NetGuard",
    "publication_figure": 5,
    "version": "FINAL_V3",
    "title": (
        "Current network attack detection "
        "and attack-type classification"
    ),
    "layout_correction": (
        "Separated full scientific figure caption "
        "from publication graphic, restored grid "
        "geometry, and checked all lower-axis labels."
    ),
    "caption": original_captions,
    "scientific_values_changed": False,
    "panels": 4,
    "axis_label_floor": float(axis_floor),
    "source_figure": {
        "path": str(
            PATHS11["png"].relative_to(ROOT11)
        ),
        "sha256": sha256_file(PATHS11["png"])
    },
    "generated_files": [
        {
            "path": str(
                paths[key].relative_to(ROOT11)
            ),
            "sha256": sha256_file(paths[key])
        }
        for key in [
            "pdf", "svg", "png",
            "recall", "curves", "caption"
        ]
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with paths["json"].open(
    "x",
    encoding="utf-8"
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        allow_nan=False
    )

# ======================================================================
# H. DISPLAY AND VERIFY
# ======================================================================

display(fig11)

print("\n" + "=" * 95)
print("CELL 11C COMPLETE — FIGURE 5 FINAL V3")
print("=" * 95)

for ext, path in paths.items():

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Missing or empty output: {path}"
        )

    print(
        f"PASS [{ext.upper()}] {path.name}"
    )

print("\nPublication figure spacing: CORRECTED")
print("Scientific caption saved separately: YES")
print("Original Figure 5 preserved: YES")
print("Scientific data changed: NO")
print("Models retrained: 0")
print("Official TEST accessed: NO")

print("=" * 95)


In [ ]:

# ======================================================================
# NETGUARD — RESEARCH PAPER
# CELL 12 — FINAL SYSTEM ARCHITECTURE DIAGRAM
#
# Outputs:
#   PDF  - vector publication graphic
#   SVG  - editable vector graphic
#   PNG  - high-resolution preview
#   DOT  - editable Graphviz source
#   TXT  - scientific figure caption
#   JSON - reproducibility manifest
#
# Does not load datasets, models, or TEST data.
# Does not overwrite existing files.
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image
import subprocess
import shutil
import hashlib
import json

print("=" * 95)
print("CELL 12 — NETGUARD RESEARCH ARCHITECTURE")
print("=" * 95)

# ===========================================================
# A. VERIFY GOOGLE DRIVE AND GRAPHVIZ
# ===========================================================

ROOT12 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DEST12 = (
    ROOT12
    / "Research_Paper"
    / "03_Architecture_Diagram"
)

if not ROOT12.is_dir():
    raise FileNotFoundError(
        "Google Drive project not found. "
        "Mount Drive before running this cell."
    )

if not DEST12.is_dir():
    raise FileNotFoundError(
        "Architecture directory missing: "
        f"{DEST12}"
    )

if shutil.which("dot") is None:
    raise RuntimeError(
        "Graphviz executable 'dot' not found. "
        "Install the Graphviz system package in Colab."
    )

print("\nA. ENVIRONMENT")
print("-" * 95)
print("Project directory: PASS")
print("Architecture directory: PASS")
print("Graphviz executable: FOUND")

# ===========================================================
# B. DEFINE RESEARCH ARCHITECTURE
# ===========================================================

DOT12 = r'''
digraph NetGuard {

    graph [
        rankdir=LR,
        bgcolor="white",
        pad="0.40",
        nodesep="0.45",
        ranksep="0.68",
        splines=polyline,
        compound=true,
        outputorder=edgesfirst,
        fontname="Helvetica",
        fontsize=18,
        labelloc=t,
        labeljust=l,
        label="NETGUARD  |  RESEARCH PIPELINE ARCHITECTURE\nCurrent network attack identification and future attack forecasting",
        fontcolor="#172B42"
    ];

    node [
        shape=box,
        style="rounded,filled",
        fontname="Helvetica",
        fontsize=11,
        margin="0.19,0.15",
        penwidth=1.1,
        color="#CBD7E0",
        fillcolor="#F5F8FB",
        fontcolor="#23374A"
    ];

    edge [
        color="#72869A",
        penwidth=1.5,
        arrowsize=0.8,
        arrowhead=vee
    ];

    // ======================================================
    // A. CURRENT ATTACK DETECTION
    // ======================================================

    subgraph cluster_current {

        label="A  |  CURRENT ATTACK DETECTION  (UNSW-NB15)";
        labeljust=l;
        labelloc=t;
        fontsize=14;
        fontname="Helvetica-Bold";
        fontcolor="#254D7E";

        style="rounded,filled";
        color="#B7D0E5";
        fillcolor="#F3F8FC";
        penwidth=1.5;
        margin=22;

        d_source [
            label="UNSW-NB15\ncanonical TRAIN partition",
            fillcolor="#E7F1FA",
            color="#9DBED8"
        ];

        d_features [
            label="Flow features (42)\nTrain-fit preprocessing\nCategorical encoding",
            fillcolor="#E7F1FA",
            color="#9DBED8"
        ];

        d_model [
            label="10-class XGBoost V1\nSaved classifier",
            fillcolor="#2E608D",
            color="#2E608D",
            fontcolor="white",
            fontname="Helvetica-Bold"
        ];

        d_output [
            label="Current attack results\nAttack score = 1 - P(Normal)\nAttack-type probabilities",
            fillcolor="#E0F2F0",
            color="#86B7AF"
        ];

        d_val [
            label="Development validation\n35,167 flow records",
            fillcolor="#EDF2F6",
            color="#BDCAD4"
        ];

        d_source -> d_features;
        d_features -> d_model;
        d_model -> d_output;
        d_output -> d_val;
    }

    // ======================================================
    // B. FUTURE ATTACK FORECASTING
    // ======================================================

    subgraph cluster_forecast {

        label="B  |  FUTURE ATTACK FORECASTING  (SYNTHETIC NAF v2)";
        labeljust=l;
        labelloc=t;
        fontsize=14;
        fontname="Helvetica-Bold";
        fontcolor="#166C69";

        style="rounded,filled";
        color="#A6CFCA";
        fillcolor="#F2FAF8";
        penwidth=1.5;
        margin=22;

        f_source [
            label="CIC-IDS2017 / UNSW-NB15\nderived synthetic episodes",
            fillcolor="#E1F1ED",
            color="#90C7BC"
        ];

        f_features [
            label="Minute-level traffic signals\nEpisode/window construction",
            fillcolor="#E1F1ED",
            color="#90C7BC"
        ];

        f_horizon [
            label="Horizon predictors\nLSTM / stacked LSTM / XGBoost V2\nGraph LSTM / GCN / GAT",
            fillcolor="#207E78",
            color="#207E78",
            fontcolor="white",
            fontname="Helvetica-Bold"
        ];

        f_cr [
            label="Competing-risk models\nCR-LSTM / CR-TCN",
            fillcolor="#207E78",
            color="#207E78",
            fontcolor="white",
            fontname="Helvetica-Bold"
        ];

        f_risk [
            label="Future attack risk\nH5 / H10 / H15\n(where available)",
            fillcolor="#E0F2F0",
            color="#86B7AF"
        ];

        f_eta [
            label="Conditional outputs\nFuture attack family\nExpected time-to-attack",
            fillcolor="#E0F2F0",
            color="#86B7AF"
        ];

        f_val [
            label="Development validation\n18,839 aligned windows",
            fillcolor="#EDF2F6",
            color="#BDCAD4"
        ];

        f_source -> f_features;

        f_features -> f_horizon;
        f_features -> f_cr;

        f_horizon -> f_risk;
        f_cr -> f_risk;
        f_cr -> f_eta;

        f_risk -> f_val;
        f_eta -> f_val;
    }

    // ======================================================
    // C. PLANNED INTEGRATION (NOT IMPLEMENTED)
    // ======================================================

    future [
        label="PLANNED (not implemented / validated)\nRAG explanation + dashboard integration",
        style="rounded,dashed,filled",
        fillcolor="#FFF7EC",
        color="#D39B51",
        fontcolor="#80572E",
        fontsize=10
    ];

    d_output -> future [
        style=dashed,
        color="#C19B6A",
        constraint=false
    ];

    f_eta -> future [
        style=dashed,
        color="#C19B6A",
        constraint=false
    ];
}
'''

print("\nB. ARCHITECTURE DEFINED")
print("-" * 95)
print("Current detection pipeline: INCLUDED")
print("Future forecasting pipeline: INCLUDED")
print("Competing-risk conditional outputs: INCLUDED")
print("Development validation: INCLUDED")
print("Planned RAG/dashboard: DASHED")

# ===========================================================
# C. GENERATE UNIQUE FILE NAMES
# ===========================================================

STAMP12 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f"
)

STEM12 = (
    f"NetGuard_Research_Architecture_FINAL_V1_{STAMP12}"
)

PATHS12 = {
    "dot": DEST12 / f"{STEM12}.dot",
    "pdf": DEST12 / f"{STEM12}.pdf",
    "svg": DEST12 / f"{STEM12}.svg",
    "png": DEST12 / f"{STEM12}.png",
    "caption": DEST12 / f"{STEM12}_Caption.txt",
    "json": DEST12 / f"{STEM12}.json"
}

for path in PATHS12.values():
    if path.exists():
        raise FileExistsError(path)

# ===========================================================
# D. SAVE EDITABLE GRAPHVIZ SOURCE
# ===========================================================

with PATHS12["dot"].open(
    "x",
    encoding="utf-8"
) as f:
    f.write(DOT12)

# ===========================================================
# E. EXPORT VECTOR AND RASTER DIAGRAMS
# ===========================================================

for ext in ["svg", "pdf", "png"]:

    command = [
        "dot",
        "-T" + ext
    ]

    if ext == "png":
        command.append("-Gdpi=300")

    command += [
        str(PATHS12["dot"]),
        "-o",
        str(PATHS12[ext])
    ]

    subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True
    )

print("\nC. RENDERING")
print("-" * 95)

for ext in ["pdf", "svg", "png"]:
    print(
        f"PASS [{ext.upper()}] "
        f"{PATHS12[ext].name}"
    )

# ===========================================================
# F. SCIENTIFIC CAPTION
# ===========================================================

CAPTION12 = """
NetGuard research pipeline architecture.

(A) Current attack detection uses flow-level UNSW-NB15
features, train-fitted preprocessing and a saved ten-class
XGBoost classifier. The binary attack score is derived
from the classifier's Normal-class probability.

(B) Future attack forecasting uses Synthetic NAF v2
minute-level traffic episodes derived from CIC-IDS2017
and UNSW-NB15 data. Evaluated horizon-prediction
architectures include recurrent, boosted-tree and
graph-based models. Competing-risk LSTM and TCN models
also estimate future attack family and conditional
expected time-to-attack.

The research evaluations use separate development-
validation records: 35,167 flow records for current
attack classification and 18,839 aligned decision
windows for forecasting.

Dashed connections indicate proposed RAG explanation
and dashboard integration, not experimentally validated
components. This diagram represents the evaluated
research pipelines, not an independently tested or
live-deployed end-to-end monitoring system.
""".strip()

with PATHS12["caption"].open(
    "x",
    encoding="utf-8"
) as f:
    f.write(CAPTION12 + "\n")

# ===========================================================
# G. REPRODUCIBILITY MANIFEST
# ===========================================================

def sha256_12(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

GENERATED12 = [
    PATHS12[key]
    for key in [
        "dot", "pdf", "svg", "png", "caption"
    ]
]

MANIFEST12 = {
    "project": "NetGuard",
    "artifact": "Research architecture diagram",
    "version": "FINAL_V1",
    "current_detection": {
        "dataset": "UNSW-NB15",
        "model": "XGBoost V1",
        "validation_records": 35167,
        "outputs": [
            "Attack probability",
            "Ten-class attack identification"
        ]
    },
    "forecasting": {
        "dataset": "Synthetic NAF v2",
        "validation_windows": 18839,
        "outputs": [
            "H5/H10/H15 future attack risk",
            "Conditional future attack family",
            "Conditional time-to-attack"
        ]
    },
    "rag_status": "planned_not_implemented",
    "dashboard_integration_status": "not_verified",
    "deployment_claimed": False,
    "independent_test_claimed": False,
    "generated_files": [
        {
            "path": str(
                path.relative_to(ROOT12)
            ),
            "sha256": sha256_12(path)
        }
        for path in GENERATED12
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS12["json"].open(
    "x",
    encoding="utf-8"
) as f:
    json.dump(
        MANIFEST12,
        f,
        indent=2,
        allow_nan=False
    )

# ===========================================================
# H. DISPLAY
# ===========================================================

print("\nD. DIAGRAM PREVIEW")
print("-" * 95)

display(
    Image(
        filename=str(PATHS12["png"]),
        width=1400
    )
)

# ===========================================================
# I. FINAL VERIFICATION
# ===========================================================

print("\n" + "=" * 95)
print("CELL 12 COMPLETE — NETGUARD ARCHITECTURE")
print("=" * 95)

for key, path in PATHS12.items():

    if not path.is_file():
        raise RuntimeError(
            f"Missing output: {path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Empty output: {path}"
        )

    print(
        f"PASS [{key.upper()}] {path.name}"
    )

print("\nArchitecture diagram: 1 / 1")
print("Final paper figures: 5 / 5")
print("Final paper tables: 5 / 5")
print("Editable Graphviz source: YES")
print("Scientific caption: SAVED")
print("Models retrained: NO")
print("Official TEST accessed: NO")
print("Previous files overwritten: NO")
print("=" * 95)


In [ ]:

# =====================================================================
# NETGUARD — FINAL PAPER ARCHITECTURE
# CELL 12 V2 — MONOCHROME TECHNICAL SYSTEM DIAGRAM
#
# Two parallel scientific workflows:
#   A. Current attack classification
#   B. Future attack forecasting
#
# Visual design:
#   Black / white / grayscale only
#   Rectangular, numbered modules
#   Solid arrows = primary evaluated paths
#   Dashed arrows = evaluated alternative experiments
#
# Exports PDF, SVG, PNG, DOT, TXT and JSON.
# No model training, TEST access or file overwriting.
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
from html import escape
from IPython.display import display, Image
import subprocess
import shutil
import hashlib
import json

print("=" * 90)
print("CELL 12 V2 — MONOCHROME RESEARCH ARCHITECTURE")
print("=" * 90)

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DEST = ROOT / "Research_Paper" / "03_Architecture_Diagram"

if not ROOT.is_dir():
    raise FileNotFoundError(
        "Project directory missing. Mount Google Drive first."
    )

if not DEST.is_dir():
    raise FileNotFoundError(
        "Architecture folder missing: " + str(DEST)
    )

if shutil.which("dot") is None:
    raise RuntimeError(
        "Graphviz executable not found. "
        "Install Graphviz before running."
    )

# =====================================================================
# 1. MONOCHROME NODE DESIGN
# =====================================================================

def architecture_box(title, lines, shaded=False, strong=False):

    header_bg = "#EAEAEA" if shaded else "#F5F5F5"
    border = "1.5" if strong else "1"

    content = '<BR ALIGN="LEFT"/>'.join(
        escape(line) for line in lines
    )

    return f'''<<TABLE
        BORDER="{border}"
        CELLBORDER="0"
        CELLSPACING="0"
        CELLPADDING="7"
        COLOR="#353535"
        BGCOLOR="white">

        <TR>
            <TD ALIGN="LEFT" BGCOLOR="{header_bg}">
                <FONT FACE="Helvetica" POINT-SIZE="10">
                    <B>{escape(title)}</B>
                </FONT>
            </TD>
        </TR>

        <TR>
            <TD ALIGN="LEFT">
                <FONT FACE="Helvetica" POINT-SIZE="10">
                    {content}
                </FONT>
            </TD>
        </TR>
    </TABLE>>'''

# =====================================================================
# 2. ARCHITECTURE MODULES
# =====================================================================

NODES = {

    # ---------------------------------------------------------------
    # A. CURRENT ATTACK CLASSIFICATION
    # ---------------------------------------------------------------

    "D1": architecture_box(
        "01  DATA SOURCE",
        [
            "UNSW-NB15 canonical TRAIN",
            "175,341 labeled flow records",
            "Official TEST held out"
        ],
        shaded=True
    ),

    "D2": architecture_box(
        "02  DEVELOPMENT SPLIT",
        [
            "Feature-signature grouping",
            "Train: 140,174 flows",
            "Validation: 35,167 flows"
        ]
    ),

    "D3": architecture_box(
        "03  FEATURE ENGINEERING",
        [
            "42 flow-level predictors",
            "Numeric / categorical processing",
            "Train-fitted preprocessing"
        ]
    ),

    "D4": architecture_box(
        "04  MULTICLASS DETECTOR",
        [
            "XGBoost V1 | 10 classes",
            "Saved model + preprocessing",
            "Normal + 9 attack types"
        ],
        shaded=True,
        strong=True
    ),

    "D5": architecture_box(
        "05  DECISION OUTPUT",
        [
            "Binary score = 1 - P(Normal)",
            "Predicted attack category",
            "Class-probability distribution"
        ]
    ),

    "D6": architecture_box(
        "06  MODEL EVALUATION",
        [
            "Binary AP / ROC-AUC / calibration",
            "Ten-class macro F1 / confusion",
            "Threshold and class trade-offs"
        ],
        shaded=True
    ),

    "D7": architecture_box(
        "ADDITIONAL MODEL EXPERIMENTS",
        [
            "Weighted XGBoost V2",
            "Conflict-aware XGBoost",
            "CatBoost baseline"
        ]
    ),

    # ---------------------------------------------------------------
    # B. FUTURE ATTACK FORECASTING
    # ---------------------------------------------------------------

    "F1": architecture_box(
        "01  DATA SOURCES",
        [
            "Synthetic NAF v2 episodes",
            "CIC-IDS2017 + UNSW-NB15 derived",
            "Synthetic; not live traffic"
        ],
        shaded=True
    ),

    "F2": architecture_box(
        "02  TEMPORAL FEATURES",
        [
            "Minute-level traffic aggregations",
            "Sequence / graph representations",
            "Event labels: H5, H10, H15"
        ]
    ),

    "F3": architecture_box(
        "03b  SEQUENCE / TABULAR",
        [
            "Single and stacked LSTM",
            "XGBoost V2 (H5, H10 only)"
        ]
    ),

    "F4": architecture_box(
        "03a  GRAPH-BASED",
        [
            "Original / optimized Graph LSTM",
            "GCN-LSTM, GAT-LSTM"
        ]
    ),

    "F5": architecture_box(
        "03c  COMPETING RISK",
        [
            "CR-LSTM and CR-TCN",
            "Multi-output future-event models"
        ],
        shaded=True,
        strong=True
    ),

    "F6": architecture_box(
        "04a  HORIZON RISKS",
        [
            "P(event within next 5 min)",
            "P(event within next 10 min)",
            "P(event within next 15 min)"
        ]
    ),

    "F7": architecture_box(
        "04b  CONDITIONAL OUTPUTS",
        [
            "Future attack family: 5 groups",
            "Expected time-to-attack (1-15 min)",
            "Conditional on a true future event"
        ]
    ),

    "F8": architecture_box(
        "05  MODEL EVALUATION",
        [
            "18,839 aligned validation windows",
            "AP / ECE / fixed-threshold alerts",
            "Family F1 / ETA MAE / bootstrap CIs"
        ],
        shaded=True
    )
}

# =====================================================================
# 3. GRAPH STRUCTURE AND CONNECTIONS
# =====================================================================

DOT = r'''
digraph NetGuard {

    graph [
        rankdir=TB,
        bgcolor="white",
        pad="0.28",
        nodesep="0.30",
        ranksep="0.42",
        compound=true,
        newrank=true,
        splines=polyline,
        outputorder=edgesfirst,
        fontname="Helvetica",
        fontsize=18,
        labeljust="l",
        labelloc="t",
        label="NETGUARD  |  DUAL-PATH RESEARCH ARCHITECTURE\nEvaluated pipelines (not an integrated live deployment)",
        dpi=180
    ];

    node [
        shape=plain,
        fontname="Helvetica",
        fontsize=10
    ];

    edge [
        color="#444444",
        penwidth=1.0,
        arrowsize=0.7,
        fontname="Helvetica",
        fontsize=9
    ];
'''

for name, label in NODES.items():
    DOT += f'    {name} [label={label}];\n'

DOT += r'''

    // ---------------------------------------------------------------
    // FUTURE FORECASTING
    // ---------------------------------------------------------------

    subgraph cluster_forecast {

        label="B  |  FUTURE ATTACK FORECASTING  /  MINUTE WINDOWS";
        labelloc=t;
        labeljust=l;
        fontname="Helvetica-Bold";
        fontsize=14;
        color="#777777";
        penwidth=1.2;
        margin=21;

        F1; F2; F3; F4; F5; F6; F7; F8;

        F1 -> F2;

        F2 -> F3;
        F2 -> F4;
        F2 -> F5;

        F3 -> F6;
        F4 -> F6;
        F5 -> F6;

        F5 -> F7;

        F6 -> F8;
        F7 -> F8;
    }

    // ---------------------------------------------------------------
    // CURRENT DETECTION
    // ---------------------------------------------------------------

    subgraph cluster_detection {

        label="A  |  CURRENT ATTACK DETECTION  /  FLOW RECORDS";
        labelloc=t;
        labeljust=l;
        fontname="Helvetica-Bold";
        fontsize=14;
        color="#777777";
        penwidth=1.2;
        margin=21;

        D1; D2; D3; D4; D5; D6; D7;

        D1 -> D2;
        D2 -> D3;
        D3 -> D4;
        D4 -> D5;
        D5 -> D6;

        D3 -> D7 [
            style=dashed,
            color="#888888"
        ];

        D7 -> D6 [
            style=dashed,
            color="#888888"
        ];
    }

    // ---------------------------------------------------------------
    // ALIGN TWO PRIMARY RESEARCH BRANCHES
    // ---------------------------------------------------------------

    {rank=same; F1; D1;}
    {rank=same; F2; D2;}
    {rank=same; F5; D4;}
    {rank=same; F6; D5;}
    {rank=same; F8; D6;}
}
'''

# =====================================================================
# 4. UNIQUE RESEARCH-PAPER EXPORT PATHS
# =====================================================================

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

STEM = (
    f"NetGuard_Academic_Architecture_MONO_V2_{STAMP}"
)

PATHS = {
    "dot": DEST / f"{STEM}.dot",
    "pdf": DEST / f"{STEM}.pdf",
    "svg": DEST / f"{STEM}.svg",
    "png": DEST / f"{STEM}.png",
    "caption": DEST / f"{STEM}_Caption.txt",
    "manifest": DEST / f"{STEM}.json"
}

for path in PATHS.values():
    if path.exists():
        raise FileExistsError(path)

# =====================================================================
# 5. SAVE SOURCE AND GENERATE FIGURES
# =====================================================================

with PATHS["dot"].open(
    "x",
    encoding="utf-8"
) as f:
    f.write(DOT)

for ext in ["pdf", "svg", "png"]:

    args = ["dot"]

    if ext == "png":
        args.append("-Gdpi=300")

    args += [
        f"-T{ext}",
        str(PATHS["dot"]),
        "-o",
        str(PATHS[ext])
    ]

    result = subprocess.run(
        args,
        capture_output=True,
        text=True,
        check=True
    )

print("\nA. VECTOR AND RASTER EXPORTS")
print("-" * 90)

for ext in ["pdf", "svg", "png"]:
    print("PASS:", PATHS[ext].name)

# =====================================================================
# 6. ARCHITECTURE CAPTION
# =====================================================================

CAPTION = """
NetGuard dual-path research architecture.

Panel A depicts current network attack detection on UNSW-NB15
flow records. The pipeline consists of a grouped development
split, train-fitted feature preprocessing, a ten-class XGBoost
classifier, derived binary attack probabilities, and multiclass
attack-type identification. Weighted XGBoost, conflict-aware
XGBoost and CatBoost were evaluated as comparison models.

Panel B depicts the future-attack forecasting experiments
performed on Synthetic NAF v2 minute-level episodes derived
from CIC-IDS2017 and UNSW-NB15 sources. Evaluated model
families include recurrent, boosted-tree, graph-based and
competing-risk architectures. The outputs comprise event
probabilities for 5-, 10- and 15-minute horizons, with the
15-minute prediction unavailable for XGBoost V2, together
with conditional future attack-family and time-to-attack
estimates from competing-risk models.

Both pipelines were evaluated on development-validation data,
not independent TEST data. The current attack and forecasting
branches were not validated as a single live-deployed system.
The planned RAG explanation and dashboard integrations are
outside the scope of this evaluated architecture.

Solid arrows identify primary evaluated processing paths.
Dashed arrows denote additional evaluated classifier variants.
""".strip()

with PATHS["caption"].open(
    "x",
    encoding="utf-8"
) as f:
    f.write(CAPTION + "\n")

# =====================================================================
# 7. REPRODUCIBILITY MANIFEST
# =====================================================================

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

MANIFEST = {
    "project": "NetGuard",
    "artifact": "Scientific system architecture",
    "version": "MONO_V2",
    "design": "Black-and-white dual-path technical diagram",
    "current_detection": {
        "dataset": "UNSW-NB15",
        "model": "XGBoost V1",
        "development_train": 140174,
        "development_validation": 35167
    },
    "future_forecasting": {
        "dataset": "Synthetic NAF v2",
        "validation_windows": 18839,
        "forecast_horizons_minutes": [5, 10, 15],
        "conditional_models": ["CR-LSTM", "CR-TCN"]
    },
    "rag_integration": "Not depicted; not validated",
    "live_deployment_claim": False,
    "independent_test_claim": False,
    "generated_files": [
        {
            "path": str(PATHS[key].relative_to(ROOT)),
            "sha256": sha256_file(PATHS[key])
        }
        for key in [
            "dot", "pdf", "svg", "png", "caption"
        ]
    ],
    "models_retrained": 0,
    "official_test_accessed": False
}

with PATHS["manifest"].open(
    "x",
    encoding="utf-8"
) as f:
    json.dump(
        MANIFEST,
        f,
        indent=2,
        allow_nan=False
    )

# =====================================================================
# 8. SHOW AND VALIDATE
# =====================================================================

print("\nB. FINAL ARCHITECTURE PREVIEW")
print("-" * 90)

display(
    Image(
        filename=str(PATHS["png"]),
        width=1100
    )
)

print("\n" + "=" * 90)
print("CELL 12 V2 COMPLETE")
print("=" * 90)

for key, path in PATHS.items():

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Invalid output: {path}"
        )

    print(f"PASS [{key.upper()}] {path.name}")

print("\nArchitecture: MONOCHROME V2")
print("Original architecture preserved: YES")
print("Editable DOT source: YES")
print("Scientific caption saved: YES")
print("Models retrained: NO")
print("Official TEST accessed: NO")
print("Existing exports overwritten: NO")
print("=" * 90)
